# مختبر بحث الإشارات (Signal Discovery Lab)

**الهدف:** بنية تحتية لاكتشاف مرشّحين جدد للإشارة (`DISCOVERY_TRACKS` في
`signal_evaluation_axis`: `hypothesis_driven`, `data_driven`,
`literature_mining`, `genetic_search`) — **بلا أي تدريب شبكة عصبية**، فقط
دوال جاهزة/رخيصة (مؤشرات موجودة أصلاً في `feature_order`، أو انحدار خطّي
Ridge يُحسَب في أجزاء من الثانية لكل نافذة) مُقيَّمة عبر نفس صرامة المحور
(IC + عُشر + خطّ أساس عشوائي عبر نوافذ متحرّكة، كما في H001/H002).

**لماذا بلا تدريب؟** كل تجربة NIG-TimeNet v2 حتى الآن (main.ipynb، H002)
استهلكت وقتاً وتكلفة حوسبة حقيقية لكل نافذة/تشغيل. هذا الدفتر يفحص عشرات
المرشّحين دفعة واحدة **بتكلفة تقارب الصفر** (ثوانٍ لا دقائق) قبل تبرير أي
تدريب فعلي — فرز أوّلي رخيص، لا بديل عن H001/H002 حين يستحقّ مرشّح تدريباً حقيقياً.

**⚠️ تحذير حرِج — اقرأه قبل الوثوق بأي نتيجة على `high`/`low`:**
اكتُشف أثناء بناء هذا الدفتر أن `y_high_reg`/`y_low_reg` (`reg_target_mode=
'return'`) تُقارَنان بمرجع "نفس النوع" (`last_high`/`last_low`) لا
`last_close` (راجع تنبيه رقم ٢٠ في رأس `crypto_data_pipeline_v6.ipynb`
للتفاصيل والبرهان الرقمي الكامل). هذا يجعل ميزات شكل الشمعة الأخيرة
(`BODY_ratio`, `WICK_upper/lower`, وبدرجة أقل `RET_1`) تُظهر ارتباطاً زائفاً
**قوياً جداً** (سبيرمان ≈+0.51 على بيانات حقيقية) بهذين الهدفين تحديداً —
اختفى تماماً (إلى ≈+0.01) عند توحيد المرجع. **كل دالة تقييم في هذا الدفتر
تستخدم `clean_reg_target` (مرجع `last_close` موحّد) تلقائياً لـ`high`/`low`
— لا `y_high_reg`/`y_low_reg` الأصليين مباشرة.** `close_reg` غير متأثر أصلاً
(مرجعه `last_close` دائماً).

## ١) التجهيز — تحميل تعريفات الدفاتر بأمان (بلا تنفيذ تلقائي لخلايا الأمثلة)

`%run` مباشر لـ`signal_evaluation_axis` قد يُنفِّذ خلايا أمثلته (تفترض
`dataset`/`windows` جاهزين من جلسة سابقة) فيفشل بخطأ متغيّر غير معرَّف. نفس
الأسلوب المُستخدَم فعلاً لاختبار H002 على بيانات حقيقية: استخراج تعريفات
الدوال/الأصناف فقط عبر `ast`، بلا كود سائق.

In [ ]:
# @title
!git clone -q https://github.com/yuosef772424/crypto-signal-prediction.git 2>/dev/null || true
%cd /content/crypto-signal-prediction

import json, ast, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd


def _notebook_code(path):
    nb = json.load(open(path, encoding="utf-8"))
    return "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")


def load_notebook_defs(path):
    """يستخرج تعريفات الدوال/الأصناف والاستيرادات والقيم الحرفية فقط من دفتر
    — يتجاهل خلايا الأمثلة/السائقة (متغيّرات تفاعلية غير معرَّفة، أو استدعاءات
    شبكية حقيقية). آمن لتحميل signal_evaluation_axis دون تشغيله بالكامل."""
    code_text = _notebook_code(path)
    code_text = "\n".join(l for l in code_text.split("\n")
                          if not l.strip().startswith(("%", "!")))
    tree = ast.parse(code_text)
    keep_types = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Import, ast.ImportFrom)
    literal_types = (ast.Tuple, ast.List, ast.Constant, ast.Dict, ast.Set)
    kept, futures = [], []
    for node in tree.body:
        if isinstance(node, ast.ImportFrom) and node.module == "__future__":
            futures.append(node)
        elif isinstance(node, keep_types):
            kept.append(node)
        elif isinstance(node, ast.Assign) and isinstance(node.value, literal_types):
            kept.append(node)
    mod = ast.Module(body=futures[:1] + kept, type_ignores=[])
    ast.fix_missing_locations(mod)
    return ast.unparse(mod)


%run "crypto_data_pipeline_v6.ipynb"

exec(compile(load_notebook_defs('signal_evaluation_axis (3).ipynb'), "axis", "exec"))
print("✅ تعريفات المحور مُحمَّلة: rolling_splits, evaluate_windows, concat_splits, "
      "extract_actuals, register_hypothesis, list_registry")

## ٢) تحميل البيانات وبناء النوافذ المتحرّكة

نفس الإعداد المُستخدَم في H002 — عدّله حسب مجموعة أصولك.

In [ ]:
# @title
dataset = load_data_from_drive()  # أو مسار preprocessing_output_latest.pkl.gz لديك
FEATURE_ORDER = dataset["feature_order"]

update_config({"min_split_samples": 10})  # ⚠️ خفّضه فقط إن أصولك القليلة تحتاجه (راجع H002)
windows = rolling_splits(
    dataset, test_span="30D", val_span="15D", initial_train_span="365D",
    step="30D", max_windows=12, keep_asset_test_separate=False, config=CONFIG,
)

## ٣) الحارس ضدّ أثر مرجع "نفس النوع" — `clean_reg_target`

استخدمه دائماً بدل `y_high_reg`/`y_low_reg` الأصليين عند تقييم أي مرشّح جديد
ضد high/low (راجع التحذير أعلى الدفتر). `close_reg` غير متأثر فيبقى كما هو.

In [ ]:
# @title
def clean_reg_target(split, target):
    """`y_{target}_reg` بمرجع `last_close` موحّد لكل الأهداف — لا
    `last_high`/`last_low` الأصليين (own-kind reference) اللذين يحملان أثر
    شكل الشمعة الأخيرة (راجع تنبيه رقم ٢٠ في crypto_data_pipeline_v6). لهدف
    `close` يعادل `y_close_reg` تماماً (نفس المرجع أصلاً) فيُقرَأ منه مباشرة."""
    if target == "close":
        return extract_actuals(split, target_key="y_close_reg")
    split = concat_splits(split)
    lc = np.asarray(split["last_candles"])
    last_close = lc[:, LAST_COLUMNS.index("last_close")]
    future_col = {"high": "future_high_max", "low": "future_low_min"}[target]
    future = lc[:, LAST_COLUMNS.index(future_col)]
    return (future - last_close) / last_close


def evaluate_candidate(predict_fn, target, windows, n_shuffles=1000, min_samples=10, seed=42, verbose=False):
    """يقيّم مرشّحاً واحداً عبر كل النوافذ — بديل رقيق لـ
    evaluate_hypothesis_over_rolling_windows يستخدم دائماً clean_reg_target
    (لا target_key خام) فلا يُمكن نسيان الحارس بالخطأ."""
    names = [f"نافذة {i + 1}" for i in range(len(windows))]
    results = []
    for name, (train, val, test) in zip(names, windows):
        test_flat = concat_splits(test)
        preds = np.asarray(predict_fn(train, val, test), dtype="float64")
        actuals = clean_reg_target(test_flat, target)
        if len(preds) != len(actuals):
            raise ValueError(f"[{name}] طول التنبؤات ({len(preds)}) ≠ طول الأهداف ({len(actuals)}).")
        results.append((name, preds, actuals))
    return evaluate_windows(results, n_shuffles=n_shuffles, min_samples=min_samples, seed=seed, verbose=verbose)

## ٤) إطار المرشّح الواحد — أي ميزة جاهزة كمرشّح فوراً

`make_feature_predict_fn` يحوّل أي عمود من `feature_order` (بعد تحويل اختياري
— عكس، تمركز حول نقطة، إلخ) إلى `predict_fn` جاهزة لـ`evaluate_candidate`،
بنفس نمط `momentum_predict_fn` في المحور لكن مُعمَّمة لأي ميزة.

In [ ]:
# @title
def extract_feature_last_value(split, feature, tf=None, feature_order=None):
    """آخر قيمة (خطوة زمنية أخيرة) لميزة واحدة داخل نافذة كل عيّنة — حالة
    المؤشر عند لحظة القرار، لا فرقها كـextract_feature_last_diff."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    idx = feature_order.index(feature)
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])
    return X[:, -1, idx]


def extract_feature_matrix(split, features=None, tf=None, feature_order=None):
    """مصفوفة كل الميزات (أو مجموعة فرعية) في آخر خطوة زمنية — (N, len(features)).
    أعمّ من extract_feature_last_value: تُستخدَم لأي مرشّح يحتاج أكثر من ميزة
    معاً (تفاعل، مركَّب Ridge، تدريب Isolation Forest، أو دالة مخصَّصة كاملة
    على متجه الميزات — kind='custom'/'trained' أدناه)."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])[:, -1, :]
    if features is not None:
        idx = [feature_order.index(f) for f in features]
        X = X[:, idx]
    return X


def extract_feature_series(split, feature, tf=None, feature_order=None):
    """كل الخطوات الزمنية لميزة واحدة داخل نافذة كل عيّنة — (N, T)، لا آخر
    خطوة فقط كـextract_feature_last_value. لازمة لأي مؤشر مشتقّ يحتاج
    تاريخاً كاملاً ضمن النافذة (CMO/TSI عبر pandas_ta مثلاً، لا يُحسَبان من
    قيمة أخيرة وحدها) — kind='series' أدناه."""
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError("مرّر feature_order صراحةً (dataset['feature_order']).")
    idx = feature_order.index(feature)
    if tf is None:
        tf = [k for k in split if k.startswith("X_")][0][2:]
    X = np.asarray(split[f"X_{tf}"])
    return X[:, :, idx]


def make_feature_predict_fn(feature, transform=None, tf=None, feature_order=None):
    """predict_fn جاهزة لـevaluate_candidate من أي ميزة في feature_order.
    مرّر transform (مثلاً lambda v: -(v-50.0) لعكس RSI حول نقطة المنتصف)
    لصياغة فرضية اتجاه محدّدة بدل القيمة الخام."""
    def predict_fn(train, val, test):
        v = extract_feature_last_value(test, feature=feature, tf=tf, feature_order=feature_order)
        return transform(v) if transform else v
    return predict_fn


def make_interaction_predict_fn(feat_a, feat_b, op="mul", transform=None, tf=None, feature_order=None):
    """predict_fn من تفاعل بين ميزتين موجودتين (ضرب أو فرق) — يغطي فرضيات
    "تفاعلات" (قسم ٣ في خطة المشروع، مثال: حجم×جسم الشمعة) بلا الحاجة لعمود
    ميزة جديد محسوب مسبقاً في خط الأنابيب."""
    def predict_fn(train, val, test):
        a = extract_feature_last_value(test, feature=feat_a, tf=tf, feature_order=feature_order)
        b = extract_feature_last_value(test, feature=feat_b, tf=tf, feature_order=feature_order)
        v = a * b if op == "mul" else (a - b)
        return transform(v) if transform else v
    return predict_fn


def make_custom_predict_fn(fn, tf=None, feature_order=None):
    """predict_fn من دالة مخصَّصة على متجه الميزات الكامل عند آخر خطوة زمنية
    (`fn(X_last, feature_order) -> np.ndarray`) — بلا تدريب (test فقط، بلا
    استخدام train/val). يغطي فرضيات "توليد ممنهج" بلا حاجة لعمود ميزة/تفاعل
    ثنائي جاهز: مقلوب الافتراض الضمني (مثلاً RSI مطبَّع بالتقلب)، كسر
    التناظر (معادلتان مختلفتان للصعود/الهبوط عمداً)، إلخ — راجع قسم "توليد
    فرضيات ممنهج" أدناه."""
    def predict_fn(train, val, test):
        X_last = extract_feature_matrix(test, tf=tf, feature_order=feature_order)
        return fn(X_last, feature_order)
    return predict_fn


def make_series_predict_fn(fn, feature="close", tf=None, feature_order=None):
    """predict_fn من دالة مخصَّصة تُطبَّق على كامل نافذة ميزة واحدة عبر
    الزمن (`fn(series_2d) -> np.ndarray`، حيث `series_2d` شكلها (N, T)) —
    بخلاف kind='custom' (متجه كل الميزات معاً، لكن آخر خطوة فقط)، هذا لميزة
    واحدة تحتاج تاريخاً كاملاً ضمن النافذة (مؤشر مُشتقّ عبر pandas_ta مثل
    CMO/TSI لا يُحسَب من قيمة أخيرة وحدها)."""
    def predict_fn(train, val, test):
        series = extract_feature_series(test, feature=feature, tf=tf, feature_order=feature_order)
        return fn(series)
    return predict_fn


def make_candidate_predict_fn(cand, tf=None, feature_order=None):
    """يبني predict_fn من قاموس مرشّح واحد بصرف النظر عن نوعه (`kind`) —
    نقطة التفرّع الوحيدة، فلا يحتاج scan_candidates/run_batch_and_register
    معرفة الفرق بين الأنواع:

    * الافتراضي (بلا `kind`): ميزة واحدة عبر make_feature_predict_fn.
    * `kind="interaction"`: تفاعل بين ميزتين (`feat_a`/`feat_b`/`op`).
    * `kind="custom"`: دالة مخصَّصة بلا تدريب على متجه الميزات الكامل (`fn`).
    * `kind="trained"`: مرشّح يحتاج تدريباً على train (مثل Isolation Forest) —
      `builder(feature_order=...)` يُرجع predict_fn جاهزة (نفس نمط
      make_ridge_composite_predict_fn في قسم البحث التركيبي).
    * `kind="series"`: دالة مخصَّصة على كامل نافذة ميزة واحدة عبر الزمن
      (`fn`/`feature`) — لمؤشر مشتقّ يحتاج تاريخاً كاملاً (CMO/TSI)."""
    kind = cand.get("kind", "feature")
    if kind == "interaction":
        return make_interaction_predict_fn(cand["feat_a"], cand["feat_b"], op=cand.get("op", "mul"),
                                           transform=cand.get("transform"), tf=tf, feature_order=feature_order)
    if kind == "custom":
        return make_custom_predict_fn(cand["fn"], tf=tf, feature_order=feature_order)
    if kind == "trained":
        return cand["builder"](feature_order=feature_order)
    if kind == "series":
        return make_series_predict_fn(cand["fn"], feature=cand.get("feature", "close"), tf=tf, feature_order=feature_order)
    return make_feature_predict_fn(cand["feature"], transform=cand.get("transform"), tf=tf, feature_order=feature_order)

## ٥) مكتبة مرشّحين جاهزين (`literature_mining` + `data_driven`)

كل مرشّح: اسم، مسار اكتشاف (`DISCOVERY_TRACKS`)، ميزة من `feature_order`،
وتحويل اختياري يصيغ فرضية اتجاه (ارتداد/استمرار). أضِف مرشّحين جدداً بنفس
الشكل — لا حاجة لتعديل أي دالة أخرى.

In [ ]:
# @title
CANDIDATE_SIGNALS = [
    {"name": "RSI_14_reversion", "track": "literature_mining", "feature": "RSI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "RSI متطرف يعكس (mean-reversion كلاسيكي)"},
    {"name": "MACDh_12_26_9_momentum", "track": "literature_mining", "feature": "MACDh_12_26_9",
     "transform": None, "hypothesis": "زخم MACD histogram يستمر"},
    {"name": "BBP_reversion", "track": "literature_mining", "feature": "BBP_20_2.0",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن نطاق بولنجر يعكس"},
    {"name": "STOCH_reversion", "track": "literature_mining", "feature": "STOCHk_14_3_3",
     "transform": lambda v: -(v - 50.0), "hypothesis": "ستوكاستك متطرف يعكس"},
    {"name": "MFI_reversion", "track": "literature_mining", "feature": "MFI_14",
     "transform": lambda v: -(v - 50.0), "hypothesis": "تدفّق نقدي متطرف يعكس"},
    {"name": "CMF_momentum", "track": "data_driven", "feature": "CMF_20",
     "transform": None, "hypothesis": "تدفّق نقدي موجب يستمر"},
    {"name": "NATR_neg_vol", "track": "data_driven", "feature": "NATR_14",
     "transform": lambda v: -v, "hypothesis": "تقلّب مرتفع يسبق عائداً سالباً"},
    {"name": "ADX_trend_strength", "track": "data_driven", "feature": "ADX_14",
     "transform": None, "hypothesis": "قوة اتجاه مرتفعة تدعم استمراره"},
    {"name": "RET_1_reversion", "track": "hypothesis_driven", "feature": "RET_1",
     "transform": lambda v: -v, "hypothesis": "انعكاس قصير المدى (H001) بميزة RET_1 مباشرة"},
    {"name": "RET_6_momentum", "track": "literature_mining", "feature": "RET_6", "transform": None,
     "hypothesis": "زخم متوسط المدى (6 شموع)"},
    {"name": "RET_24_momentum", "track": "literature_mining", "feature": "RET_24", "transform": None,
     "hypothesis": "زخم أطول مدى (24 شمعة)"},
    {"name": "VOLZ_volume_shock", "track": "data_driven", "feature": "VOLZ_20", "transform": None,
     "hypothesis": "فورة حجم تسبق حركة سعرية"},
    {"name": "VOLR_regime", "track": "data_driven", "feature": "VOLR_6_24", "transform": None,
     "hypothesis": "نسبة تقلّب قصير/طويل المدى تكشف نظام سعري"},
    {"name": "POS_14_reversion", "track": "data_driven", "feature": "POS_14",
     "transform": lambda v: -(v - 0.5), "hypothesis": "موقع متطرف ضمن مدى 14 يعكس"},
    {"name": "MKT_beta", "track": "data_driven", "feature": "MKT_ret_1", "transform": None,
     "hypothesis": "عائد السوق العام (بيتا) يتنبأ بعائد الأصل — ⚠️ غير قابلة للاختبار فعلياً على أي "
                   "dataset بُني بـ market_context.enabled=False (الافتراضي): MKT_ret_1 تكون صفراً "
                   "حرفياً لكل عيّنة (تحقّق مباشر)، فـIC=0.0 دقيقاً لا يعني رفض الفرضية بدليل، بل "
                   "أنها لم تُختبَر بعد — راجع حاشية 'سياق عابر للأصول' في خطة المشروع"},
    {"name": "WICK_upper_rejection", "track": "literature_mining", "feature": "WICK_upper",
     "transform": lambda v: -v, "hypothesis": "ذيل علوي طويل إشارة رفض صعود"},
    {"name": "WICK_lower_rejection", "track": "literature_mining", "feature": "WICK_lower",
     "transform": None, "hypothesis": "ذيل سفلي طويل إشارة رفض هبوط"},
]
print(f"{len(CANDIDATE_SIGNALS)} مرشّحاً جاهزاً — أضِف المزيد بنفس الشكل أعلاه.")

### ملحق) مرشّحون إضافيون — تفاعلات + funding rate (المرحلتان ١-٢ من الخطة)

يغطي `EXPLORATORY_CANDIDATES` نمطين لم تغطِّهما `CANDIDATE_SIGNALS` أعلاه:

* **تفاعل بين ميزتين** (`kind="interaction"`، عبر `make_interaction_predict_fn`) —
  مثال "حجم×جسم الشمعة" من قسم "الفرضيات الاستكشافية" في الخطة: حجم غير
  عادي (`VOLZ_20`) مع جسم شمعة كبير (`BODY_ratio`) قد يعني دخول لاعب كبير،
  لا ضجيجاً عادياً — بضرب ميزتين موجودتين أصلاً، بلا عمود جديد في خط الأنابيب.
* **funding rate كمقياس تموضع متطرف** — الفرضية الثالثة في جدول "أساس" بالخطة
  (`FUND_rate_z`، مبنية أصلاً في خط الأنابيب عبر `add_funding_oi_features`
  لكن غير مُفعَّلة في `feature_order` بشكل افتراضي؛ راجع الخطوة التالية في
  README). المرشّح هنا جاهز لأي `dataset` يحمل هذه الميزة فعلياً — سيفشل
  بخطأ واضح (`ValueError`) على بيانات لا تحمل `FUND_rate_z`، لا صمتاً.

In [ ]:
# @title
EXPLORATORY_CANDIDATES = [
    {"name": "VOLZ_x_BODY", "track": "data_driven", "kind": "interaction",
     "feat_a": "VOLZ_20", "feat_b": "BODY_ratio", "op": "mul", "transform": None,
     "hypothesis": "حجم غير عادي × جسم شمعة كبير = دخول لاعب كبير، لا ضجيج عادي"},
    {"name": "FUND_rate_extreme_position", "track": "hypothesis_driven", "feature": "FUND_rate_z",
     "transform": lambda v: -v, "hypothesis": (
         "funding مرتفع جداً = طويلون مفرطون بالرافعة → خطر تصفية → انعكاس هبوطي محتمل "
         "(العكس لـfunding سالب جداً)")},
]
print(f"{len(EXPLORATORY_CANDIDATES)} مرشّحاً استكشافياً إضافياً — "
      "FUND_rate_extreme_position يحتاج dataset يحمل FUND_rate_z فعلياً.")

<cell_type>markdown</cell_type>### ملحق ٢) توليد فرضيات ممنهج — آليات الخطة الخمس، لا "جرّب مؤشراً جديداً"

قسم "طرق توليد فرضيات جديدة" في خطة المشروع يُلاحظ أن المؤشرات القياسية
استُهلكت (RSI، MACD، Bollinger...) لكن **آليات** توليد الفرضية أقلّ استهلاكاً
بكثير من الأدوات نفسها. `GENERATIVE_CANDIDATES` يجسّد ثلاثاً من الخمس بكود
حقيقي (لا وصفاً نظرياً)، كلّ واحدة عبر `kind="custom"`/`"trained"` الجديدين
في `make_candidate_predict_fn` أعلاه:

| الآلية (من الخطة) | المرشّح | الفكرة |
| --- | --- | --- |
| **مقلوب الافتراض الضمني** | `RSI_vol_adjusted_reversion` | RSI يفترض أن عتبتي 30/70 ثابتتان بصرف النظر عن حالة السوق — هنا يُطبَّع الانحراف عن 50 بالتقلب الحالي (`NATR_14`)، فتصير العتبة **نسبية لنظام التقلّب**، لا مطلقة |
| **كسر التناظر عمداً** | `Asymmetric_momentum_downside_weighted` | أغلب المؤشرات تعامل الصعود والهبوط بنفس المعادلة؛ سلوكياً الذعر يتحرّك أسرع من الجشع — هنا معادلتان مختلفتان فعلاً حسب الاتجاه: في الهبوط زخم قصير المدى (`RET_1`)، وفي الصعود زخم أبطأ (`RET_6`) (⚠️ ليس مجرّد تحجيم `RET_1` بوزن ثابت — ذلك تحويل رتيب لا يغيّر ترتيب سبيرمان، فيُعطي IC مطابقاً لـRET_1 خام تماماً؛ استخدام متغيّرين مختلفين حسب الحالة يكسر هذا الرتابة فعلاً) |
| **النقل من مجال مختلف** | `IsolationForest_anomaly_score` | تقنية كشف شذوذ (لا "مؤشر تداول" أصلاً) — Isolation Forest يُدرَّب على train لكل نافذة (`kind="trained"`، بنفس نمط المركَّب Ridge)، ودرجة الشذوذ على test هي التنبؤ: هل الشذوذ الإحصائي نفسه يحمل معلومة اتجاه؟ |

الآليتان المتبقّيتان في الخطة (**تقطير الرؤية المتأخّرة** — يحتاج هدفاً
جديداً كلياً لا مجرّد ميزة، مثال DPO/ATR الموثَّق في الخطة؛ و**تحليل الأخطاء
المنهجية** — يحتاج نموذجاً مُدرَّباً فعلياً لفحص أخطائه) تحتاجان بنية تتجاوز
نطاق "مرشّح بلا تدريب" لهذا الدفتر تحديداً — موثَّقتان هنا كخطوة تالية
صريحة، لا مُنفَّذتين قسراً بشكل مبتور.

In [ ]:
# @title
def _rsi_vol_adjusted(X_last, feature_order):
    """(RSI-50) مطبَّعة بالتقلب الحالي (NATR_14) — عتبة تطرف نسبية للنظام
    السعري، لا 30/70 الثابتة. سالبة (نفترض ارتداداً، لا استمراراً)."""
    rsi = X_last[:, feature_order.index("RSI_14")]
    natr = X_last[:, feature_order.index("NATR_14")]
    return -(rsi - 50.0) / (natr + 1e-6)


def _asymmetric_momentum(X_last, feature_order):
    """في الهبوط (RET_1<0) نعتمد الزخم القصير (RET_1) — استجابة سريعة؛ في
    الصعود نعتمد زخماً أبطأ (RET_6) — تأكيداً أبطأ (الذعر أسرع من الجشع).
    ⚠️ تنبيه للحذر عند تصميم مرشّحين مماثلين: تحجيم بسيط (`ret1 * وزن`، لو
    كان الوزن ثابتاً في كل شقّ) هو تحويل رتيب لـRET_1 عالمياً، وسبيرمان
    (المقياس المُستخدَم في IC هنا) لا يتأثر بتحويل رتيب — أي IC سيتطابق مع
    IC خام RET_1 تماماً رغم اختلاف الشكل. هنا نستخدم متغيّرين مختلفين حسب
    الحالة (تحويل غير رتيب)، فترتيبه لا يطابق ترتيب RET_1 ولا RET_6 وحدهما
    (تحقّق تجريبي على بيانات حقيقية: سبيرمان مع RET_1 ≈0.73، مع RET_6 ≈0.66
    — لا ±1.0 لأيّهما)."""
    ret1 = X_last[:, feature_order.index("RET_1")]
    ret6 = X_last[:, feature_order.index("RET_6")]
    return np.where(ret1 < 0, ret1, ret6)


def make_isolation_forest_predict_fn(features, feature_order=None, contamination=0.1, random_state=42):
    """`kind="trained"`: يُدرِّب Isolation Forest على train (مُطبَّع بمتوسط/
    انحراف train نفسه)، ثم يُرجع درجة الشذوذ (`decision_function`، أعلى =
    أقلّ شذوذاً) على test — نفس نمط make_ridge_composite_predict_fn تماماً،
    لكن بلا هدف (كشف شذوذ غير مُشرَف، لا انحدار)."""
    def predict_fn(train, val, test):
        from sklearn.ensemble import IsolationForest
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = IsolationForest(contamination=contamination, random_state=random_state, n_estimators=100)
        model.fit((Xtr - mu) / sigma)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.decision_function((Xte - mu) / sigma)
    return predict_fn


GENERATIVE_CANDIDATES = [
    {"name": "RSI_vol_adjusted_reversion", "track": "hypothesis_driven", "kind": "custom",
     "fn": _rsi_vol_adjusted, "mechanism": "assumption_inversion",
     "hypothesis": "عتبات RSI الثابتة (30/70) لا تناسب كل نظام تقلّب — تطبيع الانحراف عن 50 "
                   "بالتقلّب (NATR) يلتقط تطرّفاً نسبياً حقيقياً، لا مطلقاً"},
    {"name": "Asymmetric_momentum_downside_weighted", "track": "hypothesis_driven", "kind": "custom",
     "fn": _asymmetric_momentum, "mechanism": "asymmetry_hunting",
     "hypothesis": "الذعر يتحرّك أسرع من الجشع — في الهبوط زخم قصير المدى (RET_1) أدقّ، وفي "
                   "الصعود زخم أبطأ (RET_6) أنسب؛ معادلتان مختلفتان عمداً حسب الاتجاه"},
    {"name": "IsolationForest_anomaly_score", "track": "data_driven", "kind": "trained",
     "builder": lambda feature_order: make_isolation_forest_predict_fn(
         [f for f in feature_order if f != "close"], feature_order=feature_order),
     "mechanism": "analogical_transfer",
     "hypothesis": "شذوذ إحصائي في متجه الميزات قد يعكس حدثاً حقيقياً (تصفية، خبر) يحمل معلومة "
                   "اتجاه، لا ضجيجاً محايداً"},
]
print(f"{len(GENERATIVE_CANDIDATES)} مرشّحات توليد ممنهج — راجع جدول الآليات أعلاه.")

### ملحق ٣) إعادة اختبار أقوى نتائج استكشاف سابق (باكتيست خام ← المحور الصارم)

شارك صاحب المشروع نتائج استكشاف قديم خاص به: `generate_signals` (٧٢ إشارة
شراء من مكتبة مؤشرات `pandas_ta_classic` كاملة تقريباً) اختُبرت فردياً عبر
باكتيست مباشر (ربح/نسبة نجاح/profit factor)، **بلا أي تصحيح لمشكلة
الاختبارات المتعددة** — بالضبط الفخّ الذي تحذّر منه الخطة في قسم "البحث
الموجَّه" (آلاف التوليفات، فبعضها سينجح صدفة حتماً). أقوى النتائج بمعيار
profit_factor: `CCI`≈1.52، `STOCH`/`KDJ`≈1.37 (مُختبَر أصلاً أعلاه كـ
`STOCH_reversion`)، `TSI`≈1.36، `AD`/`WCP`≈1.33، `CMO`≈1.31 — لكن نسبة
النجاح لمعظمها منخفضة جداً (~7%)، نمط مثير للريبة (رابح بصفقات نادرة جداً)
يستحقّ فحص IC/عُشر/خطّ أساس عشوائي لا الاكتفاء بربح إجمالي.

**قيد إعادة الاختبار هنا:** `feature_order` الحالي لا يحمل high/low/volume
الخام (فقط نِسَباً مشتقّة منها: `RANGE_rel`، `BODY_ratio`، `VOLZ`/`VOLR`)،
فمؤشرات تحتاجها مباشرة (`CCI`، `AD`/`OBV`، `WCP`، `UO`، `QSTICK`، Vortex)
**لا يمكن إعادة اختبارها من هذا الدفتر بلا تعديل خط الأنابيب** لإضافتها
كميزات خام — خطوة مؤجَّلة، بنفس منطق تأجيل funding/OI، لا تُنفَّذ إلا بطلب
صريح. فقط `CMO`، `TSI`، و`DPO` (يحتاجان `close` فقط، وهي مُتاحة عبر كل
خطوات النافذة الزمنية لا آخر خطوة فقط) قابلة لإعادة الاختبار الآن، عبر
`kind="series"` الجديد في `make_candidate_predict_fn` أعلاه.

**⚠️ ملاحظة أمانة تقنية (TSI):** `TSI(13,25)` الأصلي (بارامترات
`generate_signals`) لا يتقارب إطلاقاً ضمن `window_size=32` الحالي لخط
الأنابيب (يحتاج نحو ٥١ خطوة إحماء) — قيمته الأخيرة تبقى NaN طوال النافذة،
فتحقّقتُ تجريبياً وعدّلت البارامترات إلى `(5, 13, 5)` (نفس آلية التنعيم
المزدوج، أفق أقصر) لتتقارب ضمن ٣٢ خطوة — **ليست نفس المعايرة الأصلية
المُختبَرة في الباكتيست القديم**، بل تكيّف مضطرّ بسبب طول النافذة، موثَّق
هنا صراحة لا مخفيّاً.

**تحديث DPO — تجربة مباشرة إضافية من صاحب المشروع:** بخلاف "تقطير الرؤية
المتأخّرة" (DPO المُعاد رسمه، `centered=True`، الموثَّق أعلى الدفتر كهدف لا
كمدخل)، هذه تجربة بصيغة `centered=False` **غير المُعاد رسمها** (لا تسرّب
معلومات مستقبلية، قابلة للتنفيذ حيّاً كمدخل مباشر) — اختُبرت فعلياً على
BTC/1H: `DPO(14, centered=False)` أعطى profit_factor=0.89 (خاسر، 78 صفقة)،
بينما `DPO(2, centered=False)` أعطى profit_factor=1.14 (ربح صافٍ موجب، 166
صفقة، نجاح 65.66%) — تباين حادّ بمجرّد تغيير الطول من 14 إلى 2 يستحقّ فحصاً
عبر IC/عُشر/خطّ أساس عشوائي عبر أصول متعدّدة، لا الاكتفاء بنتيجة أصل واحد.
كلا الطولين مُضافان أدناه (`DPO_2_reversion`/`DPO_14_reversion`) للمقارنة
المباشرة، بصيغة عكسية (فرضية ارتداد — DPO متطرف سالب/موجب يعكس، بنفس منطق
استراتيجية صاحب المشروع الأصلية: شراء عند DPO سالب جداً، بيع عند موجب جداً).

In [ ]:
# @title
def _cmo_reversal(series_2d, length=14):
    """CMO (Chande Momentum Oscillator) عبر pandas_ta_classic على كل عيّنة
    على حدة — عكسه لصياغة فرضية ارتداد (CMO متطرف يعكس)، بنفس منطق
    RSI_14_reversion. من أقوى نتائج الاستكشاف القديم (profit_factor≈1.31
    في الباكتيست الخام، بلا تصحيح للاختبارات المتعددة هناك)."""
    import pandas_ta_classic as ta
    out = np.full(series_2d.shape[0], np.nan)
    for i in range(series_2d.shape[0]):
        cmo = ta.cmo(pd.Series(series_2d[i]), length=length)
        if cmo is not None and len(cmo):
            out[i] = cmo.iloc[-1]
    return -out


def _tsi_momentum(series_2d, fast=5, slow=13, signal=5):
    """TSI (True Strength Index) عبر pandas_ta_classic — زخم مزدوج التنعيم،
    بلا عكس (فرضية استمرار، لا ارتداد). بارامترات مُقصَّرة (5/13/5 بدل
    13/25/13 الأصلية في الباكتيست القديم) — راجع "ملحق ٣" أعلاه لسبب هذا
    التكيّف (`window_size=32` لا يكفي لتقارب TSI(13,25))."""
    import pandas_ta_classic as ta
    out = np.full(series_2d.shape[0], np.nan)
    for i in range(series_2d.shape[0]):
        tsi = ta.tsi(pd.Series(series_2d[i]), fast=fast, slow=slow, signal=signal)
        if tsi is not None and len(tsi):
            out[i] = tsi.iloc[-1, 0]
    return out


def make_dpo_reversion(length):
    """DPO (Detrended Price Oscillator) بصيغته `centered=False` — لا إزاحة
    للخلف، بلا تسرّب معلومات مستقبلية، قابل للتنفيذ حيّاً (بخلاف
    `centered=True` المُوثَّق أعلى الدفتر كمثال "تقطير الرؤية المتأخّرة"،
    ذاك يُستخدَم كهدف لا كمدخل). عكسه لصياغة فرضية ارتداد — بنفس منطق
    استراتيجية صاحب المشروع الأصلية (شراء عند DPO سالب جداً). تجربة مباشرة
    من صاحب المشروع على BTC/1H: طول=2 → profit_factor=1.14، طول=14 →
    profit_factor=0.89 (خاسر) — كلا الطولين هنا للمقارنة عبر IC الصارم."""
    import pandas_ta_classic as ta

    def _fn(series_2d):
        out = np.full(series_2d.shape[0], np.nan)
        for i in range(series_2d.shape[0]):
            dpo = ta.dpo(pd.Series(series_2d[i]), length=length, centered=False)
            if dpo is not None and len(dpo):
                out[i] = dpo.iloc[-1]
        return -out
    return _fn


LEGACY_BACKTEST_CANDIDATES = [
    {"name": "CMO_14_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": _cmo_reversal,
     "hypothesis": "CMO متطرف يعكس (من أقوى نتائج باكتيست خام سابق للمشروع، profit_factor≈1.31)"},
    {"name": "TSI_5_13_5_momentum", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": _tsi_momentum,
     "hypothesis": "زخم TSI مزدوج التنعيم يستمر (من أقوى نتائج باكتيست خام سابق، "
                   "profit_factor≈1.36 على معايرة 13/25 الأصلية — هنا معايرة أقصر 5/13/5)"},
    {"name": "DPO_2_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": make_dpo_reversion(2),
     "hypothesis": "DPO(2, centered=False) متطرف يعكس — تجربة مباشرة لصاحب المشروع "
                   "على BTC/1H أعطت profit_factor=1.14"},
    {"name": "DPO_14_reversion", "track": "literature_mining", "kind": "series",
     "feature": "close", "fn": make_dpo_reversion(14),
     "hypothesis": "DPO(14, centered=False) متطرف يعكس — نفس التجربة بطول أطول، "
                   "أعطت profit_factor=0.89 (خاسر) على BTC/1H — للمقارنة المباشرة مع DPO_2"},
]
print(f"{len(LEGACY_BACKTEST_CANDIDATES)} مرشّحاً من إعادة اختبار الاستكشاف القديم — "
      "الباقي (CCI/AD/OBV/WCP/UO/QSTICK/Vortex) يحتاج high/low/volume الخام في feature_order (مؤجَّل).")

### النتيجة الفعلية لإعادة اختبار الاستكشاف القديم (تشغيل حقيقي)

أُجري التقييم فعلياً على نفس بيانات H002 (5 أصول، 12 نافذة)، وسُجِّلت الـ12
مدخلاً (4 مرشّحين × 3 أهداف) في `experiment_registry/registry.json`
الحقيقي — **كلّها `مرفوضة`**، لا مرشّح واحد اجتاز `consistent_sign=True`.

**الأهمّ: `DPO_2_reversion`** — الذي أعطى profit_factor=1.14 (ربح صافٍ
موجب) في تجربة صاحب المشروع المباشرة على أصل واحد (BTC/1H) — **يُظهر IC
شبه معدوم فعلياً عبر الأصول الخمسة** (`close`≈+0.049، `high`≈-0.001،
`low`≈-0.001، لا أي منها معنوي إحصائياً). هذا مثال حيّ ودقيق تماماً لما
حذّرت منه الخطة عن الباكتيست الخام بلا محور تقييم: نتيجة تبدو رابحة على
عيّنة/أصل واحد قد تكون ضجيجاً بحتاً لا يصمد أمام أصول أخرى أو خطّ أساس
عشوائي — **بالضبط الفرق بين "بدا مربحاً" و"إشارة حقيقية"**.

الباقي (`CMO_14_reversion`، `TSI_5_13_5_momentum`، `DPO_14_reversion`)
أضعف من ذلك أو غير متّسق الاتجاه عبر النوافذ. لا شيء من هذه الدفعة يستحقّ
تفعيلاً في الإنتاج حتى الآن.

## ٦) الماسح الآلي — تقييم كل المرشّحين × كل الأهداف دفعة واحدة

In [ ]:
# @title
def scan_candidates(candidates, windows, targets=("close", "high", "low"),
                    feature_order=None, **eval_kwargs):
    """يُقيِّم كل مرشّح × كل هدف عبر evaluate_candidate (مع حارس
    clean_reg_target تلقائياً)، ويُرجع لوحة قيادة (leaderboard) مُرتَّبة —
    اتساق الإشارة أوّلاً، ثم قوة IC المطلقة. لا يتوقّف عند أوّل خطأ (يُسجَّل
    ويُكمل بقية المرشّحين) — مفيد خاصة لمرشّحين قد لا يحملهما كل dataset
    (مثل FUND_rate_extreme_position)."""
    rows = []
    for cand in candidates:
        predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
        for target in targets:
            try:
                report = evaluate_candidate(predict_fn, target, windows, **eval_kwargs)
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "hypothesis": cand.get("hypothesis", ""),
                            "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                            "frac_significant": report["frac_significant"],
                            "consistent_sign": report["consistent_sign"],
                            "n_ok": report["n_ok"], "status": "ok"})
            except Exception as e:
                rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                            "status": f"error: {type(e).__name__}: {e}"})
    df = pd.DataFrame(rows)
    ok = df[df["status"] == "ok"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["consistent_sign", "abs_mean_ic"], ascending=[False, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] != "ok"]], ignore_index=True)
    return df


leaderboard = scan_candidates(
    CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES + LEGACY_BACKTEST_CANDIDATES,
    windows, feature_order=FEATURE_ORDER)
pd.set_option("display.width", 160)
print(leaderboard.to_string(index=False))

### النتيجة الفعلية (تشغيل حقيقي — 5 أصول من `history_1d`، نفس بيانات H002)

أُجري هذا الماسح فعلياً على نفس بيانات H002 (5 أصول، 12 نافذة). **لا مرشّح
واحد من الـ17 حقّق `consistent_sign=True`** — أقوى النتائج (`NATR_14_neg` على
`close`، mean_ic=-0.234؛ `RSI_14_reversion` على `close`، mean_ic=+0.221)
غير متّسقة الاتجاه عبر النوافذ، مطابقة لصعوبة إيجاد إشارة يومية موثوقة التي
وثّقتها H001/H002 (سقف قريب من العملة المعدنية العادلة). **لم تُسجَّل أي
فرضية من هذا التشغيل في `experiment_registry`** — لا شيء عبر عتبة الثقة
(`consistent_sign=True` + معنوية كافية) يستحقّ تسجيلاً؛ راجع القسم ٨ أدناه
لكيفية التسجيل يدوياً حين يتوفّر مرشّح يستحقّه.

## ٧) البحث التركيبي الرخيص (`data_driven`/`genetic_search`) — بلا شبكة عصبية

مرشّح مركَّب: انحدار Ridge خطّي (`RidgeCV`، يُحسَب مغلقاً بلا حِقَب — أجزاء
من الثانية لكل نافذة) على مجموعة الميزات كلّها معاً، بدل مؤشر واحد. ليست
شبكة عصبية ولا تدريباً تكرارياً — أرخص بآلاف المرّات من تدريب نموذج NIG-TimeNet
لكل نافذة (راجع H002)، لكنها قد تلتقط تفاعلات بين الميزات لا يلتقطها أي
مرشّح فردي أعلاه.

In [ ]:
# @title
from sklearn.linear_model import RidgeCV

COMPOSITE_FEATURES = [c["feature"] for c in CANDIDATE_SIGNALS if c["feature"] != "MKT_ret_1"]
# ✅ extract_feature_matrix مُعرَّفة مرّة واحدة في قسم ٤ (إطار المرشّح) —
# يُعاد استخدامها هنا كما هي، ومن make_isolation_forest_predict_fn لاحقاً.


def make_ridge_composite_predict_fn(features, target, feature_order=None, alphas=(0.1, 1.0, 10.0, 100.0)):
    """يُدرِّب RidgeCV على train (مغلق، بلا حِقَب) متنبّئاً بـclean_reg_target
    لنفس target، ثم يُنبئ على test — نفس عقد predict_fn المُستخدَم مع
    evaluate_candidate."""
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        Xtr = extract_feature_matrix(train_flat, features, feature_order=feature_order)
        ytr = clean_reg_target(train_flat, target)
        mu, sigma = Xtr.mean(axis=0), Xtr.std(axis=0) + 1e-9
        model = RidgeCV(alphas=alphas)
        model.fit((Xtr - mu) / sigma, ytr)
        Xte = extract_feature_matrix(test, features, feature_order=feature_order)
        return model.predict((Xte - mu) / sigma)
    return predict_fn


composite_rows = []
for target in ("close", "high", "low"):
    pf = make_ridge_composite_predict_fn(COMPOSITE_FEATURES, target, feature_order=FEATURE_ORDER)
    report = evaluate_candidate(pf, target, windows)
    composite_rows.append({"name": "ridge_composite_all_features", "target": target,
                           "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                           "frac_significant": report["frac_significant"],
                           "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(composite_rows).to_string(index=False))

### النتيجة الفعلية للمركَّب

على نفس البيانات: `close` mean_ic=+0.041 (غير معنوي)، `high` mean_ic=+0.209،
`low` mean_ic=+0.139 — **كلاهما `consistent_sign=False`**. لا تحسّن ذا شأن عن
أفضل مرشّح فردي، ولا اجتياز لعتبة القبول. **تنبيه مهم لمن يُعيد هذا الفحص:**
أوّل تشغيل لهذا المركَّب (قبل تطبيق حارس `clean_reg_target`) أعطى نتائج
خارقة زائفة (`high` mean_ic=+0.595، `consistent_sign=True`!) — وهذا بالضبط
ما كشف أثر مرجع "نفس النوع" الموثَّق أعلى الدفتر. أي نتيجة مستقبلية على
high/low أعلى من هذا المدى المتواضع تستحقّ فحصاً مضاعَفاً قبل تصديقها، لا
احتفالاً فورياً.

## ٨) المُشغّل الدفعي (Batch Runner) — تقييم متوازٍ + تسجيل تلقائي

طبقاً لـ"خطة بناء النظام — تسريع التجارب" في خطة المشروع، هذا يكمل الركيزتين
المتبقيتين من الأربع (الأخريان — واجهة موحّدة للفرضية، وذاكرة تخزين مؤقت —
مُلبّاتان أصلاً: `make_candidate_predict_fn` هو الواجهة الموحّدة، و`dataset`/
`windows` يُبنيان مرّة واحدة ويُعاد استخدامهما لكل مرشّح، بلا إعادة تحميل أو
إعادة حساب ميزات — بنفس روح `_checkpoint_fingerprint` في دفتر التحضير):

* **التوازي**: `imap_ordered`/`default_workers` من دفتر التحضير نفسه (لا
  إعادة كتابة) — خيوط، لا عمليات منفصلة (نفس مبرر التحضير: numpy/pandas
  تُحرِّر GIL).
* **معيار قبول موحّد** (`classify_result`): `مقبولة` فقط إن `consistent_sign=True`
  و`frac_significant` ≥ حدّ أدنى (0.34 افتراضياً — أي أكثر من ثلث النوافذ
  معنوية بنفس اتجاه المتوسط، دون تعسّف رقم أعلى بلا مبرّر). أقلّ من حدّ أدنى
  من النوافذ الناجحة (`n_ok`) → `قيد الاختبار` (لا حكم بعد). غير ذلك → `مرفوضة`.
* **تسجيل تلقائي**: كل (مرشّح × هدف) يُسجَّل في `experiment_registry` بمعرّف
  `SCAN_{اسم}_{هدف}` — **تمييز مهم**: هذه سجلّات آلية خفيفة من مسح دفعي، لا
  فرضيات مُنسَّقة يدوياً بعمق كـH001/H002 (تلك تبقى بمراجعة بشرية وتوثيق
  أوسع لكل شذوذ/ملاحظة). كلاهما في نفس السجلّ، والفائدة نفسها: يمنع اختبار
  نفس المرشّح مرتين لاحقاً بعد نسيان أنه فشل.

التسجيل اليدوي (بنمط H001/H002) يبقى الخيار الصحيح لأي فرضية تستحقّ تحليلاً
أعمق (شذوذ، مقارنة بخطّ أساس، تفسير سلوكي) — راجع تلك الدفاتر كمرجع للنمط.

In [ ]:
# @title
def classify_result(report, min_frac_significant=0.34, min_n_ok=5):
    """معيار قبول موحّد — نفس المنطق يُطبَّق بصرف النظر عن مصدر المرشّح
    (راجع "كيف تُقيَّم نتائج كل هذه الأدوات" في خطة المشروع). `n_ok` أقلّ من
    الحدّ الأدنى يعني عدد نوافذ ناجحة غير كافٍ للحكم أصلاً — لا "مرفوضة"
    مُتسرِّعة على دليل ضعيف."""
    if report.get("n_ok", 0) < min_n_ok:
        return "قيد الاختبار"
    if report.get("consistent_sign") and report.get("frac_significant", 0) >= min_frac_significant:
        return "مقبولة"
    return "مرفوضة"


def _batch_eval_job(job):
    cand, target, windows_, feature_order, eval_kwargs = job
    predict_fn = make_candidate_predict_fn(cand, feature_order=feature_order)
    try:
        report = evaluate_candidate(predict_fn, target, windows_, **eval_kwargs)
        return {"cand": cand, "target": target, "report": report, "status": "ok"}
    except Exception as e:
        return {"cand": cand, "target": target, "status": "error", "error": f"{type(e).__name__}: {e}"}


def run_batch_and_register(candidates, windows, targets=("close", "high", "low"), feature_order=None,
                           id_prefix="SCAN", max_workers=None, registry_path=None, **eval_kwargs):
    """المُشغّل الدفعي الكامل: يقيّم كل (مرشّح × هدف) بالتوازي عبر
    imap_ordered/default_workers (من دفتر التحضير)، يصنّف كل نتيجة عبر
    classify_result، ويسجّلها تلقائياً في experiment_registry. يُرجع
    (leaderboard, registered_ids) — الأولى للعرض السريع، والثانية لتتبّع ما
    كُتب فعلاً.

    ``registry_path``: مرّره (مثلاً tempfile) لتوجيه التسجيل بعيداً عن السجلّ
    الحقيقي — مفيد للاختبار الذاتي؛ اتركه ``None`` للمسار الافتراضي الحقيقي.
    """
    jobs = [(cand, target, windows, feature_order, eval_kwargs) for cand in candidates for target in targets]
    n_workers = max_workers or default_workers(len(jobs))

    rows, registered = [], []
    for res in imap_ordered(_batch_eval_job, jobs, max_workers=n_workers):
        cand, target = res["cand"], res["target"]
        if res["status"] == "error":
            rows.append({"name": cand["name"], "track": cand["track"], "target": target,
                        "status": "error", "error": res["error"]})
            continue
        report = res["report"]
        status = classify_result(report)
        hyp_id = f"{id_prefix}_{cand['name']}_{target}"
        register_hypothesis(
            hyp_id=hyp_id,
            hypothesis=f"{cand.get('hypothesis', cand['name'])} (هدف: {target})",
            source=cand["track"],
            status=status,
            report={"per_window": report["per_window"], "mean_ic": report["mean_ic"],
                    "std_ic": report["std_ic"], "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]},
            notes=(f"مُسجَّلة آلياً عبر run_batch_and_register. mean_ic={report['mean_ic']:.4f}, "
                  f"consistent_sign={report['consistent_sign']}, "
                  f"frac_significant={report['frac_significant']:.2f}."),
            registry_path=registry_path,
        )
        registered.append(hyp_id)
        rows.append({"name": cand["name"], "track": cand["track"], "target": target, "status": status,
                    "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                    "frac_significant": report["frac_significant"],
                    "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})

    df = pd.DataFrame(rows)
    ok = df[df["status"] != "error"].copy()
    if len(ok):
        ok["abs_mean_ic"] = ok["mean_ic"].abs()
        ok = ok.sort_values(["status", "abs_mean_ic"], ascending=[True, False])
        df = pd.concat([ok.drop(columns="abs_mean_ic"), df[df["status"] == "error"]], ignore_index=True)
    return df, registered


# مثال استخدام حقيقي (يكتب في experiment_registry الحقيقي — شغّله عمداً، لا تلقائياً):
# batch_leaderboard, registered_ids = run_batch_and_register(
#     CANDIDATE_SIGNALS + EXPLORATORY_CANDIDATES + GENERATIVE_CANDIDATES + LEGACY_BACKTEST_CANDIDATES,
#     windows, feature_order=FEATURE_ORDER)
# print(batch_leaderboard.to_string(index=False))
# print(f"سُجِّل {len(registered_ids)} مدخلاً: {registered_ids}")

### النتيجة الفعلية لتشغيل المُشغّل الدفعي الكامل (على السجلّ الحقيقي)

شُغِّل `run_batch_and_register` فعلياً (لا مثالاً معلَّقاً) على نفس بيانات
H002 (5 أصول، 12 نافذة) — **21 مرشّحاً قابلاً للتقييم × 3 أهداف = 63 مدخلاً
سُجِّلت في `experiment_registry/registry.json` الحقيقي** (المرشّح الثاني
والعشرون، `FUND_rate_extreme_position`، فشل بخطأ واضح على الأهداف الثلاثة
كما هو مصمَّم — `FUND_rate_z` غير مُفعَّلة في هذا الـdataset، فلم يُسجَّل).

**كل الـ64 مدخلاً (63 + `H002_nig_timenet_classification_head` سابقاً)
بحالة `مرفوضة`** — لا مرشّح واحد اجتاز `classify_result` (اتساق الاتجاه +
معنوية كافية) على هذه العيّنة من 5 أصول. هذا يُكمل **بوّابة خروج المرحلتين
١-٢** في خطة المشروع حرفياً ("توثيق كل نتيجة، مقبولة أو مرفوضة — لا حاجة
لنتيجة إيجابية للانتقال، فقط لدليل موثَّق"): بوّابة الخروج **اجتيزت
بالتوثيق**، لا بإيجاد إشارة. طبقاً لنص الخطة، هذا "مؤشّر مهم (لا نهائي) على
أن ميزات إضافية من نفس العائلة (OHLCV+مشتقّاتها المباشرة) لن تضيف كثيراً" —
يرفع أولوية إمّا (أ) توسيع عيّنة الأصول قبل الحكم النهائي (5 أصول عيّنة
صغيرة)، أو (ب) الانتقال للمرحلة ٣ في الخطة (Matrix Profile، SHAP
interactions، العنقدة — أدوات بلا قواعد مسبقة)، وليس تكرار مزيد من مرشّحين
من نفس العائلة. **قرار الانتقال يبقى صريحاً بيد صاحب المشروع، لا انزلاقاً
تلقائياً** (نفس مبدأ "الترتيب الزمني" في الخطة).

## ٩) المرحلة ٣ — أدوات اكتشاف بلا قواعد مسبقة (Matrix Profile + SHAP)

بقرار صريح من صاحب المشروع (بعد أن رفضت المرحلتان ١-٢ كل الفرضيات — ٧٦
مدخلاً موثَّقاً في السجلّ حتى الآن)، ننتقل للمرحلة ٣ في الخطة: أدوات لا
تفترض شكل الإشارة مسبقاً، بل تكتشف البنية من البيانات نفسها. كل أداة هنا
تُنتج مرشّحاً، لا إشارة مؤكَّدة — تمرّ عبر نفس محور IC + عُشر + خطّ أساس
عشوائي قبل أي قرار، مطابقةً لمبدأ "لا فرق في المعاملة بين نمط اكتشفته شجرة
قرار ونمط اقترحه حدسك" في الخطة.

### ٩-أ) Matrix Profile — تنبؤ بأقرب نافذة تاريخية مشابهة شكلياً

الأساس الرياضي لـMatrix Profile: بحث أقرب جار (k-NN) بمسافة إقليدية بعد
تطبيع-Z، بلا افتراض مسبق عن شكل الزخرفة (motif). محسوب هنا مباشرة (لا عبر
مكتبة `stumpy`) لأن بيانات هذا الدفتر مُقسَّمة مسبقاً لنوافذ منفصلة عبر
`rolling_splits`، لا سلسلة خام متصلة واحدة لكل أصل تصلح لمسح `stumpy`
التقليدي عبر حدود النوافذ — نفس النتيجة الرياضية (بحث أقرب جار
z-normalized Euclidean)، بلا افتعال حدود اصطناعية عبر دمج نوافذ منفصلة في
سلسلة واحدة. `target`-محدَّد كالمركَّب Ridge أعلاه (يحتاج عائد train
الفعلي كـ"مكتبة" للتنبؤ)، فيُقيَّم يدوياً لكل هدف على حدة، لا عبر
`scan_candidates` تلقائياً.

In [ ]:
# @title
def make_matrix_profile_predict_fn(target, feature="close", feature_order=None, k=3):
    """أساس Matrix Profile: بحث أقرب جار (k-NN) بمسافة إقليدية بعد تطبيع-Z
    — لكل نافذة اختبار، تُطبَّع نافذة `feature` (افتراضياً `close`)، تُقارَن
    بمكتبة نوافذ train كلّها، ويُتنبَّأ بمتوسط العائد الفعلي (`target`) الذي
    تلا أقرب k نافذة تاريخياً مشابهة شكلياً. راجع الشرح أعلاه لسبب الحساب
    المباشر بدل `stumpy.mass`."""
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        train_series = extract_feature_series(train_flat, feature, feature_order=feature_order)
        train_targets = clean_reg_target(train_flat, target)
        test_series = extract_feature_series(test, feature, feature_order=feature_order)

        def znorm(a):
            mu = a.mean(axis=1, keepdims=True)
            sd = a.std(axis=1, keepdims=True) + 1e-9
            return (a - mu) / sd

        Ztr, Zte = znorm(train_series), znorm(test_series)
        preds = np.empty(len(Zte))
        for i in range(len(Zte)):
            d = np.linalg.norm(Ztr - Zte[i], axis=1)
            k_eff = min(k, len(d))
            nn_idx = np.argpartition(d, k_eff - 1)[:k_eff]
            preds[i] = train_targets[nn_idx].mean()
        return preds
    return predict_fn


matrix_profile_rows = []
for target in ("close", "high", "low"):
    pf = make_matrix_profile_predict_fn(target, feature="close", feature_order=FEATURE_ORDER, k=3)
    report = evaluate_candidate(pf, target, windows)
    matrix_profile_rows.append({"name": "matrix_profile_knn3_close_shape", "target": target,
                                "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                                "frac_significant": report["frac_significant"],
                                "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(matrix_profile_rows).to_string(index=False))

### النتيجة الفعلية لـMatrix Profile

على نفس بيانات H002 (5 أصول، 12 نافذة): `close` mean_ic=-0.019، `high`
mean_ic=+0.012، `low` mean_ic=-0.028 — **ضعيفة جداً وغير معنوية عملياً على
الثلاثة**، `consistent_sign=False`. أقرب k=3 نافذة تاريخية مشابهة شكلياً
لسعر الإغلاق وحده لا تحمل معلومة اتجاه مفيدة في هذا الإعداد — **مرفوضة**.
لم تُختبَر بعد: أشكال مبنية على أكثر من `close` (مثلاً نافذة الشمعة الكاملة
عبر عدّة ميزات معاً)، أو قيم `k` أخرى — خطوة تالية محتملة لا مُنفَّذة الآن.

### ٩-ب) SHAP interaction values — اكتشاف تفاعلات آلياً بلا تحديد يدوي

أداة اكتشاف لا تنبؤ (كما تنص الخطة): تُدرَّب غابة عشوائية ضحلة (عمق ٤) على
كل الميزات الخام معاً لنافذة `train` واحدة، ثم تُفحَص SHAP interaction
values — تكشف أي أزواج ميزات تتفاعل فعلياً في قرارات الشجرة آلياً، بخلاف
`VOLZ_x_BODY` أعلاه (تفاعل حُدِّد يدوياً بحدس سلوكي). **تُشغَّل مرّة واحدة
فقط** (على `train` النافذة الأولى، هدف `close`) لفصل "الاكتشاف" عن
"التحقّق" — الأزواج المُكتشَفة تُختبَر لاحقاً عبر كل النوافذ الـ12 كمرشّحين
عاديين (`kind="interaction"`)، لا على نفس بيانات الاكتشاف (يمنع تسرّباً
دائرياً بين مصدر الفرضية ومحكّ قبولها).

In [ ]:
# @title
!pip install -q shap 2>/dev/null


def discover_shap_interaction_pairs(train, target, feature_order, top_k=5, max_depth=4,
                                    n_estimators=200, random_state=42):
    """يُدرِّب RandomForestRegressor ضحلاً على متجه الميزات الكامل (آخر خطوة
    زمنية، عبر extract_feature_matrix)، يحسب SHAP interaction values
    (`shap.TreeExplainer`)، ويُرجع أقوى top_k أزواج ميزات (بمعزل عن القطر —
    الأثر الرئيسي المنفرد لكل ميزة، لا تفاعلاً) مرتّبة تنازلياً بمقدار
    التفاعل المطلق المتوسط."""
    from sklearn.ensemble import RandomForestRegressor
    import shap
    train_flat = concat_splits(train)
    X = extract_feature_matrix(train_flat, feature_order=feature_order)
    y = clean_reg_target(train_flat, target)
    model = RandomForestRegressor(max_depth=max_depth, n_estimators=n_estimators,
                                  random_state=random_state, n_jobs=-1).fit(X, y)
    explainer = shap.TreeExplainer(model)
    interaction_values = explainer.shap_interaction_values(X)
    mean_abs = np.abs(interaction_values).mean(axis=0)
    np.fill_diagonal(mean_abs, 0.0)
    F = len(feature_order)
    pairs = [(i, j, mean_abs[i, j]) for i in range(F) for j in range(i + 1, F)]
    pairs.sort(key=lambda t: t[2], reverse=True)
    return [(feature_order[i], feature_order[j], float(mag)) for i, j, mag in pairs[:top_k]]


shap_train, _shap_val, _shap_test = windows[0]
discovered_pairs = discover_shap_interaction_pairs(shap_train, "close", FEATURE_ORDER, top_k=5)
print("أقوى 5 أزواج تفاعل (SHAP interaction values، نافذة ١ فقط، هدف close):")
for a, b, mag in discovered_pairs:
    print(f"  {a} × {b}: {mag:.5f}")

SHAP_DISCOVERED_CANDIDATES = [
    {"name": f"SHAP_{a}_x_{b}", "track": "data_driven", "kind": "interaction",
     "feat_a": a, "feat_b": b, "op": "mul", "transform": None,
     "hypothesis": f"اكتشاف آلي عبر SHAP interaction values (غابة عشوائية ضحلة على train النافذة ١) — "
                   f"{a}×{b} من أقوى الأزواج تفاعلاً (مقدار≈{mag:.4f})"}
    for a, b, mag in discovered_pairs
]
print(f"\n{len(SHAP_DISCOVERED_CANDIDATES)} مرشّح تفاعل مُكتشَف آلياً — يُختبَر أدناه عبر كل النوافذ الـ12.")

shap_leaderboard = scan_candidates(SHAP_DISCOVERED_CANDIDATES, windows, feature_order=FEATURE_ORDER)
print(shap_leaderboard.to_string(index=False))

### النتيجة الفعلية لـSHAP interactions

على نفس بيانات H002: SHAP اكتشف آلياً (على `train` النافذة ١ فقط، هدف
`close`) أن `VOLZ_20` يتفاعل مع أربع ميزات أخرى أكثر من أي زوج آخر —
`NATR_14`، `BODY_ratio`، `VOLR_12_48`، `RET_3`، `STOCHk_14_3_3` — منطقي
سلوكياً (فورة حجم تُفسَّر بالتقلّب/شكل الشمعة/الزخم القصير معاً، لا بمعزل
عنها). عند اختبار الأزواج الخمسة عبر كل النوافذ الـ12: **لا شيء اجتاز
`consistent_sign=True`**، لكن `NATR_14×VOLZ_20` على `high` هو **أقرب نتيجة
لعتبة القبول في هذا الدفتر بأكمله** — `mean_ic=+0.225`، `frac_significant
=0.333` (تحت الحدّ الأدنى 0.34 بفارق ضئيل جداً)، رغم `consistent_sign
=False`. **مرفوضة رسمياً بمعيار `classify_result`، لكنها الأقرب من بين كل
ما اختُبر في هذا الدفتر** — تستحقّ تكراراً على عيّنة أصول أوسع قبل الحسم
النهائي، لا تفعيلاً فورياً. باقي الأزواج (`BODY_ratio`/`VOLR_12_48`/
`RET_3`/`STOCHk_14_3_3` × `VOLZ_20`) أضعف بوضوح.

### ٩-ج) أهمية ميزات مُجمَّعة (RF+GB+XGB) — من استكشاف سابق لصاحب المشروع

شارك صاحب المشروع دفتر استكشاف `xgboost.ipynb` قديماً خاصاً به: يُدرِّب
Random Forest + Gradient Boosting + XGBoost معاً على ~150 مؤشر خام
(تصنيف اتجاه الشمعة التالية)، ثم يطبع أهمّ 25 ميزة بمتوسط `feature_
importances_` الثلاثة موزونة بأوزان الدمج المُحسَّنة على validation.
**تقييم الطريقة الأصلية:** دقتها 51-54% وAUC 0.51-0.55 عبر كل تشغيلاتها —
ضعيفة جداً (قريبة من التخمين العشوائي)، **متّسقة تماماً مع كل ما وجدناه في
هذا المشروع بمنهجية مختلفة كلياً** (دليل مستقلّ إضافي على صعوبة إشارة
الاتجاه اليومي/الشمعة القادمة)، لكن التقييم نفسه ضعيف منهجياً: تقسيم
زمني واحد فقط (لا `rolling_splits`)، بلا تصحيح للاختبارات المتعددة، وأهمية
الميزات غير مُتحقَّق من ثباتها عبر فترات مختلفة.

**الفكرة القابلة لإعادة الاستخدام** (لا الأرقام، بل الآلية): إجماع أهمية
ميزات من عدّة نماذج مختلفة الطبيعة (أشجار مستقلّة/معزَّزة تدريجياً/معزَّزة
تدرّجياً بضبط أدقّ) أكثر متانة من نموذج واحد — نفس روح "غابة عشوائية ضحلة +
SHAP" أعلاه، لكن على مستوى **الميزة المفردة** لا التفاعل الثنائي، وبإجماع
ثلاثة نماذج بدل واحد. مُطبَّقة هنا بانضباط هذا الدفتر: انحدار لا تصنيف
(نتوافق مع `clean_reg_target` الصارم لا هدفاً ثنائياً)، اكتشاف مرّة واحدة
فقط على `train` النافذة الأولى (لا كل نافذة)، والميزات المُكتشَفة تُختبَر
لاحقاً عبر كل النوافذ كمرشّحين عاديين — نفس فصل "الاكتشاف عن التحقّق"
المُطبَّق مع SHAP.

In [ ]:
# @title
!pip install -q xgboost 2>/dev/null


def discover_ensemble_feature_ranking(train, target, feature_order, top_k=10, random_state=42):
    """إجماع أهمية ميزات من ثلاثة نماذج مختلفة الطبيعة (RandomForestRegressor
    + GradientBoostingRegressor + XGBRegressor)، بمتوسط `feature_importances_`
    الثلاثة بالتساوي. انحدار على `clean_reg_target` (لا تصنيف ثنائي كالأصل
    في `xgboost.ipynb`) ليتوافق مع صرامة هذا الدفتر. تُشغَّل مرّة واحدة فقط
    (على `train` النافذة الأولى) — نفس فصل الاكتشاف عن التحقّق المُطبَّق
    مع SHAP أعلاه."""
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    from xgboost import XGBRegressor
    train_flat = concat_splits(train)
    X = extract_feature_matrix(train_flat, feature_order=feature_order)
    y = clean_reg_target(train_flat, target)
    rf = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=20,
                               random_state=random_state, n_jobs=-1).fit(X, y)
    gb = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05,
                                   random_state=random_state).fit(X, y)
    xgbr = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=random_state,
                        n_jobs=-1, tree_method="hist", verbosity=0).fit(X, y)
    imp = (rf.feature_importances_ + gb.feature_importances_ + xgbr.feature_importances_) / 3.0
    order = np.argsort(imp)[::-1][:top_k]
    return [(feature_order[i], float(imp[i])) for i in order]


ens_train, _ens_val, _ens_test = windows[0]
ranked_features = discover_ensemble_feature_ranking(ens_train, "close", FEATURE_ORDER, top_k=10)
print("أقوى 10 ميزات (إجماع RF+GB+XGB، نافذة ١ فقط، هدف close):")
for name, imp in ranked_features:
    print(f"  {name}: {imp:.5f}")

ENSEMBLE_DISCOVERED_CANDIDATES = [
    {"name": f"ENSFI_{name}", "track": "data_driven", "feature": name, "transform": None,
     "hypothesis": f"اكتشاف آلي عبر إجماع أهمية ميزات RF+GB+XGB (train النافذة ١) — {name} من "
                   f"أقوى 10 ميزات (أهمية≈{imp:.4f})"}
    for name, imp in ranked_features
]
print(f"\n{len(ENSEMBLE_DISCOVERED_CANDIDATES)} مرشّح مُكتشَف آلياً — يُختبَر أدناه عبر كل النوافذ.")

ensemble_leaderboard = scan_candidates(ENSEMBLE_DISCOVERED_CANDIDATES, windows, feature_order=FEATURE_ORDER)
print(ensemble_leaderboard.to_string(index=False))

### النتيجة الفعلية لأهمية الميزات المُجمَّعة

على نفس بيانات H002: الإجماع (RF+GB+XGB على `train` النافذة ١، هدف
`close`) صنّف `VOLZ_20` أولاً بفارق واضح (أهمية 0.111 مقابل 0.068 للثانية،
`WICK_lower`) — **نفس الميزة التي اكتشفتها SHAP كمحور التفاعلات أعلاه،
بأداة مختلفة كلياً وبلا أي تنسيق بينهما** — تقارب مستقلّ يستحقّ الانتباه
حتى لو لم يترجم بعد لمرشّح مقبول. سُجِّلت الـ30 مدخلاً (10 ميزات × 3
أهداف) في `experiment_registry` الحقيقي، **كلّها `مرفوضة` رسمياً**، لكن
`RANGE_rel` على `high` نتيجة لافتة: `consistent_sign=True` (نادر في هذا
الدفتر) بـ`mean_ic=+0.220`، لكن `frac_significant=0.083` بعيد جداً عن
الحدّ الأدنى 0.34 — اتجاه ثابت لكن ضعيف الحدّة، لا يكفي للقبول. `NATR_14`
(0.234/0.141/0.119 على close/high/low) و`BBB_20_2.0` أيضاً من أقوى
النتائج رقمياً لكن `consistent_sign=False` لكليهما.

### ٩-د) اختبار تاريخي واسع (WIDEHIST) — إغلاق سؤال "العيّنة الصغيرة"

كل النتائج أعلاه (الأقسام ٦-٩) استخدمت `max_windows=12` فقط — أي ما يقارب
سنة واحدة من النوافذ المتحرّكة (بداية أوّل نافذة نوفمبر ٢٠٢١، آخرها مارس
٢٠٢٢). هذا فتح سؤالاً مُتكرِّراً منذ H002: هل النتائج السلبية (رفض كل
الفرضيات) بسبب ضعف حقيقي في الإشارات، أم بسبب عيّنة صغيرة جداً (12 نافذة
فقط، على 5 أصول مترابطة الحركة أصلاً)؟

**الاكتشاف**: نفس بيانات الـ5 أصول المخزَّنة محلياً (`history_1d`) تمتدّ
فعلياً لسنوات أطول بكثير مما استُخدم — SOLUSDT حتى نوفمبر ٢٠٢٠، والبقية
حتى ٢٠٢٤/٢٠٢٥ — بلا حاجة لأي جلب شبكي جديد (طلبات مباشرة لـ
`api.binance.com`/`fapi.binance.com` مرفوضة أصلاً بسياسة الشبكة هنا،
تأكَّد ذلك عبر `curl` مباشرة: `connect_rejected`/403).

رفع `max_windows` من 12 إلى 60 (بقيّة الإعدادات كما هي:
`test_span="30D"`, `initial_train_span="365D"`, `step="30D"`) أنتج **56
نافذة فعلية** تمتدّ من 2022-02-08 حتى 2026-09-14 — أي كامل التاريخ
المتاح فعلياً، لا استطالة تعسّفية.

**ما أُعيد اختباره على الـ56 نافذة**:
- كل مرشّحي الأقسام ٥-٧ (`CANDIDATE_SIGNALS` + `EXPLORATORY_CANDIDATES`
  + `GENERATIVE_CANDIDATES` + `LEGACY_BACKTEST_CANDIDATES` من ملحق ٣) —
  26 مرشّحاً × 3 أهداف.
- 5 أزواج تفاعل SHAP المُكتشَفة في القسم ٩-ب.
- مرشّح Matrix Profile (القسم ٩-أ).

**النتيجة**: **93 تجربة `WIDEHIST_*` مُسجَّلة، كلّها بحالة "مرفوضة"** —
ولا تجربة واحدة منها حقّقت `consistent_sign=True` عبر الـ56 نافذة (أعلى
قيمة `|mean_ic|` هي 0.149 لـ`VOLZ_volume_shock/high`، لكن بإشارة غير
ثابتة عبر النوافذ، تماماً كنمط كل التجارب السابقة في هذا المشروع).

هذا **يُغلق سؤال "العيّنة الصغيرة" نهائياً**: النتيجة السلبية ليست أثراً
لِـ12 نافذة أو لتاريخ قصير — هي نفسها عبر 56 نافذة تمتدّ نحو 4.5 سنوات من
كل التاريخ المتاح فعلياً لهذه الأصول الخمسة. أي فرضية مستقبلية تحتاج إمّا
بيانات أصول إضافية غير مترابطة، أو آلية اكتشاف مختلفة جذرياً — لا مجرّد
نافذة أطول على نفس الأصول.

### ٩-هـ) العنقدة غير المُشرَفة (K-means / HDBSCAN) — استكمال المرحلة ٣

آخر أداة من "المرحلة ٣" في خطة المشروع (راجع "العنقدة غير المُشرَفة" هناك):
تُجمِّع نوافذ شكل السعر (`close` بعد تطبيع-Z، نفس تحويل Matrix Profile أعلاه)
في عناقيد متشابهة الشكل على `train`، ثم تفحص: **هل عضوية عنقود معيّن ترتبط
بعائد مستقبلي مختلف عن البقية؟** التنبؤ لكل عيّنة اختبار = متوسط العائد
الفعلي لأعضاء نفس العنقود في `train` (بدل قيمة خام مباشرة كما في المرشّحين
السابقين). خوارزميتان مقارنتان عمداً:

* **K-means** (`k=5` ثابت): كل نقطة تنضمّ إجبارياً لأقرب مركز — خط أساس بسيط.
* **HDBSCAN**: لا يفترض عدد عناقيد مسبقاً، ويترك النقاط غير المتماسكة "ضجيجاً"
  (`label=-1`) بلا تصنيف بدل إجبارها على عنقود لا تنتمي إليه فعلياً — أنسب
  نظرياً لبيانات مالية نادراً ما تُشكِّل كتلاً كروية نظيفة (نقاط الضجيج تأخذ
  متوسط `train` العام بدل متوسط عنقود وهمي).

اختُبرت الخوارزميتان مباشرة على نطاق **WIDEHIST** (56 نافذة، كامل التاريخ
المتاح — القسم ٩-د أعلاه) بدل نطاق 12 نافذة الافتراضي لهذا الدفتر: بما أن
سؤال "هل الرفض بسبب عيّنة صغيرة؟" أُغلق نهائياً هناك، إعادة الاختبار على
نطاق أضعف إحصائياً كانت ستضيف عملاً بلا معلومة جديدة.

In [ ]:
# @title
def make_cluster_regime_predict_fn(target, feature="close", feature_order=None, algo="kmeans",
                                    n_clusters=5, min_cluster_size=None, random_state=42):
    # حلقة يدوية بنفس نمط Matrix Profile/Ridge composite أعلاه — يحتاج
    # target صراحةً فلا يلائم بناء builder(feature_order) العام في
    # make_candidate_predict_fn. يُدرَّب على train (تجميع + متوسط عائد كل
    # عنقود)، ثم يُسقِط test على نفس العناقيد.
    def predict_fn(train, val, test):
        train_flat = concat_splits(train)
        train_series = extract_feature_series(train_flat, feature, feature_order=feature_order)
        train_targets = clean_reg_target(train_flat, target)
        test_series = extract_feature_series(test, feature, feature_order=feature_order)

        def znorm(a):
            mu = a.mean(axis=1, keepdims=True)
            sd = a.std(axis=1, keepdims=True) + 1e-9
            return (a - mu) / sd

        Ztr, Zte = znorm(train_series), znorm(test_series)
        global_mean = train_targets.mean()

        if algo == "kmeans":
            from sklearn.cluster import KMeans
            k_eff = max(2, min(n_clusters, len(Ztr) // 5))
            model = KMeans(n_clusters=k_eff, random_state=random_state, n_init=10)
            train_labels = model.fit_predict(Ztr)
            test_labels = model.predict(Zte)
        elif algo == "hdbscan":
            import hdbscan
            mcs = min_cluster_size or max(5, len(Ztr) // 20)
            model = hdbscan.HDBSCAN(min_cluster_size=mcs, prediction_data=True)
            train_labels = model.fit_predict(Ztr)
            test_labels, _ = hdbscan.approximate_predict(model, Zte)
        else:
            raise ValueError(f"algo غير مدعوم: {algo}")

        # نقاط الضجيج (label=-1 في train) لا تُشكِّل عنقوداً حقيقياً — تُستبعَد
        # من قاموس المتوسطات، فيرث أي test تُسقَط عليها (أو على عنقود غير
        # موجود في train) متوسط train العام بدل متوسط وهمي.
        cluster_mean = {lbl: train_targets[train_labels == lbl].mean()
                        for lbl in np.unique(train_labels) if lbl != -1}
        return np.array([cluster_mean.get(lbl, global_mean) for lbl in test_labels])
    return predict_fn


cluster_rows = []
for algo, algo_name in (("kmeans", "KMeans5_close_shape"), ("hdbscan", "HDBSCAN_close_shape")):
    for target in ("close", "high", "low"):
        pf = make_cluster_regime_predict_fn(target, feature="close", feature_order=FEATURE_ORDER, algo=algo)
        report = evaluate_candidate(pf, target, windows)
        cluster_rows.append({"name": algo_name, "target": target,
                             "mean_ic": report["mean_ic"], "std_ic": report["std_ic"],
                             "frac_significant": report["frac_significant"],
                             "consistent_sign": report["consistent_sign"], "n_ok": report["n_ok"]})
print(pd.DataFrame(cluster_rows).to_string(index=False))

### النتيجة الفعلية للعنقدة (تشغيل حقيقي — نطاق WIDEHIST، 56 نافذة)

**6 تجارب `WIDEHIST_Cluster_*` مُسجَّلة، كلّها `مرفوضة`**، ولا واحدة منها
`consistent_sign=True`:

| الخوارزمية | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| KMeans (k=5) | close | -0.037 | 0.071 | False |
| KMeans (k=5) | high | +0.042 | 0.071 | False |
| KMeans (k=5) | low | +0.038 | 0.143 | False |
| HDBSCAN | close | -0.067 | 0.089 | False |
| HDBSCAN | high | -0.024 | 0.107 | False |
| HDBSCAN | low | +0.006 | 0.143 | False |

أعلى `|mean_ic|` هو 0.067 فقط (HDBSCAN/close) — أضعف بكثير من أي مرشّح مقبول
في هذا الدفتر، وبإشارة متذبذبة تماماً عبر النوافذ (`consistent_sign=False`
للجميع، تماماً كنمط كل نتائج المرحلة ٣). لا حاجة لتفسير سلوكي بأثر رجعي هنا
(بوّابة الخروج المذكورة في خطة المشروع) لأن النتيجة أصلاً لا تحمل نمطاً
يستحقّ التفسير — لا فرق منهجي بين HDBSCAN (يترك الضجيج بلا تصنيف) وK-means
(يُجبر كل نقطة على الانضمام): كلاهما فشل بنفس الدرجة تقريباً، وهو دليل إضافي
(لا حاسم) على أن **شكل نافذة `close` وحدها** (بمعزل عن باقي الميزات) لا يحمل
معلومة تنبّئية كافية، بصرف النظر عن طريقة تجميعها.

**بهذا تكتمل المرحلة ٣ من الخطة (Matrix Profile + SHAP interactions +
العنقدة)** — كل أدواتها الثلاث اختُبرت على النطاق التاريخي الكامل المتاح
(56 نافذة)، وكلّها بلا استثناء رفضت كل الفرضيات المُختبَرة عبرها.

## ١٠) اختبار رخيص لفرضية "الترابط العابر للأصول" — قبل أي معمارية مشتركة

فكرة استُقبِلت من صاحب المشروع: بدل اعتبار الارتباط العالي بين الأصول
الخمسة عائقاً إحصائياً فقط (راجع ملاحظة H002)، **استغلاله** عبر معمارية
نموذج مشترك — نموذج فرعي لكل عملة ينتهي بطبقة تجميع مشتركة تُخرج متجهاً
بحجم عدد العملات، بحيث يستفيد التنبؤ بكل عملة من حركة بقيّة العملات في نفس
اللحظة. قبل الالتزام بتصميم معماري جديد يحتاج تدريباً كاملاً لتقييمه (تكلفة
حوسبة عالية بلا GPU)، هذا اختبار رخيص لا يحتاج شبكة عصبية إطلاقاً: هل
معلومة بسيطة عن الأصول الأخرى (متوسط عائدها في نفس اللحظة) تحمل أي ارتباط
حقيقي بعائد كل عملة على حدة؟ إن لم تحمل حتى أبسط صيغة لهذه المعلومة أي IC،
فهذا مؤشّر (لا حاسم) على أن معمارية أعقد ستُصارع لاستخراج شيء من عدم.

**ملاحظة مهمة**: ميزة "القوة النسبية لـBTC" تحديداً (`MKT_beta`) غير قابلة
للاختبار حالياً — بيانات BTC نفسها غير متوفّرة محلياً (فقط الأصول الخمسة
alt، وBinance محجوبة بسياسة الشبكة). لكن الفكرة الأعمّ — تبادل معلومات بين
الأصول الخمسة أنفسهم، بلا حاجة لـBTC — قابلة للاختبار الآن بالكامل ببيانات
موجودة فعلاً.

**التصميم**: لكل عيّنة اختبار لأصل A عند لحظة t، يُجمَع `RET_1`/`RET_6`
(معلومة متزامنة متاحة فعلياً وقت القرار، لا مستقبلية) لبقيّة الأصول B≠A
**عند نفس اللحظة t بالضبط** (محاذاة بالطابع الزمني الفعلي في `last_candles`،
لا بالترتيب)، ثم يُؤخَذ متوسطها كتنبّؤ. هذا يحتاج شكل بيانات خاصاً
(`rolling_splits(..., keep_asset_test_separate=True)`) بخلاف كل مرشّحي هذا
الدفتر — الشكل المُجمَّع الافتراضي (`keep_asset_test_separate=False`) لا
يُبقي أي أثر لحدود الأصول داخل `test`/`train` (تحقّق مباشر من مصدر `_take`
في خط الأنابيب)، فيستحيل معرفة أي الصفوف تخصّ أي عملة منه.

**قيد فعلي مهم اكتُشف أثناء التنفيذ**: بما أن الأصول الخمسة لا تشترك كلّها
في نفس المدى الزمني (SOLUSDT منذ 2020، والبقية منذ 2024/2025)، أغلب نوافذ
WIDEHIST المبكرة (٢٠٢٢-أوائل ٢٠٢٤) تحوي **أصلاً واحداً فقط** في `test` —
لا معنى لـ"عائد الأصول الأخرى" فيها فتُستبعَد تلقائياً (`evaluate_windows`
يتجاهل نوافذ بعيّنات صالحة أقل من الحدّ الأدنى، لا يُسقِط التقييم كلّه).
تداخل حقيقي بين الأصول (تراكب كامل 30/30 يوماً) يبدأ فعلياً من النافذة ٢٥
تقريباً (~2024) وحتى نهاية WIDEHIST.

In [ ]:
# @title
def make_cross_asset_predict_fn(feature="RET_1", feature_order=None, agg="mean"):
    # يحتاج test = {اسم_الأصل: قسم} (keep_asset_test_separate=True) — يبني
    # قاموس بحث لكل أصل آخر (الطابع الزمني -> قيمة feature)، ثم لكل عيّنة
    # في كل أصل يُتوسَّط قيمة بقيّة الأصول عند نفس الطابع الزمني بالضبط
    # (NaN إن لم يوجد أي أصل آخر بنفس اللحظة تماماً — تُستبعَد لاحقاً في
    # حساب IC، لا تُصفَّر).
    def predict_fn(train, val, test):
        asset_feat, asset_ts = {}, {}
        for name, split in test.items():
            asset_feat[name] = extract_feature_last_value(split, feature=feature, feature_order=feature_order)
            asset_ts[name] = np.asarray(split["last_candles"])[:, TS_COL]

        preds = []
        for name in test.keys():
            ts = asset_ts[name]
            n = len(ts)
            other_names = [o for o in test if o != name]
            other_lookup = {o: dict(zip(asset_ts[o].tolist(), asset_feat[o].tolist())) for o in other_names}
            out = np.full(n, np.nan)
            for i in range(n):
                t = ts[i]
                vals = [other_lookup[o][t] for o in other_names if t in other_lookup[o]]
                if vals:
                    out[i] = float(np.mean(vals)) if agg == "mean" else float(np.median(vals))
            preds.append(out)
        return np.concatenate(preds)
    return predict_fn


# ملاحظة تشغيل: يحتاج windows_sep = rolling_splits(dataset, ..., keep_asset_test_separate=True)
# (لا windows الافتراضية أعلاه — راجع الشرح فوق) و evaluate_candidate العام (يقبل test بأي شكل
# طالما predict_fn يتعامل معه بنفسه). النتائج الفعلية أدناه أُنتِجت على نطاق WIDEHIST (56 نافذة).

### النتيجة الفعلية (تشغيل حقيقي — نطاق WIDEHIST، 56 نافذة، keep_asset_test_separate=True)

**33 من 56 نافذة صالحة** (البقية أصل واحد فقط، مُستبعَدة تلقائياً كما هو
متوقَّع). **6 تجارب `WIDEHIST_CrossAsset_*` مُسجَّلة، كلّها `مرفوضة`**:

| الميزة | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| متوسط RET_1 للأصول الأخرى | close | +0.009 | 0.091 | False |
| متوسط RET_1 للأصول الأخرى | high | -0.037 | 0.182 | False |
| متوسط RET_1 للأصول الأخرى | low | +0.100 | 0.242 | False |
| متوسط RET_6 للأصول الأخرى | close | -0.038 | 0.061 | False |
| متوسط RET_6 للأصول الأخرى | high | -0.042 | 0.242 | False |
| متوسط RET_6 للأصول الأخرى | low | -0.002 | 0.242 | False |

أعلى `|mean_ic|` هو 0.0996 فقط (RET_1/low)، بإشارة متذبذبة (`consistent_sign
=False` للجميع) — نفس نمط كل نتائج المشروع حتى الآن. **الخلاصة لصالح فرضية
المعمارية المشتركة**: أبسط صيغة ممكنة لمعلومة عابرة للأصول (متوسط عائد
الأصول الأربعة الأخرى في نفس اللحظة) **لا تحمل ارتباطاً خطياً/رتبياً قابلاً
للاستغلال** مع عائد أي هدف. هذا **لا يستبعد نهائياً** أن معمارية عميقة قد
تكتشف تفاعلاً غير خطي أعقد (مثلاً: تفاعل مشروط بالتقلّب، أو نمط قيادة-تبعية
بين زوج أصول محدَّد بدل متوسط الأربعة كلّهم) — اختبار IC خطي/رتبي بسيط لا
يستطيع كشف ذلك بالتصميم. لكنه **يرفع عبء الإثبات** على أي استثمار في معمارية
معقّدة ومكلفة تدريباً: لا دليل أوّلي رخيص يدعمها بعد، والأصول الخمسة نفسها
صغيرة العدد ومترابطة أصلاً (سبيرمان زوجي ≈0.64 بين عوائدها، H002) — عدد
"إشارات مستقلّة" فعلي محدود جداً بصرف النظر عن تعقيد المعمارية.

## ١١) 🎯 أول فرضية مقبولة من هذا الإطار — عيّنة موسّعة (50 أصلاً، تفعيل `market_context`)

استُغِلّ توفّر مجلد `history_1d` على Drive بمئات العملات (لا 5 فقط) لبناء
عيّنة أوسع بكثير وأقلّ ترابطاً: **50 أصلاً حقيقياً** (الأصول الخمسة
الأصلية + BTCUSDT + 44 عملة راسخة متنوّعة — ETH، XRP، ADA، DOGE، DOT، LTC،
ATOM، AVAX، LINK، AAVE، وغيرها)، مع تفعيل `market_context.enabled=True`
صراحةً (يُصلح خلل `MKT_beta` الموثَّق في حاشية خطة المشروع) و`BTCUSDT`
كعملة مرجعية حقيقية. النتيجة: **94,961 عيّنة، 37 ميزة، 30 نافذة متحرّكة**
(بدل 12-56 نافذة على 5 أصول فقط سابقاً).

أُعيد اختبار كل المرشّحين القياسيين الـ26 × 3 أهداف (٧٥ تجربة `EXPANDED_*`
جديدة) — **و ظهرت لأول مرّة في تاريخ هذا المشروع بأكمله 4 نتائج `مقبولة`
رسمياً** (لا `قيد الاختبار` ولا `مرفوضة`):

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `NATR_neg_vol` (`-NATR_14`) | high | **-0.197** | **93.3%** | **True (30/30)** |
| `RSI_vol_adjusted_reversion` (`-(RSI_14-50)/NATR_14`) | high | **-0.198** | **93.3%** | **True (30/30)** |
| `NATR_neg_vol` | low | **+0.164** | **96.7%** | **True (30/30)** |
| `RSI_vol_adjusted_reversion` | low | **+0.167** | **96.7%** | **True (30/30)** |

**تحقّق من المتانة**: فحصتُ الجدول الكامل لكل نافذة (30 نافذة) للمرشّح
`NATR_neg_vol/high` — **كل نافذة على حدة سالبة الإشارة بلا استثناء واحد**
(من -0.018 إلى -0.39)، والقيمة تشتدّ تدريجياً مع النوافذ الأحدث (حين يتوفّر
عدد أصول أكبر حيّة معاً، فيقلّ ضجيج القياس) — نمط متّسق مع أثر حقيقي
متزايد الوضوح مع القوة الإحصائية، لا صدفة ناتجة عن نافذة أو أصل شاذّ
واحد.

**⚠️ تحذير منهجي مهم قبل الاحتفال**: النتيجتان المقبولتان **ليستا
اكتشافين مستقلّين فعلياً** — `NATR_neg_vol` هي ببساطة `-NATR_14` (تقلّب
مُطبَّع)، و`RSI_vol_adjusted_reversion` هي `-(RSI_14-50)/NATR_14` (RSI
مقسوماً على نفس `NATR_14`). تقارب حجم الأثر شبه التام بينهما (-0.197 مقابل
-0.198) يشير إلى أن **`NATR_14` نفسه هو المحرِّك الأساسي لكلا الإشارتين**،
لا تأكيداً مستقلّاً من مؤشّرين مختلفين. الفرضية الفعلية الواحدة التي تستحقّ
التسجيل الرسمي: **"تقلّب مرتفع حالياً (NATR_14) يسبق ارتداداً في نطاق
الشمعة القادمة (`high` أدنى، `low` أعلى) — أي انكماش نطاق الحركة بعد فورة
تقلّب"** — تفسير سلوكي معقول (تفريغ التقلّب/استنفاد الزخم بعد حركة حادّة)،
لا نمط بلا معنى.

**الخطوة التالية (قرار صريح مطلوب من صاحب المشروع، لا تنفيذ تلقائي)**:
هذه أول نتيجة في المشروع بأكمله تجتاز بوّابة القبول رسمياً — تستحقّ تسجيلاً
كاملاً بمنهجية H001/H002 في `signal_evaluation_axis (3).ipynb` (فرضية H003)
مع اختبارات إضافية قبل الاعتماد عليها فعلياً: (أ) هل تصمد على تحويل رتيب
مختلف لتفكيك أثر `NATR_14` عن التطبيع بـRSI تحديداً؟ (ب) هل تصمد بعد تصحيح
الاختبارات المتعددة (~26 مرشّحاً × 3 أهداف = 78 اختباراً في هذا التشغيل
وحده)؟ (ج) هل تنتقل فعلاً لتحسين حقيقي في تدريب `NIG-TimeNet v2` الكامل
عند إضافتها كميزة، أم تبقى IC نظرياً بلا أثر عملي على النموذج؟

## ١٢) مرشّح للاختبار القادم — فركتالات الانعكاس (Williams Fractals)

يفحص هذا المرشّح سؤال H003 المفتوح (ب) من زاوية مستقلّة تماماً عن
`NATR_14`: هل "الارتداد بعد تطرّف محلي" أثر حقيقي أوسع من مجرّد تقلّب
مُطبَّع، أم خاص بـ`NATR_14`/`RSI` تحديداً؟ الفركتال (Bill Williams) يحدّد
نقاط تطرّف محلي مباشرة من `high`/`low` الخام — بلا أي علاقة رياضية بـ
`NATR_14` أو `RSI` — فإن أظهر `consistent_sign=True` أيضاً، هذا دليل
مستقلّ حقيقي على نفس فكرة H003 (لا تكراراً لها)؛ وإن لم يُظهر، هذا يُضيّق
التفسير نحو أن `NATR_14` تحديداً (لا "التطرّف المحلي" عموماً) هو المحرِّك.

**الصيغة المقترَحة (من صاحب المشروع)**:

```python
window = 5
df['fractal_high'] = (df['high'] == df['high'].rolling(window, center=True).max()).astype(int)
df['fractal_low'] = (df['low'] == df['low'].rolling(window, center=True).min()).astype(int)
```

**⚠️ عائقان يجب حلّهما قبل التشغيل الفعلي — موثَّقان هنا صراحةً بدل تنفيذ الصيغة كما وردت بلا تدقيق:**

1. **`high`/`low` مُستبعَدتان من `feature_order` افتراضياً.** `DEFAULT_CONFIG["exclude_from_features"] = ["open", "high", "low", "volume"]` في `crypto_data_pipeline_v6.ipynb` (السبب الموثَّق هناك: ترابط 0.99 مع `close` بعد التطبيع، تُغطّى معلومتهما بـ`RANGE_rel`/`BODY_ratio`/`WICK_*`). الحلّ **بسيط ولا يحتاج ميزة جديدة في خط الأنابيب**: `exclude_features` قابلة للتراجع الجزئي — يكفي بناء `dataset` القادم بـ`exclude_from_features=["open", "volume"]` (إبقاء `high`/`low` فقط، لا استعادة الأربعة) ليصبح كلٌّ من `extract_feature_series(split, "high", ...)` و`extract_feature_series(split, "low", ...)` صالحاً فوراً بنفس نمط `kind="series"` المستخدَم أصلاً لـCMO/TSI/DPO في "ملحق ٣".
2. **`center=True` تُعيد رسم الماضي (نفس فخّ DPO الموثَّق في خطة المشروع، قسم "تقطير الرؤية المتأخّرة").** فركتال عند الخطوة `t` يحتاج معرفة `high`/`low` حتى `t+window//2` — غير متاحة فعلياً عند اتخاذ القرار عند `t`. الصيغة أدناه **لا تستخدم الفركتال عند آخر خطوة** (undefined فعلياً)، بل عند **آخر خطوة "مؤكَّدة"** — أي `t = T-1-window//2` — التي أصبح فيها `window//2` من "المستقبل" اللازم لتأكيدها متاحاً فعلياً ضمن النافذة المُشاهَدة نفسها (نفس حلّ "تأخير شمعة واحدة" الذي أبقى نتائج DPO ممتازة في الخطة).

In [ ]:
# @title
def _fractal_reversal_signal(high_2d, low_2d, window=5):
    """فركتال ويليامز على آخر خطوة "مؤكَّدة" فقط (T-1-window//2)، لا آخر
    خطوة في النافذة — تفادياً لتسرّب معلومة مستقبلية عبر center=True (راجع
    الخلية النصية أعلاه). -1 = آخر تطرّف مؤكَّد كان قمّة (نتوقّع ارتداداً
    هبوطياً)، +1 = كان قاعاً (نتوقّع ارتداداً صعودياً)، 0 = لا تطرّف عندها."""
    if window % 2 == 0:
        raise ValueError("window يجب أن يكون فردياً (مركز واضح لكل جهة).")
    half = window // 2
    confirmed_idx = high_2d.shape[1] - 1 - half
    if confirmed_idx < half:
        raise ValueError(
            f"طول النافذة الزمنية ({high_2d.shape[1]}) أقصر من اللازم "
            f"لتأكيد فركتال بعرض {window} — لا نقطة زمنية صالحة."
        )
    out = np.zeros(high_2d.shape[0])
    for i in range(high_2d.shape[0]):
        h, l = high_2d[i], low_2d[i]
        lo, hi = confirmed_idx - half, confirmed_idx + half + 1
        is_high = h[confirmed_idx] == h[lo:hi].max()
        is_low = l[confirmed_idx] == l[lo:hi].min()
        if is_high and not is_low:
            out[i] = -1.0
        elif is_low and not is_high:
            out[i] = 1.0
    return out


def make_fractal_reversal_predict_fn(feature_order=None, window=5):
    """`kind="series"` مزدوج (high وlow معاً) — يفشل بخطأ صريح إن لم تحمل
    dataset الحالية high/low فعلياً (راجع العائق ١ أعلاه)، لا صمتاً بصفر."""
    def predict_fn(train, val, test):
        test_flat = concat_splits(test)
        fo = feature_order if feature_order is not None else test_flat.get("feature_order")
        if not fo or "high" not in fo or "low" not in fo:
            raise ValueError(
                "fractal_reversal يحتاج 'high' و'low' في feature_order — "
                "أعد بناء dataset بـ exclude_from_features=['open', 'volume'] "
                "(بدل القائمة الافتراضية التي تستبعد high/low أيضاً)."
            )
        high_2d = extract_feature_series(test_flat, "high", feature_order=fo)
        low_2d = extract_feature_series(test_flat, "low", feature_order=fo)
        return _fractal_reversal_signal(high_2d, low_2d, window=window)
    return predict_fn


FRACTAL_REVERSAL_CANDIDATE = {
    # kind="trained" مُعاد استخدامه هنا لآلية "builder(feature_order=...) →
    # predict_fn" فقط (نفس نقطة التفرّع في make_candidate_predict_fn) — لا
    # تدريب فعلياً، بل إغلاق (closure) على window الثابتة، تماماً كنمط
    # make_ridge_composite_predict_fn/make_isolation_forest_predict_fn.
    "name": "Fractal_reversal_w5", "track": "literature_mining", "kind": "trained",
    "builder": make_fractal_reversal_predict_fn,
    "hypothesis": (
        "تطرّف محلي مؤكَّد في high/low (فركتال ويليامز، عرض 5) يسبق ارتداداً "
        "— اختبار مستقلّ عن NATR_14/RSI لسؤال H003 المفتوح (ب): هل الارتداد "
        "بعد تطرّف أثر عام أم خاص بالتقلّب المُطبَّع تحديداً؟"
    ),
}
print("✅ Fractal_reversal_w5 جاهز — يحتاج dataset مبنيّاً بـ "
      "exclude_from_features=['open', 'volume'] (لا الافتراضي) ليُشغَّل فعلياً؛ "
      "لم يُشغَّل بعد على بيانات حقيقية في هذه الجلسة.")

## ١٣) اختبار مرشّحي pandas_ta المؤجَّلين سابقاً — نتيجة سلبية شاملة

قسم "ملحق ٣" أعلاه أجّل صراحةً `CCI`، `AD`/`OBV`، `WCP`، `UO`، `QSTICK`، `Vortex` لأنها تحتاج `high`/`low`/`volume` الخام، غير المتوفّرة وقتها في `feature_order` (مُستبعَدة افتراضياً في `crypto_data_pipeline_v6.ipynb`). المسح الشامل لكل مؤشرات `pandas_ta_classic` (`pandas_ta_full_survey.ipynb`) أثبت أن 192 من 193 مؤشراً يعملان رياضياً بلا مشاكل عبر 14 عملة حقيقية متنوّعة — لكنه اختبار **نجاح الاستدعاء فقط**، لا قيمة تنبّئية. هذا القسم يسدّ الفجوة: اختبار عيّنة فعلية من مرشّحي `SURVEY_CANDIDATES_ROBUST` عبر محور `signal_evaluation_axis` الصارم على بيانات حقيقية.

**الإعداد**: أُعيد بناء dataset حقيقي (14 عملة متنوّعة نفسها المُستخدَمة في المسح — BTC/ETH/XRP/ADA/DOGE/DOT/LTC/ATOM/AVAX/LINK/AAVE/TRX/ZEC/JASMY، 31,604 عيّنة) بـ`exclude_from_features=[]` (بلا استبعاد — `open`/`high`/`low`/`volume` الخام كلّها متاحة الآن)، مع 12 نافذة متحرّكة (`test_span=30D`, `val_span=15D`, `initial_train_span=365D`, `step=30D`، أفق أقصر من H003 عمداً — عيّنة تصفية أوّلية رخيصة، لا الحكم النهائي).

**المرشّحون المُختبَرون (9، مقصورون على مؤشّرات نسبية/محدودة المدى — لا مطلقة القيمة كـ`ATR`/`AD`/`OBV`/`WCP`/`QSTICK` الخام، التي تحمل مشكلة مقارنة عبر أصول مختلفة السعر تماماً كما حذّرت فئة `overlap` في المسح، وتحتاج تطبيعاً أوّلاً — خطوة تالية منفصلة، لا هذا الاختبار)**: `CCI`، `WILLR`، `VORTEX` (VTXP)، `DX`، `UO`، `CTI`، `SKEW`، `FISHER`، `RVI` — كلّها بفترة 14 (أو fast/slow الكلاسيكية 12/26 لـUO).

### النتيجة الفعلية (تشغيل حقيقي — 14 عملة، 12 نافذة، 27 توليفة مرشّح×هدف)

| المرشّح | الهدف | mean_ic | std_ic | frac_significant | consistent_sign | n_ok |
|---|---|---|---|---|---|---|
| `CCI_14` | close | -0.0356 | 0.1209 | 25% | ❌ False | 12 |
| `CCI_14` | high | -0.0122 | 0.1175 | 25% | ❌ False | 12 |
| `CCI_14` | low | -0.0174 | 0.1271 | 25% | ❌ False | 12 |
| `WILLR_14` | close | -0.0471 | 0.1234 | 25% | ❌ False | 12 |
| `WILLR_14` | high | -0.0535 | 0.0971 | 42% | ❌ False | 12 |
| `WILLR_14` | low | -0.0089 | 0.1229 | 25% | ❌ False | 12 |
| `VORTEX_14` | close | -0.0539 | 0.0852 | 33% | ❌ False | 12 |
| `VORTEX_14` | high | -0.0108 | 0.1220 | 25% | ❌ False | 12 |
| `VORTEX_14` | low | -0.0467 | 0.0983 | 33% | ❌ False | 12 |
| `DX_14` | close | +0.0243 | 0.0738 | 17% | ❌ False | 12 |
| `DX_14` | high | +0.0607 | 0.0574 | 17% | ❌ False | 12 |
| `DX_14` | low | -0.0638 | 0.0922 | 33% | ❌ False | 12 |
| `UO_12_26` | close | -0.0696 | 0.0880 | 25% | ❌ False | 12 |
| `UO_12_26` | high | -0.0471 | 0.1165 | 25% | ❌ False | 12 |
| `UO_12_26` | low | -0.0504 | 0.1087 | 33% | ❌ False | 12 |
| `CTI_14` | close | -0.0529 | 0.1080 | 25% | ❌ False | 12 |
| `CTI_14` | high | -0.0006 | 0.1165 | 25% | ❌ False | 12 |
| `CTI_14` | low | -0.0781 | 0.1156 | 33% | ❌ False | 12 |
| `SKEW_14` | close | -0.0221 | 0.0773 | 25% | ❌ False | 12 |
| `SKEW_14` | high | +0.0033 | 0.1157 | 17% | ❌ False | 12 |
| `SKEW_14` | low | -0.0423 | 0.1108 | 33% | ❌ False | 12 |
| `FISHER_14` | close | -0.0647 | 0.0936 | 42% | ❌ False | 12 |
| `FISHER_14` | high | -0.0130 | 0.1453 | 25% | ❌ False | 12 |
| `FISHER_14` | low | -0.0902 | 0.0756 | 58% | ❌ False | 12 |
| `RVI_14` | close | -0.0853 | 0.0954 | 58% | ❌ False | 12 |
| `RVI_14` | high | -0.0814 | 0.0982 | 50% | ❌ False | 12 |
| `RVI_14` | low | -0.0452 | 0.1053 | 42% | ❌ False | 12 |

**النتيجة: 27 من 27 توليفة `مرفوضة`** — لا واحدة منها `consistent_sign=True` (المعيار: `consistent_sign=True` و`frac_significant≥0.34` و`n_ok≥5`). بعضها أظهر `frac_significant` مرتفعاً نسبياً بمعزل (`FISHER_14/low`=58%، `RVI_14/close`=58%) لكن بلا اتساق إشارة عبر النوافذ — بالضبط النمط الذي تحذّر منه الخطة: دلالة إحصائية جزئية بلا اتجاه ثابت لا تكفي للقبول.

هذا يغلق فرعاً استكشافياً كاملاً كان مؤجَّلاً منذ "ملحق ٣": إتاحة `high`/`low`/`volume` الخام لم تكشف إشارة جديدة في هذه العائلة من المؤشرات (تذبذب/اتجاه نسبي محدود المدى) — يتّسق مع النمط العام لهذا المشروع بأكمله: الإشارة الوحيدة المقبولة حتى الآن (H003) مصدرها التقلّب (`NATR_14`)، لا مؤشرات الزخم/الاتجاه التقليدية.

### تسجيل الدفعة كاملة في سجلّ التجارب

In [ ]:
# @title
PANDAS_TA_DEFERRED_RESULTS = [
    ("CCI_14", {'length': 14}, "close", -0.035556, 0.120899, 0.25, 12),
    ("CCI_14", {'length': 14}, "high", -0.012228, 0.117548, 0.25, 12),
    ("CCI_14", {'length': 14}, "low", -0.017408, 0.127051, 0.25, 12),
    ("WILLR_14", {'length': 14}, "close", -0.047091, 0.123429, 0.25, 12),
    ("WILLR_14", {'length': 14}, "high", -0.053535, 0.097087, 0.416667, 12),
    ("WILLR_14", {'length': 14}, "low", -0.008946, 0.12289, 0.25, 12),
    ("VORTEX_14", {'length': 14}, "close", -0.053949, 0.085214, 0.333333, 12),
    ("VORTEX_14", {'length': 14}, "high", -0.010756, 0.12201, 0.25, 12),
    ("VORTEX_14", {'length': 14}, "low", -0.046734, 0.098275, 0.333333, 12),
    ("DX_14", {'length': 14}, "close", 0.024275, 0.07376, 0.166667, 12),
    ("DX_14", {'length': 14}, "high", 0.060734, 0.057382, 0.166667, 12),
    ("DX_14", {'length': 14}, "low", -0.063752, 0.092245, 0.333333, 12),
    ("UO_12_26", {'fast': 12, 'slow': 26}, "close", -0.069622, 0.08804, 0.25, 12),
    ("UO_12_26", {'fast': 12, 'slow': 26}, "high", -0.047053, 0.11651, 0.25, 12),
    ("UO_12_26", {'fast': 12, 'slow': 26}, "low", -0.05036, 0.108659, 0.333333, 12),
    ("CTI_14", {'length': 14}, "close", -0.052915, 0.107985, 0.25, 12),
    ("CTI_14", {'length': 14}, "high", -0.000637, 0.116455, 0.25, 12),
    ("CTI_14", {'length': 14}, "low", -0.078096, 0.115599, 0.333333, 12),
    ("SKEW_14", {'length': 14}, "close", -0.022056, 0.077349, 0.25, 12),
    ("SKEW_14", {'length': 14}, "high", 0.003345, 0.115745, 0.166667, 12),
    ("SKEW_14", {'length': 14}, "low", -0.04229, 0.110815, 0.333333, 12),
    ("FISHER_14", {'length': 14}, "close", -0.064674, 0.093576, 0.416667, 12),
    ("FISHER_14", {'length': 14}, "high", -0.012965, 0.145321, 0.25, 12),
    ("FISHER_14", {'length': 14}, "low", -0.090164, 0.075584, 0.583333, 12),
    ("RVI_14", {'length': 14}, "close", -0.085308, 0.0954, 0.583333, 12),
    ("RVI_14", {'length': 14}, "high", -0.081411, 0.098172, 0.5, 12),
    ("RVI_14", {'length': 14}, "low", -0.045162, 0.10533, 0.416667, 12)
]

# RVI/FISHER: أعلى frac_significant عبر عدّة أهداف معاً (لا نافذة معزولة) رغم
# فشل اختبار الاتساق — محور IC الخطّي ضعيف عمداً أمام نمط مشروط (يتفعّل في
# نظام سوقي معيّن فقط)، قد يستغلّه نموذج غير خطّي كامل. تُوثَّق هنا صراحةً
# كمرشّحين "ضعيفين" يستحقّان اختباراً كميزتين فعليتين داخل NIG-TimeNet v2،
# لا محور IC وحده — راجع docs/research/pandas_ta_deferred_candidates.md.
FLAGGED_FOR_FULL_MODEL_TEST = {"RVI_14", "FISHER_14"}

for name, params, target, mean_ic, std_ic, frac_sig, n_ok in PANDAS_TA_DEFERRED_RESULTS:
    notes = (
        "مُسجَّلة من تشغيل حقيقي فعلي (14 عملة متنوّعة، 31,604 عيّنة، 12 نافذة "
        "متحرّكة، exclude_from_features=[]) — لا بيانات وهمية. consistent_sign=False "
        "يمنع القبول بصرف النظر عن frac_significant. مقصورة على مؤشّرات نسبية/"
        "محدودة المدى (لا ATR/AD/OBV/WCP/QSTICK الخام — تلك تحتاج تطبيعاً أوّلاً "
        "قبل أي اختبار، بنفس مشكلة فئة overlap الموثَّقة في pandas_ta_full_survey.ipynb)."
    )
    if name in FLAGGED_FOR_FULL_MODEL_TEST:
        notes += (
            " ⚠️ مُعلَّمة كمرشّح 'ضعيف موثَّق' يستحقّ إعادة اختبار لاحقة: frac_significant "
            "مرتفع نسبياً عبر أكثر من هدف لنفس المؤشّر (راجع الجدول في "
            "docs/research/pandas_ta_deferred_candidates.md) رغم رفضه هنا بمعيار المحور "
            "الصارم — احتمال علاقة مشروطة بنظام السوق لا يكتشفها IC الخطّي المنفرد. "
            "الخطوة التالية (قرار صريح مطلوب): إضافته كميزة سببية فعلية في feature_order "
            "وقياس أثره داخل تدريب NIG-TimeNet v2 كامل، لا إعادة اختباره عبر المحور بمعزل."
        )
    register_hypothesis(
        hyp_id=f"PANDAS_TA_DEFERRED_{name}_{target}",
        hypothesis=(
            f"{name} ({params}) على هدف {target} — من مرشّحي pandas_ta المؤجَّلين "
            "سابقاً (يحتاجون high/low/volume الخام، متاحة الآن)؛ راجع signal_discovery_lab.ipynb القسم ١٣ للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": False, "n_ok": n_ok,
        },
        notes=notes,
    )

print(f'✅ سُجِّلت {len(PANDAS_TA_DEFERRED_RESULTS)} توليفة — كلّها مرفوضة '
      f'({sum(1 for r in PANDAS_TA_DEFERRED_RESULTS if r[0] in FLAGGED_FOR_FULL_MODEL_TEST)} '
      'منها معلَّمة لإعادة اختبار عبر نموذج غير خطّي كامل).')

## ١٤) الدفعة الثانية من مرشّحي pandas_ta — أول نتيجتين تعبران معيار المحور

استكمالاً للقسم ١٣ (الدفعة الأولى، 9 مرشّحين، 27/27 مرفوضة): 40 مرشّحاً جديداً من بقية `SURVEY_CANDIDATES_ROBUST` (بفترة 14 أو بارامترات pandas_ta الافتراضية)، عبر فئات momentum/trend/volatility/volume/statistics/cycles/performance، على نفس dataset الـ14 عملة ونفس 12 نافذة (`real_dataset.pkl`/`windows.pkl` من القسم ١٣ — بلا إعادة بناء).

**تبسيط منهجي جديد**: بدل البحث عن الحدّ الأدنى الدقيق من الأعمدة الخام لكل مؤشر (كما في القسم ١٣)، مُرِّرت الأعمدة الخام الخمسة كلّها (`open/high/low/close/volume`) موحّداً لكل مرشّح — pandas_ta يقرأ فقط ما يحتاجه بالاسم ويتجاهل الباقي، وكلّها متوفّرة فعلاً في هذا الـdataset.

**اكتشاف تقني مهمّ أثناء التشغيل**: قيمة `close` الأخيرة (`[-1]`) في نافذة كل عيّنة تُساوي **صفراً دائماً** — تعريف `extract_feature_series` يُرسي الشمعة الحالية عند صفر (مرجع نسبي، لا سعراً مطلقاً). هذا يجعل أيّ مؤشر تُحسب قيمته الأخيرة كدالة مباشرة في `close` الحالي (قسمة عليه أو نسبة إليه) **غير قابل للاختبار بهذا الشكل**: إمّا `inf` (يُعامَل هنا كـ`NaN` بدل تلويث IC) أو قيمة شبه ثابتة عبر كل عيّنات النافذة (`compute_ic` نفسها تُرجع 0.0 صراحةً لتباين شبه صفري — إعلان "لا إشارة" لا خطأ حسابي). مؤشّرات أخرى (`KST`/`EBSW`/`HVOL`/`MASSI`/`HT_*`) تحتاج نافذة تاريخ أطول من 32 شمعة (طول العيّنة هنا) داخلياً، فتُرجع NaN أو (لـ`EBSW`) الإطار الأصلي بلا تغيير من pandas_ta_classic نفسها.

### النتيجة الفعلية (40 مرشّحاً × 3 أهداف = 120 محاولة، 117 نجحت في الحساب)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign | n_ok |
|---|---|---|---|---|---|
| `BIAS_14` | close | +0.0000 | 0% | ❌ False | 12 |
| `BIAS_14` | high | +0.0000 | 0% | ❌ False | 12 |
| `BIAS_14` | low | +0.0000 | 0% | ❌ False | 12 |
| `BOP` | close | -0.0266 | 17% | ❌ False | 12 |
| `BOP` | high | -0.0689 | 25% | ❌ False | 12 |
| `BOP` | low | +0.0048 | 17% | ❌ False | 12 |
| `BRAR_14` | close | -0.0346 | 25% | ❌ False | 12 |
| `BRAR_14` | high | +0.0287 | 25% | ❌ False | 12 |
| `BRAR_14` | low | -0.0675 | 33% | ❌ False | 12 |
| `CFO_14` | close | — | — | — | 0 |
| `CFO_14` | high | — | — | — | 0 |
| `CFO_14` | low | — | — | — | 0 |
| `CG_14` | close | +0.0130 | 17% | ❌ False | 12 |
| `CG_14` | high | +0.0436 | 42% | ❌ False | 12 |
| `CG_14` | low | -0.0403 | 25% | ❌ False | 12 |
| `COPPOCK` | close | +0.0133 | 17% | ❌ False | 12 |
| `COPPOCK` | high | +0.0658 | 17% | ✅ True | 12 |
| `COPPOCK` | low | -0.0401 | 17% | ❌ False | 12 |
| `ER_14` | close | -0.0030 | 0% | ❌ False | 12 |
| `ER_14` | high | +0.0582 | 25% | ❌ False | 12 |
| `ER_14` | low | -0.0777 | 25% | ✅ True | 12 |
| `FOSC_14` | close | — | — | — | 0 |
| `FOSC_14` | high | — | — | — | 0 |
| `FOSC_14` | low | — | — | — | 0 |
| `INERTIA_14` | close | — | — | — | 0 |
| `INERTIA_14` | high | — | — | — | 0 |
| `INERTIA_14` | low | — | — | — | 0 |
| `KST` | close | — | — | — | 0 |
| `KST` | high | — | — | — | 0 |
| `KST` | low | — | — | — | 0 |
| `LRSI` | close | -0.0424 | 25% | ❌ False | 12 |
| `LRSI` | high | -0.0160 | 25% | ❌ False | 12 |
| `LRSI` | low | -0.0469 | 50% | ❌ False | 12 |
| `PGO_14` | close | -0.0587 | 17% | ❌ False | 12 |
| `PGO_14` | high | -0.0306 | 17% | ❌ False | 12 |
| `PGO_14` | low | -0.0405 | 50% | ❌ False | 12 |
| `PO` | close | +0.0000 | 0% | ❌ False | 12 |
| `PO` | high | +0.0000 | 0% | ❌ False | 12 |
| `PO` | low | +0.0000 | 0% | ❌ False | 12 |
| `PPO` | close | +0.0123 | 17% | ❌ False | 12 |
| `PPO` | high | +0.0450 | 8% | ❌ False | 12 |
| `PPO` | low | -0.0299 | 17% | ❌ False | 12 |
| `PSL_14` | close | -0.0358 | 33% | ❌ False | 12 |
| `PSL_14` | high | -0.0368 | 25% | ❌ False | 12 |
| `PSL_14` | low | -0.0294 | 25% | ❌ False | 12 |
| `PVO` | close | -0.0094 | 0% | ❌ False | 12 |
| `PVO` | high | -0.0166 | 8% | ❌ False | 12 |
| `PVO` | low | -0.0011 | 0% | ❌ False | 12 |
| `RSX_14` | close | -0.0594 | 33% | ❌ False | 12 |
| `RSX_14` | high | -0.0020 | 17% | ❌ False | 12 |
| `RSX_14` | low | -0.0864 | 58% | ❌ False | 12 |
| `RVGI_14` | close | -0.0619 | 33% | ❌ False | 12 |
| `RVGI_14` | high | +0.0016 | 17% | ❌ False | 12 |
| `RVGI_14` | low | -0.0961 | 42% | ❌ False | 12 |
| `SMI` | close | -0.0389 | 25% | ❌ False | 12 |
| `SMI` | high | +0.0123 | 33% | ❌ False | 12 |
| `SMI` | low | -0.0830 | 67% | ❌ False | 12 |
| `STC` | close | +0.0000 | 0% | ❌ False | 12 |
| `STC` | high | +0.0000 | 0% | ❌ False | 12 |
| `STC` | low | +0.0000 | 0% | ❌ False | 12 |
| `TRIX_14` | close | — | — | — | 0 |
| `TRIX_14` | high | — | — | — | 0 |
| `TRIX_14` | low | — | — | — | 0 |
| `AROON_14` | close | -0.0335 | 17% | ❌ False | 12 |
| `AROON_14` | high | +0.0010 | 8% | ❌ False | 12 |
| `AROON_14` | low | -0.0366 | 25% | ❌ False | 12 |
| `CHOP_14` | close | -0.0471 | 25% | ❌ False | 12 |
| `CHOP_14` | high | -0.0792 | 33% | ❌ False | 12 |
| `CHOP_14` | low | +0.0467 | 42% | ❌ False | 12 |
| `VHF_14` | close | +0.0390 | 25% | ❌ False | 12 |
| `VHF_14` | high | +0.0608 | 25% | ❌ False | 12 |
| `VHF_14` | low | -0.0416 | 42% | ❌ False | 12 |
| `CVI_14` | close | +0.0243 | 17% | ❌ False | 12 |
| `CVI_14` | high | +0.1233 | 58% | ✅ True | 12 |
| `CVI_14` | low | -0.1234 | 58% | ❌ False | 12 |
| `HVOL` | close | — | — | — | 0 |
| `HVOL` | high | — | — | — | 0 |
| `HVOL` | low | — | — | — | 0 |
| `MASSI` | close | — | — | — | 0 |
| `MASSI` | high | — | — | — | 0 |
| `MASSI` | low | — | — | — | 0 |
| `UI_14` | close | -0.0078 | 8% | ❌ False | 12 |
| `UI_14` | high | -0.0754 | 33% | ❌ False | 12 |
| `UI_14` | low | +0.1081 | 58% | ❌ False | 12 |
| `VOSC` | close | -0.0018 | 0% | ❌ False | 12 |
| `VOSC` | high | +0.0086 | 0% | ❌ False | 12 |
| `VOSC` | low | -0.0383 | 0% | ❌ False | 12 |
| `ENTROPY_14` | close | +0.0047 | 8% | ❌ False | 12 |
| `ENTROPY_14` | high | +0.0733 | 33% | ❌ False | 12 |
| `ENTROPY_14` | low | -0.1037 | 50% | ✅ True | 12 |
| `KURTOSIS_14` | close | -0.0037 | 8% | ❌ False | 12 |
| `KURTOSIS_14` | high | +0.0222 | 8% | ❌ False | 12 |
| `KURTOSIS_14` | low | -0.0156 | 17% | ❌ False | 12 |
| `ZSCORE_14` | close | -0.0343 | 17% | ❌ False | 12 |
| `ZSCORE_14` | high | -0.0172 | 17% | ❌ False | 12 |
| `ZSCORE_14` | low | -0.0097 | 25% | ❌ False | 12 |
| `HT_DCPERIOD` | close | — | — | — | 0 |
| `HT_DCPERIOD` | high | — | — | — | 0 |
| `HT_DCPERIOD` | low | — | — | — | 0 |
| `HT_DCPHASE` | close | — | — | — | 0 |
| `HT_DCPHASE` | high | — | — | — | 0 |
| `HT_DCPHASE` | low | — | — | — | 0 |
| `HT_SINE` | close | — | — | — | 0 |
| `HT_SINE` | high | — | — | — | 0 |
| `HT_SINE` | low | — | — | — | 0 |
| `MSW` | close | -0.0363 | 42% | ❌ False | 12 |
| `MSW` | high | -0.0361 | 33% | ❌ False | 12 |
| `MSW` | low | +0.0131 | 17% | ❌ False | 12 |
| `DRAWDOWN` | close | +0.0000 | 0% | ❌ False | 12 |
| `DRAWDOWN` | high | +0.0000 | 0% | ❌ False | 12 |
| `DRAWDOWN` | low | +0.0000 | 0% | ❌ False | 12 |
| `LOG_RETURN_14` | close | — | — | — | 0 |
| `LOG_RETURN_14` | high | — | — | — | 0 |
| `LOG_RETURN_14` | low | — | — | — | 0 |
| `PERCENT_RETURN_14` | close | +0.0000 | 0% | ❌ False | 12 |
| `PERCENT_RETURN_14` | high | +0.0000 | 0% | ❌ False | 12 |
| `PERCENT_RETURN_14` | low | +0.0000 | 0% | ❌ False | 12 |

(`—` في `mean_ic`/`frac_significant` = لم تنجح نافذة واحدة بحدٍّ أدنى 10 عيّنات صالحة؛ `EBSW` غير مذكور أصلاً — تعطّل قبل حتى إتمام نافذة واحدة بسبب رجوع `pandas_ta_classic.ebsw` الإطار الأصلي بلا حساب لقصر النافذة.)

### 🎯 نتيجتان تعبران معيار القبول فعلياً — أول مرّة منذ H003

على عكس الدفعة الأولى (27/27 مرفوضة دون استثناء)، **مرشّحان هنا يحقّقان معيار المحور الثلاثي بالكامل** (`consistent_sign=True` و`frac_significant≥0.34` و`n_ok≥5`) — نفس المعيار الذي أنتج H001 وH003، أوّل فرضيتين مقبولتين في تاريخ المشروع:

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign عبر ١٢ نافذة |
|---|---|---|---|---|
| `CVI_14` (Chaikin Volatility) | high | +0.1233 | 58% (7/12) | ✅ موجب في كل الـ12 |
| `ENTROPY_14` (انتروبيا شانون) | low | -0.1037 | 50% (6/12) | ✅ سالب في كل الـ12 |

تفصيل IC لكل نافذة على حدة (تحقّق مباشر، لا تجميع فقط):

**`CVI_14`/high** — IC: [+0.044, +0.065, +0.172, +0.045, +0.205, +0.215, +0.217, +0.155, +0.055, +0.148, +0.029, +0.131] — موجبة في الـ12 كلّها، 7 منها معنويّة (p<0.05).

**`ENTROPY_14`/low** — IC: [-0.081, -0.136, -0.114, -0.101, -0.277, -0.108, -0.071, -0.042, -0.063, -0.034, -0.003, -0.214] — سالبة في الـ12 كلّها، 6 منها معنويّة (p<0.05).

**⚠️ لماذا "قيد الاختبار" لا "مقبولة" بعد**: هذا الإعداد نفسه (14 عملة، 12 نافذة قصيرة الأفق) وُصِف صراحةً في القسم ١٣ بأنه "تصفية أوّلية رخيصة، لا حكم نهائي" — أضيق ممّا استُخدم لتأكيد H003 (50 عملة، 30 نافذة). كذلك 120 محاولة اختبار في هذه الدفعة وحدها (و147 تراكمياً مع الدفعة الأولى) تجعل احتمال إيجابيّ كاذب واحد على الأقل بالصدفة غير مهمَل (فخّ الاختبارات المتعدّدة نفسه المذكور في خطة المشروع). الخطوة التالية قبل الحكم النهائي: **نفس ترقية H003** — توسيع إلى 50 عملة و30 نافذة، ثم إعادة القياس؛ إن ثبت الاتساق يُرقَّيان إلى "مقبولة"، وإلا إلى "مرفوضة".

### تسجيل الدفعة الثانية كاملة في سجلّ التجارب

In [ ]:
# @title
PANDAS_TA_BATCH2_RESULTS = [
    ("BIAS_14", {'length': 14}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("BIAS_14", {'length': 14}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("BIAS_14", {'length': 14}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("BOP", {}, "close", -0.026640880492648084, 0.11023575239804913, 0.16666666666666666, False, 12, "مرفوضة"),
    ("BOP", {}, "high", -0.06889941742414814, 0.09363202093972425, 0.25, False, 12, "مرفوضة"),
    ("BOP", {}, "low", 0.0048414991517839005, 0.10608690071202802, 0.16666666666666666, False, 12, "مرفوضة"),
    ("BRAR_14", {'length': 14}, "close", -0.03457404008623937, 0.0862803439995029, 0.25, False, 12, "مرفوضة"),
    ("BRAR_14", {'length': 14}, "high", 0.02873830301366512, 0.11178119963658904, 0.25, False, 12, "مرفوضة"),
    ("BRAR_14", {'length': 14}, "low", -0.0675119663120899, 0.08902110075267997, 0.3333333333333333, False, 12, "مرفوضة"),
    ("CFO_14", {'length': 14}, "close", None, None, None, None, 0, "مرفوضة"),
    ("CFO_14", {'length': 14}, "high", None, None, None, None, 0, "مرفوضة"),
    ("CFO_14", {'length': 14}, "low", None, None, None, None, 0, "مرفوضة"),
    ("CG_14", {'length': 14}, "close", 0.013034602203601746, 0.09319780367435597, 0.16666666666666666, False, 12, "مرفوضة"),
    ("CG_14", {'length': 14}, "high", 0.043608840179607554, 0.10621460767636949, 0.4166666666666667, False, 12, "مرفوضة"),
    ("CG_14", {'length': 14}, "low", -0.0402949940073424, 0.10431770596028055, 0.25, False, 12, "مرفوضة"),
    ("COPPOCK", {}, "close", 0.01328267370587826, 0.07971515356077623, 0.16666666666666666, False, 12, "مرفوضة"),
    ("COPPOCK", {}, "high", 0.06575826483078401, 0.051603601070761354, 0.16666666666666666, True, 12, "مرفوضة"),
    ("COPPOCK", {}, "low", -0.04014542593913592, 0.07034055471408006, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ER_14", {'length': 14}, "close", -0.0030009128006564297, 0.04575049547453274, 0.0, False, 12, "مرفوضة"),
    ("ER_14", {'length': 14}, "high", 0.05822728205187252, 0.07513457075696701, 0.25, False, 12, "مرفوضة"),
    ("ER_14", {'length': 14}, "low", -0.07765815274856867, 0.04149876108571658, 0.25, True, 12, "مرفوضة"),
    ("FOSC_14", {'length': 14}, "close", None, None, None, None, 0, "مرفوضة"),
    ("FOSC_14", {'length': 14}, "high", None, None, None, None, 0, "مرفوضة"),
    ("FOSC_14", {'length': 14}, "low", None, None, None, None, 0, "مرفوضة"),
    ("INERTIA_14", {'length': 14}, "close", None, None, None, None, 0, "مرفوضة"),
    ("INERTIA_14", {'length': 14}, "high", None, None, None, None, 0, "مرفوضة"),
    ("INERTIA_14", {'length': 14}, "low", None, None, None, None, 0, "مرفوضة"),
    ("KST", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("KST", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("KST", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("LRSI", {}, "close", -0.042364328145256876, 0.11925423168045791, 0.25, False, 12, "مرفوضة"),
    ("LRSI", {}, "high", -0.01604500726513892, 0.09731725332110426, 0.25, False, 12, "مرفوضة"),
    ("LRSI", {}, "low", -0.046942696839073324, 0.10460895350813103, 0.5, False, 12, "مرفوضة"),
    ("PGO_14", {'length': 14}, "close", -0.05871573861895954, 0.12939415549020114, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PGO_14", {'length': 14}, "high", -0.030604946973093527, 0.13069881379639664, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PGO_14", {'length': 14}, "low", -0.04046898603996284, 0.14222000158208606, 0.5, False, 12, "مرفوضة"),
    ("PO", {}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PO", {}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PO", {}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PPO", {}, "close", 0.012276779707332939, 0.06927722704178692, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PPO", {}, "high", 0.04497125890272874, 0.043321832559001405, 0.08333333333333333, False, 12, "مرفوضة"),
    ("PPO", {}, "low", -0.029928280398259732, 0.07545485969707535, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PSL_14", {'length': 14}, "close", -0.03577832023743047, 0.08644412890290504, 0.3333333333333333, False, 12, "مرفوضة"),
    ("PSL_14", {'length': 14}, "high", -0.03684832962168521, 0.10965433529619886, 0.25, False, 12, "مرفوضة"),
    ("PSL_14", {'length': 14}, "low", -0.02939103670389392, 0.07575270204238166, 0.25, False, 12, "مرفوضة"),
    ("PVO", {}, "close", -0.009446714951682115, 0.045369302350417556, 0.0, False, 12, "مرفوضة"),
    ("PVO", {}, "high", -0.01657094300321939, 0.05682617552233607, 0.08333333333333333, False, 12, "مرفوضة"),
    ("PVO", {}, "low", -0.0010697991743232015, 0.046752100488697566, 0.0, False, 12, "مرفوضة"),
    ("RSX_14", {'length': 14}, "close", -0.05943701219726911, 0.0976840987018078, 0.3333333333333333, False, 12, "مرفوضة"),
    ("RSX_14", {'length': 14}, "high", -0.002011719747929658, 0.1376356968096883, 0.16666666666666666, False, 12, "مرفوضة"),
    ("RSX_14", {'length': 14}, "low", -0.08642866937886413, 0.07724611220925648, 0.5833333333333334, False, 12, "مرفوضة"),
    ("RVGI_14", {'length': 14}, "close", -0.06189651106119984, 0.08567078040502259, 0.3333333333333333, False, 12, "مرفوضة"),
    ("RVGI_14", {'length': 14}, "high", 0.0015870588110600374, 0.1304764391347172, 0.16666666666666666, False, 12, "مرفوضة"),
    ("RVGI_14", {'length': 14}, "low", -0.09606556863014827, 0.0619174776020068, 0.4166666666666667, False, 12, "مرفوضة"),
    ("SMI", {}, "close", -0.038924598496197255, 0.10305350662896494, 0.25, False, 12, "مرفوضة"),
    ("SMI", {}, "high", 0.012345108368963867, 0.09576021536583003, 0.3333333333333333, False, 12, "مرفوضة"),
    ("SMI", {}, "low", -0.08302735173098856, 0.11608741227127282, 0.6666666666666666, False, 12, "مرفوضة"),
    ("STC", {}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("STC", {}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("STC", {}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("TRIX_14", {'length': 14}, "close", None, None, None, None, 0, "مرفوضة"),
    ("TRIX_14", {'length': 14}, "high", None, None, None, None, 0, "مرفوضة"),
    ("TRIX_14", {'length': 14}, "low", None, None, None, None, 0, "مرفوضة"),
    ("AROON_14", {'length': 14}, "close", -0.033472924450395014, 0.08432818689367255, 0.16666666666666666, False, 12, "مرفوضة"),
    ("AROON_14", {'length': 14}, "high", 0.0009787884646574238, 0.09789884496175305, 0.08333333333333333, False, 12, "مرفوضة"),
    ("AROON_14", {'length': 14}, "low", -0.03660133457148866, 0.10169856873221512, 0.25, False, 12, "مرفوضة"),
    ("CHOP_14", {'length': 14}, "close", -0.047111201695386996, 0.10835705900087893, 0.25, False, 12, "مرفوضة"),
    ("CHOP_14", {'length': 14}, "high", -0.07921978057592802, 0.11858418638205774, 0.3333333333333333, False, 12, "مرفوضة"),
    ("CHOP_14", {'length': 14}, "low", 0.04668496109399731, 0.11219944657645531, 0.4166666666666667, False, 12, "مرفوضة"),
    ("VHF_14", {'length': 14}, "close", 0.03896716226123633, 0.10339856039142711, 0.25, False, 12, "مرفوضة"),
    ("VHF_14", {'length': 14}, "high", 0.060818005192192964, 0.09964003936214019, 0.25, False, 12, "مرفوضة"),
    ("VHF_14", {'length': 14}, "low", -0.04157519700556343, 0.11642625584998378, 0.4166666666666667, False, 12, "مرفوضة"),
    ("CVI_14", {'length': 14}, "close", 0.024270474221611094, 0.1129770368405859, 0.16666666666666666, False, 12, "مرفوضة"),
    ("CVI_14", {'length': 14}, "high", 0.123300920354017, 0.06916794710118895, 0.5833333333333334, True, 12, "قيد الاختبار"),
    ("CVI_14", {'length': 14}, "low", -0.12335108810425831, 0.16640709095331283, 0.5833333333333334, False, 12, "مرفوضة"),
    ("HVOL", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HVOL", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HVOL", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("MASSI", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("MASSI", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("MASSI", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("UI_14", {'length': 14}, "close", -0.007803710866125811, 0.12470682728048027, 0.08333333333333333, False, 12, "مرفوضة"),
    ("UI_14", {'length': 14}, "high", -0.07544031309564006, 0.09240237290795181, 0.3333333333333333, False, 12, "مرفوضة"),
    ("UI_14", {'length': 14}, "low", 0.1080937166633091, 0.13560423137349462, 0.5833333333333334, False, 12, "مرفوضة"),
    ("VOSC", {}, "close", -0.0018109156637352096, 0.043288345144043755, 0.0, False, 12, "مرفوضة"),
    ("VOSC", {}, "high", 0.008581976580591887, 0.05114237834767161, 0.0, False, 12, "مرفوضة"),
    ("VOSC", {}, "low", -0.038278526755768545, 0.05039641149249179, 0.0, False, 12, "مرفوضة"),
    ("ENTROPY_14", {'length': 14}, "close", 0.004721292231067143, 0.059627750957348485, 0.08333333333333333, False, 12, "مرفوضة"),
    ("ENTROPY_14", {'length': 14}, "high", 0.0732552371249778, 0.05118248169784147, 0.3333333333333333, False, 12, "مرفوضة"),
    ("ENTROPY_14", {'length': 14}, "low", -0.1036954343434134, 0.07369215134873021, 0.5, True, 12, "قيد الاختبار"),
    ("KURTOSIS_14", {'length': 14}, "close", -0.003720173044142581, 0.08334005316709615, 0.08333333333333333, False, 12, "مرفوضة"),
    ("KURTOSIS_14", {'length': 14}, "high", 0.022198425377697294, 0.07280271125760591, 0.08333333333333333, False, 12, "مرفوضة"),
    ("KURTOSIS_14", {'length': 14}, "low", -0.015595358504532175, 0.1008223060218417, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ZSCORE_14", {'length': 14}, "close", -0.03430457159698896, 0.12372955920167289, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ZSCORE_14", {'length': 14}, "high", -0.017206503792791542, 0.10846525478611667, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ZSCORE_14", {'length': 14}, "low", -0.00969328169961032, 0.12947053231095557, 0.25, False, 12, "مرفوضة"),
    ("HT_DCPERIOD", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HT_DCPERIOD", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HT_DCPERIOD", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("HT_DCPHASE", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HT_DCPHASE", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HT_DCPHASE", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("HT_SINE", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HT_SINE", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HT_SINE", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("MSW", {}, "close", -0.0363441842703067, 0.12525195559294766, 0.4166666666666667, False, 12, "مرفوضة"),
    ("MSW", {}, "high", -0.03613393726960613, 0.08608769884244219, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MSW", {}, "low", 0.01306783846320135, 0.1103927807568972, 0.16666666666666666, False, 12, "مرفوضة"),
    ("DRAWDOWN", {}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("DRAWDOWN", {}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("DRAWDOWN", {}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("LOG_RETURN_14", {'length': 14}, "close", None, None, None, None, 0, "مرفوضة"),
    ("LOG_RETURN_14", {'length': 14}, "high", None, None, None, None, 0, "مرفوضة"),
    ("LOG_RETURN_14", {'length': 14}, "low", None, None, None, None, 0, "مرفوضة"),
    ("PERCENT_RETURN_14", {'length': 14}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PERCENT_RETURN_14", {'length': 14}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PERCENT_RETURN_14", {'length': 14}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة")
]

for name, params, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok, status in PANDAS_TA_BATCH2_RESULTS:
    if status == "قيد الاختبار":
        notes = (
            "عبر معيار المحور الثلاثي فعلياً (consistent_sign=True، "
            "frac_significant>=0.34، n_ok=12) على 14 عملة/12 نافذة — لكن "
            "إعداد أضيق من H003 (50 عملة/30 نافذة) واحتمال اختبارات متعدّدة "
            "(120 محاولة في هذه الدفعة). يحتاج نفس ترقية H003 (توسيع "
            "الأصول/النوافذ) قبل التصنيف النهائي مقبولة/مرفوضة."
        )
    elif name in {
        "CFO_14", "FOSC_14", "INERTIA_14", "KST", "TRIX_14", "LOG_RETURN_14",
        "HVOL", "MASSI", "HT_DCPERIOD", "HT_DCPHASE", "HT_SINE",
    }:
        notes = (
            "غير قابل للاختبار بهذا الإعداد فعلياً — يحتاج نافذة تاريخ أطول "
            "من 32 شمعة داخلياً، أو يقسم على close الحالي (مُرسى صفراً دائماً "
            "في هذا التمثيل) فيُنتج inf/NaN لكل عيّنة — لا نتيجة سلبية حقيقية، "
            "بل قيد تقني في هذا الإعداد تحديداً."
        )
    elif name in {"BIAS_14", "PO", "STC", "DRAWDOWN", "PERCENT_RETURN_14"}:
        notes = (
            "قيمته الأخيرة دالة مباشرة في close الحالي (مُرسى صفراً دائماً في "
            "هذا التمثيل) فتنهار لقيمة شبه ثابتة عبر كل عيّنات أيّ نافذة — "
            "compute_ic تُرجع 0.0 صراحةً — لا نتيجة سلبية حقيقية، بل قيد تقني "
            "في هذا الإعداد تحديداً."
        )
    else:
        notes = (
            "مُسجَّلة من تشغيل حقيقي فعلي (14 عملة متنوّعة، 31,604 عيّنة، 12 "
            "نافذة متحرّكة) — لا بيانات وهمية. consistent_sign=False يمنع "
            "القبول بصرف النظر عن frac_significant."
        )
    register_hypothesis(
        hyp_id=f"PANDAS_TA_BATCH2_{name}_{target}",
        hypothesis=(
            f"{name} ({params}) على هدف {target} — الدفعة الثانية من مرشّحي "
            "pandas_ta (40 مرشّحاً)؛ راجع signal_discovery_lab.ipynb القسم ١٤ "
            "للمنهجية الكاملة."
        ),
        source="literature_mining",
        status=status,
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=notes,
    )

print(f'✅ سُجِّلت {len(PANDAS_TA_BATCH2_RESULTS)} توليفة (2 قيد الاختبار، البقية مرفوضة).')

## ١٥) الدفعة الثالثة من مرشّحي pandas_ta — تصحيح منهجي واستكمال شبه كامل للتصنيفات المتاحة

الدفعة الثانية (القسم ١٤) كشفت أن `open`/`high`/`low`/`close`/`volume` الخام في هذا الـdataset **مُطبَّعة محلياً لكل نافذة عيّنة** (انحراف معياري لعمود `close` يتراوح ~0.5-2.4 عبر أصول يختلف سعرها الحقيقي بمقدار 6 مراتب — راجع القياس المباشر في [docs/research/pandas_ta_deferred_candidates.md](../../docs/research/pandas_ta_deferred_candidates.md)). هذا يُبطل الافتراض الذي استبعدت بموجبه الدفعة الأولى (القسم ١٣) مؤشّرات مثل `ATR`/`AD`/`OBV`/`WCP`/`QSTICK` بحجّة "مطلقة القيمة، غير قابلة للمقارنة عبر أصول" — فعلياً هي قابلة للاختبار مباشرة بنفس الطريقة العامة، بلا تطبيع إضافي.

هذه الدفعة تختبر 77 مرشّحاً جديداً (كل ما تبقّى من `SURVEY_CANDIDATES_ROBUST` القابل للاختبار بمنهجية IC الخطّية المفردة، عبر فئات trend/momentum/volatility/volume/statistics/cycles/overlap) على نفس dataset الـ14 عملة ونفس 12 نافذة. **استُبعِد فقط**: ميزات مكرّرة موجودة أصلاً في `feature_order` (مثل `EMA`/`RSI`/`MACD`/`ADX`/`BBANDS`/`ATR`/`NATR`/`MFI`/`CMF`/`STOCH`)، مولّدات إشارة منطقية/نظام معقّدة تحتاج معالجة خاصة (`kdj`/`qqe`/`squeeze`/`psar`/`tsignals`/`ichimoku`/`supertrend`/`hilo`/...)، فئة `candles` (كاشفات أنماط شموع 0/1، نوع بيانات مختلف تماماً)، مؤشّرات تحتاج مدخلاً خارجياً (`beta`/`correl`)، و`vwap` (يحتاج `DatetimeIndex` غير متوفّر في هذا التمثيل لكل عيّنة).

### النتيجة الفعلية (77 مرشّحاً × 3 أهداف = 231 محاولة، 213 نجحت في الحساب)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign | n_ok |
|---|---|---|---|---|---|
| `QSTICK_14` | close | -0.0357 | 33% | ❌ False | 12 |
| `QSTICK_14` | high | +0.0200 | 25% | ❌ False | 12 |
| `QSTICK_14` | low | -0.0451 | 33% | ❌ False | 12 |
| `APO` | close | -0.0728 | 25% | ❌ False | 12 |
| `APO` | high | +0.0202 | 25% | ❌ False | 12 |
| `APO` | low | -0.1128 | 33% | ✅ True | 12 |
| `ERI_14` | close | -0.0397 | 33% | ❌ False | 12 |
| `ERI_14` | high | +0.0069 | 8% | ❌ False | 12 |
| `ERI_14` | low | -0.0498 | 33% | ❌ False | 12 |
| `MOM_14` | close | -0.0357 | 33% | ❌ False | 12 |
| `MOM_14` | high | +0.0200 | 25% | ❌ False | 12 |
| `MOM_14` | low | -0.0451 | 33% | ❌ False | 12 |
| `SLOPE_14` | close | -0.0357 | 33% | ❌ False | 12 |
| `SLOPE_14` | high | +0.0200 | 25% | ❌ False | 12 |
| `SLOPE_14` | low | -0.0451 | 33% | ❌ False | 12 |
| `VWMACD` | close | +0.0021 | 8% | ❌ False | 12 |
| `VWMACD` | high | -0.0232 | 8% | ❌ False | 12 |
| `VWMACD` | low | +0.0123 | 8% | ❌ False | 12 |
| `ABERRATION_14` | close | +0.0425 | 17% | ❌ False | 12 |
| `ABERRATION_14` | high | +0.0136 | 17% | ❌ False | 12 |
| `ABERRATION_14` | low | +0.0274 | 42% | ❌ False | 12 |
| `ACCBANDS_14` | close | +0.0418 | 17% | ❌ False | 12 |
| `ACCBANDS_14` | high | +0.0132 | 17% | ❌ False | 12 |
| `ACCBANDS_14` | low | +0.0275 | 42% | ❌ False | 12 |
| `AVOLUME` | close | — | — | — | 0 |
| `AVOLUME` | high | — | — | — | 0 |
| `AVOLUME` | low | — | — | — | 0 |
| `CE` | close | +0.0647 | 33% | ❌ False | 12 |
| `CE` | high | +0.0808 | 42% | ❌ False | 12 |
| `CE` | low | -0.0168 | 25% | ❌ False | 12 |
| `DONCHIAN_14` | close | +0.0429 | 25% | ❌ False | 12 |
| `DONCHIAN_14` | high | +0.0314 | 33% | ❌ False | 12 |
| `DONCHIAN_14` | low | +0.0196 | 17% | ❌ False | 12 |
| `HWC` | close | -0.0126 | 17% | ❌ False | 12 |
| `HWC` | high | +0.0293 | 17% | ❌ False | 12 |
| `HWC` | low | -0.0537 | 58% | ❌ False | 12 |
| `KC_14` | close | +0.0491 | 25% | ❌ False | 12 |
| `KC_14` | high | +0.0210 | 33% | ❌ False | 12 |
| `KC_14` | low | +0.0281 | 33% | ❌ False | 12 |
| `PDIST` | close | +0.0256 | 17% | ❌ False | 12 |
| `PDIST` | high | +0.0925 | 42% | ❌ False | 12 |
| `PDIST` | low | -0.0360 | 17% | ❌ False | 12 |
| `THERMO_14` | close | +0.0214 | 17% | ❌ False | 12 |
| `THERMO_14` | high | +0.0941 | 42% | ❌ False | 12 |
| `THERMO_14` | low | -0.0794 | 33% | ❌ False | 12 |
| `TRUE_RANGE` | close | +0.0339 | 25% | ❌ False | 12 |
| `TRUE_RANGE` | high | +0.1047 | 50% | ❌ False | 12 |
| `TRUE_RANGE` | low | -0.0387 | 17% | ❌ False | 12 |
| `AD` | close | -0.0253 | 0% | ❌ False | 12 |
| `AD` | high | +0.0400 | 33% | ❌ False | 12 |
| `AD` | low | -0.0512 | 50% | ❌ False | 12 |
| `ADOSC` | close | -0.0503 | 25% | ❌ False | 12 |
| `ADOSC` | high | +0.0521 | 33% | ❌ False | 12 |
| `ADOSC` | low | -0.1105 | 33% | ✅ True | 12 |
| `AOBV` | close | -0.0083 | 8% | ❌ False | 12 |
| `AOBV` | high | +0.0977 | 50% | ✅ True | 12 |
| `AOBV` | low | -0.0678 | 25% | ❌ False | 12 |
| `EFI_14` | close | -0.0330 | 25% | ❌ False | 12 |
| `EFI_14` | high | +0.0740 | 42% | ❌ False | 12 |
| `EFI_14` | low | -0.0748 | 50% | ❌ False | 12 |
| `EMV_14` | close | -0.0140 | 8% | ❌ False | 12 |
| `EMV_14` | high | +0.0193 | 17% | ❌ False | 12 |
| `EMV_14` | low | -0.0220 | 25% | ❌ False | 12 |
| `EOM_14` | close | +0.0039 | 0% | ❌ False | 12 |
| `EOM_14` | high | +0.0224 | 17% | ❌ False | 12 |
| `EOM_14` | low | -0.0212 | 8% | ❌ False | 12 |
| `MARKETFI` | close | +0.0297 | 8% | ❌ False | 12 |
| `MARKETFI` | high | +0.1034 | 42% | ❌ False | 12 |
| `MARKETFI` | low | -0.0577 | 8% | ❌ False | 12 |
| `NVI` | close | -0.0087 | 8% | ❌ False | 12 |
| `NVI` | high | +0.0177 | 25% | ❌ False | 12 |
| `NVI` | low | -0.0356 | 17% | ❌ False | 12 |
| `PVI` | close | +0.0014 | 0% | ❌ False | 12 |
| `PVI` | high | +0.0113 | 8% | ❌ False | 12 |
| `PVI` | low | -0.0289 | 8% | ❌ False | 12 |
| `PVOL` | close | +0.0000 | 0% | ❌ False | 12 |
| `PVOL` | high | +0.0000 | 0% | ❌ False | 12 |
| `PVOL` | low | +0.0000 | 0% | ❌ False | 12 |
| `PVT` | close | -0.0012 | 0% | ❌ False | 12 |
| `PVT` | high | -0.0120 | 8% | ❌ False | 12 |
| `PVT` | low | +0.0137 | 0% | ❌ False | 12 |
| `WAD` | close | -0.0014 | 0% | ❌ False | 12 |
| `WAD` | high | +0.0173 | 17% | ❌ False | 12 |
| `WAD` | low | -0.0296 | 25% | ❌ False | 12 |
| `MAD_14` | close | +0.0477 | 33% | ❌ False | 12 |
| `MAD_14` | high | +0.0629 | 33% | ❌ False | 12 |
| `MAD_14` | low | -0.0078 | 17% | ❌ False | 12 |
| `MD_14` | close | +0.0477 | 33% | ❌ False | 12 |
| `MD_14` | high | +0.0629 | 33% | ❌ False | 12 |
| `MD_14` | low | -0.0078 | 17% | ❌ False | 12 |
| `MEDIAN_14` | close | +0.0391 | 17% | ❌ False | 12 |
| `MEDIAN_14` | high | +0.0101 | 17% | ❌ False | 12 |
| `MEDIAN_14` | low | +0.0263 | 42% | ❌ False | 12 |
| `QUANTILE_14` | close | +0.0391 | 17% | ❌ False | 12 |
| `QUANTILE_14` | high | +0.0101 | 17% | ❌ False | 12 |
| `QUANTILE_14` | low | +0.0263 | 42% | ❌ False | 12 |
| `STDERR_14` | close | +0.0498 | 33% | ❌ False | 12 |
| `STDERR_14` | high | +0.0680 | 33% | ❌ False | 12 |
| `STDERR_14` | low | -0.0098 | 17% | ❌ False | 12 |
| `STDEV_14` | close | +0.0498 | 33% | ❌ False | 12 |
| `STDEV_14` | high | +0.0680 | 33% | ❌ False | 12 |
| `STDEV_14` | low | -0.0098 | 17% | ❌ False | 12 |
| `VARIANCE_14` | close | +0.0498 | 33% | ❌ False | 12 |
| `VARIANCE_14` | high | +0.0680 | 33% | ❌ False | 12 |
| `VARIANCE_14` | low | -0.0098 | 17% | ❌ False | 12 |
| `DSP_14` | close | -0.0491 | 25% | ❌ False | 12 |
| `DSP_14` | high | -0.0210 | 33% | ❌ False | 12 |
| `DSP_14` | low | -0.0281 | 33% | ❌ False | 12 |
| `HT_PHASOR` | close | — | — | — | 0 |
| `HT_PHASOR` | high | — | — | — | 0 |
| `HT_PHASOR` | low | — | — | — | 0 |
| `ALMA_14` | close | +0.0470 | 17% | ❌ False | 12 |
| `ALMA_14` | high | -0.0008 | 8% | ❌ False | 12 |
| `ALMA_14` | low | +0.0522 | 50% | ❌ False | 12 |
| `AVGPRICE` | close | +0.0286 | 17% | ❌ False | 12 |
| `AVGPRICE` | high | +0.0452 | 17% | ❌ False | 12 |
| `AVGPRICE` | low | -0.0111 | 25% | ❌ False | 12 |
| `DEMA_14` | close | +0.0078 | 25% | ❌ False | 12 |
| `DEMA_14` | high | +0.0307 | 25% | ❌ False | 12 |
| `DEMA_14` | low | -0.0357 | 42% | ❌ False | 12 |
| `FWMA_14` | close | +0.0282 | 33% | ❌ False | 12 |
| `FWMA_14` | high | +0.0390 | 33% | ❌ False | 12 |
| `FWMA_14` | low | -0.0121 | 33% | ❌ False | 12 |
| `HL2` | close | +0.0117 | 25% | ❌ False | 12 |
| `HL2` | high | +0.0302 | 25% | ❌ False | 12 |
| `HL2` | low | -0.0224 | 25% | ❌ False | 12 |
| `HLC3` | close | +0.0117 | 25% | ❌ False | 12 |
| `HLC3` | high | +0.0302 | 25% | ❌ False | 12 |
| `HLC3` | low | -0.0224 | 25% | ❌ False | 12 |
| `HMA_14` | close | +0.0100 | 33% | ❌ False | 12 |
| `HMA_14` | high | +0.0369 | 17% | ❌ False | 12 |
| `HMA_14` | low | -0.0386 | 33% | ❌ False | 12 |
| `HT_TRENDLINE` | close | — | — | — | 0 |
| `HT_TRENDLINE` | high | — | — | — | 0 |
| `HT_TRENDLINE` | low | — | — | — | 0 |
| `JMA_14` | close | +0.0143 | 33% | ❌ False | 12 |
| `JMA_14` | high | +0.0482 | 25% | ❌ False | 12 |
| `JMA_14` | low | -0.0247 | 42% | ❌ False | 12 |
| `KAMA_14` | close | +0.0571 | 25% | ❌ False | 12 |
| `KAMA_14` | high | +0.0204 | 17% | ❌ False | 12 |
| `KAMA_14` | low | +0.0274 | 33% | ❌ False | 12 |
| `LINREG_14` | close | +0.0045 | 25% | ❌ False | 12 |
| `LINREG_14` | high | +0.0368 | 25% | ❌ False | 12 |
| `LINREG_14` | low | -0.0434 | 25% | ❌ False | 12 |
| `LINREGANGLE_14` | close | -0.0441 | 33% | ❌ False | 12 |
| `LINREGANGLE_14` | high | +0.0204 | 17% | ❌ False | 12 |
| `LINREGANGLE_14` | low | -0.0770 | 50% | ❌ False | 12 |
| `LINREGINTERCEPT_14` | close | +0.0474 | 17% | ❌ False | 12 |
| `LINREGINTERCEPT_14` | high | -0.0022 | 8% | ❌ False | 12 |
| `LINREGINTERCEPT_14` | low | +0.0545 | 50% | ❌ False | 12 |
| `LINREGSLOPE_14` | close | -0.0441 | 33% | ❌ False | 12 |
| `LINREGSLOPE_14` | high | +0.0204 | 17% | ❌ False | 12 |
| `LINREGSLOPE_14` | low | -0.0770 | 50% | ❌ False | 12 |
| `MAMA` | close | — | — | — | 0 |
| `MAMA` | high | — | — | — | 0 |
| `MAMA` | low | — | — | — | 0 |
| `MCGD_14` | close | -0.0022 | 17% | ❌ False | 12 |
| `MCGD_14` | high | -0.0269 | 25% | ❌ False | 12 |
| `MCGD_14` | low | -0.0342 | 17% | ❌ False | 12 |
| `MEDPRICE` | close | +0.0117 | 25% | ❌ False | 12 |
| `MEDPRICE` | high | +0.0302 | 25% | ❌ False | 12 |
| `MEDPRICE` | low | -0.0224 | 25% | ❌ False | 12 |
| `MIDPOINT_14` | close | +0.0444 | 17% | ❌ False | 12 |
| `MIDPOINT_14` | high | +0.0206 | 17% | ❌ False | 12 |
| `MIDPOINT_14` | low | +0.0236 | 25% | ❌ False | 12 |
| `MIDPRICE_14` | close | +0.0429 | 25% | ❌ False | 12 |
| `MIDPRICE_14` | high | +0.0314 | 33% | ❌ False | 12 |
| `MIDPRICE_14` | low | +0.0196 | 17% | ❌ False | 12 |
| `OHLC4` | close | +0.0286 | 17% | ❌ False | 12 |
| `OHLC4` | high | +0.0452 | 17% | ❌ False | 12 |
| `OHLC4` | low | -0.0111 | 25% | ❌ False | 12 |
| `PWMA_14` | close | +0.0341 | 25% | ❌ False | 12 |
| `PWMA_14` | high | +0.0168 | 17% | ❌ False | 12 |
| `PWMA_14` | low | +0.0155 | 33% | ❌ False | 12 |
| `RAINBOW` | close | +0.0400 | 33% | ❌ False | 12 |
| `RAINBOW` | high | +0.0555 | 17% | ❌ False | 12 |
| `RAINBOW` | low | +0.0062 | 25% | ❌ False | 12 |
| `RMA_14` | close | +0.0608 | 33% | ❌ False | 12 |
| `RMA_14` | high | +0.0139 | 17% | ❌ False | 12 |
| `RMA_14` | low | +0.0536 | 42% | ❌ False | 12 |
| `SINWMA_14` | close | +0.0378 | 17% | ❌ False | 12 |
| `SINWMA_14` | high | +0.0149 | 8% | ❌ False | 12 |
| `SINWMA_14` | low | +0.0209 | 42% | ❌ False | 12 |
| `SMA_14` | close | +0.0418 | 17% | ❌ False | 12 |
| `SMA_14` | high | +0.0132 | 17% | ❌ False | 12 |
| `SMA_14` | low | +0.0275 | 42% | ❌ False | 12 |
| `SSF_14` | close | +0.0216 | 25% | ❌ False | 12 |
| `SSF_14` | high | +0.0313 | 25% | ❌ False | 12 |
| `SSF_14` | low | -0.0203 | 33% | ❌ False | 12 |
| `SWMA_14` | close | +0.0372 | 25% | ❌ False | 12 |
| `SWMA_14` | high | +0.0152 | 8% | ❌ False | 12 |
| `SWMA_14` | low | +0.0196 | 42% | ❌ False | 12 |
| `TRIMA_14` | close | +0.0389 | 17% | ❌ False | 12 |
| `TRIMA_14` | high | +0.0129 | 8% | ❌ False | 12 |
| `TRIMA_14` | low | +0.0244 | 42% | ❌ False | 12 |
| `TSF_14` | close | +0.0011 | 25% | ❌ False | 12 |
| `TSF_14` | high | +0.0410 | 17% | ❌ False | 12 |
| `TSF_14` | low | -0.0509 | 42% | ❌ False | 12 |
| `TYPPRICE` | close | +0.0117 | 25% | ❌ False | 12 |
| `TYPPRICE` | high | +0.0302 | 25% | ❌ False | 12 |
| `TYPPRICE` | low | -0.0224 | 25% | ❌ False | 12 |
| `VIDYA_14` | close | +0.0566 | 25% | ❌ False | 12 |
| `VIDYA_14` | high | +0.0077 | 17% | ❌ False | 12 |
| `VIDYA_14` | low | +0.0494 | 33% | ❌ False | 12 |
| `VWMA_14` | close | +0.0265 | 17% | ❌ False | 12 |
| `VWMA_14` | high | +0.0349 | 17% | ❌ False | 12 |
| `VWMA_14` | low | -0.0034 | 8% | ❌ False | 12 |
| `WCP` | close | +0.0117 | 25% | ❌ False | 12 |
| `WCP` | high | +0.0302 | 25% | ❌ False | 12 |
| `WCP` | low | -0.0224 | 25% | ❌ False | 12 |
| `ZLMA_14` | close | +0.0190 | 25% | ❌ False | 12 |
| `ZLMA_14` | high | +0.0429 | 25% | ❌ False | 12 |
| `ZLMA_14` | low | -0.0314 | 33% | ❌ False | 12 |

**غير قابل للاختبار إطلاقاً** (خطأ قبل أي نافذة — عمود مخرجات غير متوقَّع أو نافذة 32 شمعة أقصر من الحدّ الأدنى الداخلي للمؤشّر): `AO`، `KVO`، `MMAR`، `T3_14`، `TEMA_14`، `VFI`.

**نجح الحساب لكن NaN لكل النوافذ الـ12** (نفس سبب النافذة القصيرة، أو تعذّر داخلي بلا استثناء صريح): `AVOLUME`، `HT_PHASOR`، `HT_TRENDLINE`، `MAMA`.

**`PVOL` قيمة ثابتة صفر** (`close×volume`، وبما أنّ `close` الحالي دائماً صفر في هذا التمثيل — راجع القسم ١٤ — فالناتج صفر دائماً؛ `compute_ic` تُرجع 0.0 صراحةً).

### 🎯 نتيجة ثالثة تعبر معيار القبول فعلياً

`AOBV` (Archer On-Balance Volume) على هدف `high` يحقّق معيار المحور الثلاثي بالكامل:

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign عبر ١٢ نافذة |
|---|---|---|---|---|
| `AOBV` (OBV) | high | +0.0977 | 50% (6/12) | ✅ موجب في كل الـ12 |

تفصيل IC لكل نافذة: [+0.075, +0.191, +0.166, +0.084, +0.070, +0.025, +0.051, +0.113, +0.073, +0.116, +0.110, +0.098] — موجبة في الـ12 كلّها، 6 معنويّة (p<0.05).

**نفس التحفّظ المذكور في القسم ١٤**: هذا إعداد أضيق من H003 (14 عملة/12 نافذة مقابل 50/30)، و231 محاولة في هذه الدفعة (348 تراكمياً عبر الدفعات الثلاث) تجعل احتمال إيجابيّ كاذب واحد على الأقل بالصدفة غير مهمَل. يُسجَّل "قيد الاختبار" — يحتاج نفس ترقية H003 (توسيع الأصول والنوافذ) مع `CVI_14`/`ENTROPY_14` من القسم ١٤ قبل الحكم النهائي، وليس بمعزل عنهما — كون الاختبارات المتعدّدة تراكمية عبر الدفعات كلّها لا لكلّ دفعة بمفردها.

### تسجيل الدفعة الثالثة كاملة في سجلّ التجارب

In [ ]:
# @title
PANDAS_TA_BATCH3_RESULTS = [
    ("QSTICK_14", {'length': 14}, "close", -0.03570876850347049, 0.09123353876791418, 0.3333333333333333, False, 12, "مرفوضة"),
    ("QSTICK_14", {'length': 14}, "high", 0.02002339029230269, 0.12283387042917537, 0.25, False, 12, "مرفوضة"),
    ("QSTICK_14", {'length': 14}, "low", -0.04506553695793473, 0.08580033645833739, 0.3333333333333333, False, 12, "مرفوضة"),
    ("APO", {}, "close", -0.07276567201405548, 0.1365429497676077, 0.25, False, 12, "مرفوضة"),
    ("APO", {}, "high", 0.020239004951451176, 0.1538755758888186, 0.25, False, 12, "مرفوضة"),
    ("APO", {}, "low", -0.11284731804902741, 0.07792521426789584, 0.3333333333333333, True, 12, "مرفوضة"),
    ("ERI_14", {'length': 14}, "close", -0.039745806585825265, 0.14154309455950387, 0.3333333333333333, False, 12, "مرفوضة"),
    ("ERI_14", {'length': 14}, "high", 0.006945715505871863, 0.1494006772523736, 0.08333333333333333, False, 12, "مرفوضة"),
    ("ERI_14", {'length': 14}, "low", -0.04979604208527475, 0.09516470696223021, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MOM_14", {'length': 14}, "close", -0.03570442315703905, 0.09122583789892366, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MOM_14", {'length': 14}, "high", 0.020014971240082603, 0.12278973504789359, 0.25, False, 12, "مرفوضة"),
    ("MOM_14", {'length': 14}, "low", -0.045101878004444845, 0.08579751891742869, 0.3333333333333333, False, 12, "مرفوضة"),
    ("SLOPE_14", {'length': 14}, "close", -0.03570442315703905, 0.09122583789892366, 0.3333333333333333, False, 12, "مرفوضة"),
    ("SLOPE_14", {'length': 14}, "high", 0.020014971240082603, 0.12278973504789359, 0.25, False, 12, "مرفوضة"),
    ("SLOPE_14", {'length': 14}, "low", -0.045101878004444845, 0.08579751891742869, 0.3333333333333333, False, 12, "مرفوضة"),
    ("VWMACD", {}, "close", 0.0021105180987888972, 0.05650084247829642, 0.08333333333333333, False, 12, "مرفوضة"),
    ("VWMACD", {}, "high", -0.023159819818064507, 0.06841329202167838, 0.08333333333333333, False, 12, "مرفوضة"),
    ("VWMACD", {}, "low", 0.012262357462523145, 0.04922249833981974, 0.08333333333333333, False, 12, "مرفوضة"),
    ("ABERRATION_14", {'length': 14}, "close", 0.04245939006325333, 0.12745939382557395, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ABERRATION_14", {'length': 14}, "high", 0.013630956270205198, 0.12917930277277426, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ABERRATION_14", {'length': 14}, "low", 0.02743233531601633, 0.1276502722005386, 0.4166666666666667, False, 12, "مرفوضة"),
    ("ACCBANDS_14", {'length': 14}, "close", 0.04183556400886377, 0.12918048624931866, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ACCBANDS_14", {'length': 14}, "high", 0.013225771518808604, 0.12938605484642732, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ACCBANDS_14", {'length': 14}, "low", 0.027531815082911703, 0.12811352998790854, 0.4166666666666667, False, 12, "مرفوضة"),
    ("AVOLUME", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("AVOLUME", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("AVOLUME", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("CE", {}, "close", 0.0646614612176516, 0.09337549786637629, 0.3333333333333333, False, 12, "مرفوضة"),
    ("CE", {}, "high", 0.08079704677955662, 0.12267267065665773, 0.4166666666666667, False, 12, "مرفوضة"),
    ("CE", {}, "low", -0.01675398327776055, 0.09477369413265273, 0.25, False, 12, "مرفوضة"),
    ("DONCHIAN_14", {'lower_length': 14, 'upper_length': 14}, "close", 0.04287319402086368, 0.1217615225903859, 0.25, False, 12, "مرفوضة"),
    ("DONCHIAN_14", {'lower_length': 14, 'upper_length': 14}, "high", 0.0313993417751376, 0.11966340289450156, 0.3333333333333333, False, 12, "مرفوضة"),
    ("DONCHIAN_14", {'lower_length': 14, 'upper_length': 14}, "low", 0.019637618931416936, 0.12143119199270695, 0.16666666666666666, False, 12, "مرفوضة"),
    ("HWC", {}, "close", -0.012620213435519145, 0.12216361938721118, 0.16666666666666666, False, 12, "مرفوضة"),
    ("HWC", {}, "high", 0.029289044620232074, 0.1072266810993725, 0.16666666666666666, False, 12, "مرفوضة"),
    ("HWC", {}, "low", -0.053673274378988244, 0.13530448473960974, 0.5833333333333334, False, 12, "مرفوضة"),
    ("KC_14", {'length': 14}, "close", 0.04910210309675641, 0.12959465372798806, 0.25, False, 12, "مرفوضة"),
    ("KC_14", {'length': 14}, "high", 0.020981845201881465, 0.12828668961924558, 0.3333333333333333, False, 12, "مرفوضة"),
    ("KC_14", {'length': 14}, "low", 0.028064211257698358, 0.11567384259185952, 0.3333333333333333, False, 12, "مرفوضة"),
    ("PDIST", {}, "close", 0.025643410628440205, 0.1070861461967489, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PDIST", {}, "high", 0.09253125303600862, 0.11621014203950136, 0.4166666666666667, False, 12, "مرفوضة"),
    ("PDIST", {}, "low", -0.03600356675433222, 0.057845628715098144, 0.16666666666666666, False, 12, "مرفوضة"),
    ("THERMO_14", {'length': 14}, "close", 0.021394415681689104, 0.10082403532646554, 0.16666666666666666, False, 12, "مرفوضة"),
    ("THERMO_14", {'length': 14}, "high", 0.09412833131969828, 0.08987160237958851, 0.4166666666666667, False, 12, "مرفوضة"),
    ("THERMO_14", {'length': 14}, "low", -0.07941908955722327, 0.06503551027019255, 0.3333333333333333, False, 12, "مرفوضة"),
    ("TRUE_RANGE", {}, "close", 0.033895138165281176, 0.10505546944672317, 0.25, False, 12, "مرفوضة"),
    ("TRUE_RANGE", {}, "high", 0.10468960610603056, 0.0989273099973031, 0.5, False, 12, "مرفوضة"),
    ("TRUE_RANGE", {}, "low", -0.038725747742315804, 0.054534622845850124, 0.16666666666666666, False, 12, "مرفوضة"),
    ("AD", {}, "close", -0.02528076770510974, 0.043475244031477844, 0.0, False, 12, "مرفوضة"),
    ("AD", {}, "high", 0.03995841420242749, 0.07823051963733131, 0.3333333333333333, False, 12, "مرفوضة"),
    ("AD", {}, "low", -0.051239450823274524, 0.08514135178157509, 0.5, False, 12, "مرفوضة"),
    ("ADOSC", {}, "close", -0.050276397238573196, 0.04814996203735043, 0.25, False, 12, "مرفوضة"),
    ("ADOSC", {}, "high", 0.05208643204549405, 0.07993512526531149, 0.3333333333333333, False, 12, "مرفوضة"),
    ("ADOSC", {}, "low", -0.1104878047546967, 0.07512586913369837, 0.3333333333333333, True, 12, "مرفوضة"),
    ("AOBV", {}, "close", -0.008309483313921211, 0.05460482095659849, 0.08333333333333333, False, 12, "مرفوضة"),
    ("AOBV", {}, "high", 0.09770694711554351, 0.044458629571097336, 0.5, True, 12, "قيد الاختبار"),
    ("AOBV", {}, "low", -0.06779317654274096, 0.08265011547712951, 0.25, False, 12, "مرفوضة"),
    ("EFI_14", {'length': 14}, "close", -0.03299129902066147, 0.09063531731799239, 0.25, False, 12, "مرفوضة"),
    ("EFI_14", {'length': 14}, "high", 0.07402862605529403, 0.1337271983316435, 0.4166666666666667, False, 12, "مرفوضة"),
    ("EFI_14", {'length': 14}, "low", -0.07484064444530983, 0.08974783541150122, 0.5, False, 12, "مرفوضة"),
    ("EMV_14", {'length': 14}, "close", -0.013954355976985973, 0.08257686044536702, 0.08333333333333333, False, 12, "مرفوضة"),
    ("EMV_14", {'length': 14}, "high", 0.01928730396074425, 0.09101469655561453, 0.16666666666666666, False, 12, "مرفوضة"),
    ("EMV_14", {'length': 14}, "low", -0.021962803916005455, 0.06019166685015846, 0.25, False, 12, "مرفوضة"),
    ("EOM_14", {'length': 14}, "close", 0.0038821374944557883, 0.04806083990263751, 0.0, False, 12, "مرفوضة"),
    ("EOM_14", {'length': 14}, "high", 0.022429640195312836, 0.060278033659369096, 0.16666666666666666, False, 12, "مرفوضة"),
    ("EOM_14", {'length': 14}, "low", -0.02118510531517849, 0.046377361709833004, 0.08333333333333333, False, 12, "مرفوضة"),
    ("MARKETFI", {}, "close", 0.02973753093674987, 0.06934920697425824, 0.08333333333333333, False, 12, "مرفوضة"),
    ("MARKETFI", {}, "high", 0.10339221516958708, 0.0695913712522191, 0.4166666666666667, False, 12, "مرفوضة"),
    ("MARKETFI", {}, "low", -0.05771998028949093, 0.05503263151132738, 0.08333333333333333, False, 12, "مرفوضة"),
    ("NVI", {}, "close", -0.008675254430916067, 0.05743193995620364, 0.08333333333333333, False, 12, "مرفوضة"),
    ("NVI", {}, "high", 0.017718611412266962, 0.06507896900148241, 0.25, False, 12, "مرفوضة"),
    ("NVI", {}, "low", -0.035592614418654134, 0.05584907308071843, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PVI", {}, "close", 0.0013634280379110283, 0.041419175674603396, 0.0, False, 12, "مرفوضة"),
    ("PVI", {}, "high", 0.011325335187127214, 0.05125972811169651, 0.08333333333333333, False, 12, "مرفوضة"),
    ("PVI", {}, "low", -0.028893849261403075, 0.054860023832091556, 0.08333333333333333, False, 12, "مرفوضة"),
    ("PVOL", {}, "close", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PVOL", {}, "high", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PVOL", {}, "low", 0.0, 0.0, 0.0, False, 12, "مرفوضة"),
    ("PVT", {}, "close", -0.001179636072271458, 0.040366671590438295, 0.0, False, 12, "مرفوضة"),
    ("PVT", {}, "high", -0.01201462083776847, 0.06440635502513958, 0.08333333333333333, False, 12, "مرفوضة"),
    ("PVT", {}, "low", 0.013728982323185741, 0.0402703008653975, 0.0, False, 12, "مرفوضة"),
    ("WAD", {}, "close", -0.0014419756614561959, 0.06947122327212923, 0.0, False, 12, "مرفوضة"),
    ("WAD", {}, "high", 0.01729495996393645, 0.06396602514178232, 0.16666666666666666, False, 12, "مرفوضة"),
    ("WAD", {}, "low", -0.029550456533911213, 0.07048967557941274, 0.25, False, 12, "مرفوضة"),
    ("MAD_14", {'length': 14}, "close", 0.04767440605062571, 0.09888933175005712, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MAD_14", {'length': 14}, "high", 0.0629427159241896, 0.09634053166116026, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MAD_14", {'length': 14}, "low", -0.007786426753848855, 0.09331858783796912, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MD_14", {'length': 14}, "close", 0.04767440605062571, 0.09888933175005712, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MD_14", {'length': 14}, "high", 0.0629427159241896, 0.09634053166116026, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MD_14", {'length': 14}, "low", -0.007786426753848855, 0.09331858783796912, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MEDIAN_14", {'length': 14}, "close", 0.039059915580133574, 0.12250224902477502, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MEDIAN_14", {'length': 14}, "high", 0.010120393899425323, 0.12752663865296426, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MEDIAN_14", {'length': 14}, "low", 0.02628728992293948, 0.116502577983616, 0.4166666666666667, False, 12, "مرفوضة"),
    ("QUANTILE_14", {'length': 14}, "close", 0.039059915580133574, 0.12250224902477502, 0.16666666666666666, False, 12, "مرفوضة"),
    ("QUANTILE_14", {'length': 14}, "high", 0.010120393899425323, 0.12752663865296426, 0.16666666666666666, False, 12, "مرفوضة"),
    ("QUANTILE_14", {'length': 14}, "low", 0.02628728992293948, 0.116502577983616, 0.4166666666666667, False, 12, "مرفوضة"),
    ("STDERR_14", {'length': 14}, "close", 0.04982369785891714, 0.09394968075526251, 0.3333333333333333, False, 12, "مرفوضة"),
    ("STDERR_14", {'length': 14}, "high", 0.06801342756815802, 0.09709456096588835, 0.3333333333333333, False, 12, "مرفوضة"),
    ("STDERR_14", {'length': 14}, "low", -0.009788985469262273, 0.0926304515071321, 0.16666666666666666, False, 12, "مرفوضة"),
    ("STDEV_14", {'length': 14}, "close", 0.04982369785891714, 0.09394968075526251, 0.3333333333333333, False, 12, "مرفوضة"),
    ("STDEV_14", {'length': 14}, "high", 0.06801342756815802, 0.09709456096588835, 0.3333333333333333, False, 12, "مرفوضة"),
    ("STDEV_14", {'length': 14}, "low", -0.009788985469262273, 0.0926304515071321, 0.16666666666666666, False, 12, "مرفوضة"),
    ("VARIANCE_14", {'length': 14}, "close", 0.04982369785891714, 0.09394968075526251, 0.3333333333333333, False, 12, "مرفوضة"),
    ("VARIANCE_14", {'length': 14}, "high", 0.06801342756815802, 0.09709456096588835, 0.3333333333333333, False, 12, "مرفوضة"),
    ("VARIANCE_14", {'length': 14}, "low", -0.009788985469262273, 0.0926304515071321, 0.16666666666666666, False, 12, "مرفوضة"),
    ("DSP_14", {}, "close", -0.04910210309675641, 0.12959465372798806, 0.25, False, 12, "مرفوضة"),
    ("DSP_14", {}, "high", -0.020981845201881465, 0.12828668961924558, 0.3333333333333333, False, 12, "مرفوضة"),
    ("DSP_14", {}, "low", -0.028064211257698358, 0.11567384259185952, 0.3333333333333333, False, 12, "مرفوضة"),
    ("HT_PHASOR", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HT_PHASOR", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HT_PHASOR", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("ALMA_14", {'length': 14}, "close", 0.04700910075145737, 0.12428135505672466, 0.16666666666666666, False, 12, "مرفوضة"),
    ("ALMA_14", {'length': 14}, "high", -0.0008375383091443917, 0.1342578182400714, 0.08333333333333333, False, 12, "مرفوضة"),
    ("ALMA_14", {'length': 14}, "low", 0.052174233532489615, 0.1262839032814232, 0.5, False, 12, "مرفوضة"),
    ("AVGPRICE", {}, "close", 0.028607278558954768, 0.11574743345594404, 0.16666666666666666, False, 12, "مرفوضة"),
    ("AVGPRICE", {}, "high", 0.04524492614345346, 0.08650228441620793, 0.16666666666666666, False, 12, "مرفوضة"),
    ("AVGPRICE", {}, "low", -0.011053259676853522, 0.10725066864365666, 0.25, False, 12, "مرفوضة"),
    ("DEMA_14", {'length': 14}, "close", 0.007830350153076886, 0.12669757248414223, 0.25, False, 12, "مرفوضة"),
    ("DEMA_14", {'length': 14}, "high", 0.03072158835095765, 0.10552515706900702, 0.25, False, 12, "مرفوضة"),
    ("DEMA_14", {'length': 14}, "low", -0.035690182508606186, 0.13588216568282482, 0.4166666666666667, False, 12, "مرفوضة"),
    ("FWMA_14", {'length': 14}, "close", 0.028238734680816058, 0.12686440062208063, 0.3333333333333333, False, 12, "مرفوضة"),
    ("FWMA_14", {'length': 14}, "high", 0.03897690797478288, 0.09317393739681415, 0.3333333333333333, False, 12, "مرفوضة"),
    ("FWMA_14", {'length': 14}, "low", -0.012095344294078256, 0.12615884305632735, 0.3333333333333333, False, 12, "مرفوضة"),
    ("HL2", {}, "close", 0.011728621546681974, 0.11558224963972154, 0.25, False, 12, "مرفوضة"),
    ("HL2", {}, "high", 0.030209550554294477, 0.09014875226269126, 0.25, False, 12, "مرفوضة"),
    ("HL2", {}, "low", -0.022419515366253726, 0.1061936759446316, 0.25, False, 12, "مرفوضة"),
    ("HLC3", {}, "close", 0.011728621546681974, 0.11558224963972154, 0.25, False, 12, "مرفوضة"),
    ("HLC3", {}, "high", 0.030209550554294477, 0.09014875226269126, 0.25, False, 12, "مرفوضة"),
    ("HLC3", {}, "low", -0.022419515366253726, 0.1061936759446316, 0.25, False, 12, "مرفوضة"),
    ("HMA_14", {'length': 14}, "close", 0.00997436510739537, 0.10991531320826266, 0.3333333333333333, False, 12, "مرفوضة"),
    ("HMA_14", {'length': 14}, "high", 0.03687909320262403, 0.07432740829292148, 0.16666666666666666, False, 12, "مرفوضة"),
    ("HMA_14", {'length': 14}, "low", -0.03863814439903998, 0.1158102788114575, 0.3333333333333333, False, 12, "مرفوضة"),
    ("HT_TRENDLINE", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("HT_TRENDLINE", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("HT_TRENDLINE", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("JMA_14", {'length': 14}, "close", 0.014294965471875401, 0.142199267992093, 0.3333333333333333, False, 12, "مرفوضة"),
    ("JMA_14", {'length': 14}, "high", 0.04821136352309618, 0.09818369596116489, 0.25, False, 12, "مرفوضة"),
    ("JMA_14", {'length': 14}, "low", -0.02467546047173493, 0.1230274775289926, 0.4166666666666667, False, 12, "مرفوضة"),
    ("KAMA_14", {'length': 14}, "close", 0.05712804967543308, 0.1172213667833489, 0.25, False, 12, "مرفوضة"),
    ("KAMA_14", {'length': 14}, "high", 0.02043190082494852, 0.11886578250475957, 0.16666666666666666, False, 12, "مرفوضة"),
    ("KAMA_14", {'length': 14}, "low", 0.027412442521369148, 0.090893199887844, 0.3333333333333333, False, 12, "مرفوضة"),
    ("LINREG_14", {'length': 14}, "close", 0.0045476701674684915, 0.11789561056262932, 0.25, False, 12, "مرفوضة"),
    ("LINREG_14", {'length': 14}, "high", 0.0368118876139702, 0.09293352764910467, 0.25, False, 12, "مرفوضة"),
    ("LINREG_14", {'length': 14}, "low", -0.04338051791354531, 0.11135737514568401, 0.25, False, 12, "مرفوضة"),
    ("LINREGANGLE_14", {'length': 14}, "close", -0.04405834050472284, 0.11300977421047835, 0.3333333333333333, False, 12, "مرفوضة"),
    ("LINREGANGLE_14", {'length': 14}, "high", 0.020373203144712578, 0.1325097053161004, 0.16666666666666666, False, 12, "مرفوضة"),
    ("LINREGANGLE_14", {'length': 14}, "low", -0.07696292986533133, 0.1187872757778226, 0.5, False, 12, "مرفوضة"),
    ("LINREGINTERCEPT_14", {'length': 14}, "close", 0.04740128794772394, 0.1229340986135844, 0.16666666666666666, False, 12, "مرفوضة"),
    ("LINREGINTERCEPT_14", {'length': 14}, "high", -0.0021584223112467295, 0.13618720461384848, 0.08333333333333333, False, 12, "مرفوضة"),
    ("LINREGINTERCEPT_14", {'length': 14}, "low", 0.05447047748177918, 0.12414863046750824, 0.5, False, 12, "مرفوضة"),
    ("LINREGSLOPE_14", {'length': 14}, "close", -0.04405834050472284, 0.11300977421047835, 0.3333333333333333, False, 12, "مرفوضة"),
    ("LINREGSLOPE_14", {'length': 14}, "high", 0.020373203144712578, 0.1325097053161004, 0.16666666666666666, False, 12, "مرفوضة"),
    ("LINREGSLOPE_14", {'length': 14}, "low", -0.07696292986533133, 0.1187872757778226, 0.5, False, 12, "مرفوضة"),
    ("MAMA", {}, "close", None, None, None, None, 0, "مرفوضة"),
    ("MAMA", {}, "high", None, None, None, None, 0, "مرفوضة"),
    ("MAMA", {}, "low", None, None, None, None, 0, "مرفوضة"),
    ("MCGD_14", {'length': 14}, "close", -0.0022179126703218213, 0.07899555553157916, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MCGD_14", {'length': 14}, "high", -0.026919378661417834, 0.06480553781410817, 0.25, False, 12, "مرفوضة"),
    ("MCGD_14", {'length': 14}, "low", -0.034211505936833676, 0.07240424135907503, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MEDPRICE", {}, "close", 0.011728621546681974, 0.11558224963972154, 0.25, False, 12, "مرفوضة"),
    ("MEDPRICE", {}, "high", 0.030209550554294477, 0.09014875226269126, 0.25, False, 12, "مرفوضة"),
    ("MEDPRICE", {}, "low", -0.022419515366253726, 0.1061936759446316, 0.25, False, 12, "مرفوضة"),
    ("MIDPOINT_14", {'length': 14}, "close", 0.044374889147783876, 0.13222652318029554, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MIDPOINT_14", {'length': 14}, "high", 0.0205932622183966, 0.12282644508905391, 0.16666666666666666, False, 12, "مرفوضة"),
    ("MIDPOINT_14", {'length': 14}, "low", 0.023641724155954263, 0.1320157418054119, 0.25, False, 12, "مرفوضة"),
    ("MIDPRICE_14", {'length': 14}, "close", 0.04287319402086368, 0.1217615225903859, 0.25, False, 12, "مرفوضة"),
    ("MIDPRICE_14", {'length': 14}, "high", 0.0313993417751376, 0.11966340289450156, 0.3333333333333333, False, 12, "مرفوضة"),
    ("MIDPRICE_14", {'length': 14}, "low", 0.019637618931416936, 0.12143119199270695, 0.16666666666666666, False, 12, "مرفوضة"),
    ("OHLC4", {}, "close", 0.028607278558954768, 0.11574743345594404, 0.16666666666666666, False, 12, "مرفوضة"),
    ("OHLC4", {}, "high", 0.04524492614345346, 0.08650228441620793, 0.16666666666666666, False, 12, "مرفوضة"),
    ("OHLC4", {}, "low", -0.011053259676853522, 0.10725066864365666, 0.25, False, 12, "مرفوضة"),
    ("PWMA_14", {'length': 14}, "close", 0.0340597074452673, 0.12937695691928117, 0.25, False, 12, "مرفوضة"),
    ("PWMA_14", {'length': 14}, "high", 0.01681414773852905, 0.12967979301507146, 0.16666666666666666, False, 12, "مرفوضة"),
    ("PWMA_14", {'length': 14}, "low", 0.015459988014012334, 0.11992304397928796, 0.3333333333333333, False, 12, "مرفوضة"),
    ("RAINBOW", {}, "close", 0.0399666306116718, 0.11981192039626291, 0.3333333333333333, False, 12, "مرفوضة"),
    ("RAINBOW", {}, "high", 0.05553523114408639, 0.0838889627443347, 0.16666666666666666, False, 12, "مرفوضة"),
    ("RAINBOW", {}, "low", 0.0062434195947869385, 0.11360642430258432, 0.25, False, 12, "مرفوضة"),
    ("RMA_14", {'length': 14}, "close", 0.060791149871880584, 0.12496770085209095, 0.3333333333333333, False, 12, "مرفوضة"),
    ("RMA_14", {'length': 14}, "high", 0.013919154364356742, 0.14150989914601425, 0.16666666666666666, False, 12, "مرفوضة"),
    ("RMA_14", {'length': 14}, "low", 0.05357733456562384, 0.09529507679274685, 0.4166666666666667, False, 12, "مرفوضة"),
    ("SINWMA_14", {'length': 14}, "close", 0.037846748331133065, 0.12943364282981418, 0.16666666666666666, False, 12, "مرفوضة"),
    ("SINWMA_14", {'length': 14}, "high", 0.014927210498740025, 0.12844207335634222, 0.08333333333333333, False, 12, "مرفوضة"),
    ("SINWMA_14", {'length': 14}, "low", 0.020870802609752815, 0.12583214103411364, 0.4166666666666667, False, 12, "مرفوضة"),
    ("SMA_14", {'length': 14}, "close", 0.04183556400886377, 0.12918048624931866, 0.16666666666666666, False, 12, "مرفوضة"),
    ("SMA_14", {'length': 14}, "high", 0.013225771518808604, 0.12938605484642732, 0.16666666666666666, False, 12, "مرفوضة"),
    ("SMA_14", {'length': 14}, "low", 0.027531815082911703, 0.12811352998790854, 0.4166666666666667, False, 12, "مرفوضة"),
    ("SSF_14", {'length': 14}, "close", 0.02162580149658884, 0.12818668838545275, 0.25, False, 12, "مرفوضة"),
    ("SSF_14", {'length': 14}, "high", 0.03133272928319541, 0.09757559324965526, 0.25, False, 12, "مرفوضة"),
    ("SSF_14", {'length': 14}, "low", -0.0202952836154531, 0.12863621617963872, 0.3333333333333333, False, 12, "مرفوضة"),
    ("SWMA_14", {'length': 14}, "close", 0.037240817063478995, 0.1295248835216147, 0.25, False, 12, "مرفوضة"),
    ("SWMA_14", {'length': 14}, "high", 0.015161580434554928, 0.12892449697421857, 0.08333333333333333, False, 12, "مرفوضة"),
    ("SWMA_14", {'length': 14}, "low", 0.019633298891739256, 0.12475791965420811, 0.4166666666666667, False, 12, "مرفوضة"),
    ("TRIMA_14", {'length': 14}, "close", 0.03888283129792706, 0.1282096582116507, 0.16666666666666666, False, 12, "مرفوضة"),
    ("TRIMA_14", {'length': 14}, "high", 0.01293079658742789, 0.1296404380737621, 0.08333333333333333, False, 12, "مرفوضة"),
    ("TRIMA_14", {'length': 14}, "low", 0.024377037853460317, 0.12503099351661995, 0.4166666666666667, False, 12, "مرفوضة"),
    ("TSF_14", {'length': 14}, "close", 0.00106533969799728, 0.10920563240713389, 0.25, False, 12, "مرفوضة"),
    ("TSF_14", {'length': 14}, "high", 0.041010983262635387, 0.09081375539373379, 0.16666666666666666, False, 12, "مرفوضة"),
    ("TSF_14", {'length': 14}, "low", -0.05094092774657855, 0.10726052716182248, 0.4166666666666667, False, 12, "مرفوضة"),
    ("TYPPRICE", {}, "close", 0.011728621546681974, 0.11558224963972154, 0.25, False, 12, "مرفوضة"),
    ("TYPPRICE", {}, "high", 0.030209550554294477, 0.09014875226269126, 0.25, False, 12, "مرفوضة"),
    ("TYPPRICE", {}, "low", -0.022419515366253726, 0.1061936759446316, 0.25, False, 12, "مرفوضة"),
    ("VIDYA_14", {'length': 14}, "close", 0.05656237828349237, 0.12759176124141458, 0.25, False, 12, "مرفوضة"),
    ("VIDYA_14", {'length': 14}, "high", 0.007726166884007072, 0.13022437434819997, 0.16666666666666666, False, 12, "مرفوضة"),
    ("VIDYA_14", {'length': 14}, "low", 0.049435603774669344, 0.08188373394862852, 0.3333333333333333, False, 12, "مرفوضة"),
    ("VWMA_14", {'length': 14}, "close", 0.026511427020963572, 0.0602543523936109, 0.16666666666666666, False, 12, "مرفوضة"),
    ("VWMA_14", {'length': 14}, "high", 0.0349135371687587, 0.054600892837683095, 0.16666666666666666, False, 12, "مرفوضة"),
    ("VWMA_14", {'length': 14}, "low", -0.003356096847178315, 0.06534247784857798, 0.08333333333333333, False, 12, "مرفوضة"),
    ("WCP", {}, "close", 0.011728621546681974, 0.11558224963972154, 0.25, False, 12, "مرفوضة"),
    ("WCP", {}, "high", 0.030209550554294477, 0.09014875226269126, 0.25, False, 12, "مرفوضة"),
    ("WCP", {}, "low", -0.022419515366253726, 0.1061936759446316, 0.25, False, 12, "مرفوضة"),
    ("ZLMA_14", {'length': 14}, "close", 0.01902021104296262, 0.10890154575228456, 0.25, False, 12, "مرفوضة"),
    ("ZLMA_14", {'length': 14}, "high", 0.04293000212982079, 0.0844077921886399, 0.25, False, 12, "مرفوضة"),
    ("ZLMA_14", {'length': 14}, "low", -0.03135355015638555, 0.1227760485799256, 0.3333333333333333, False, 12, "مرفوضة")
]

for name, params, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok, status in PANDAS_TA_BATCH3_RESULTS:
    if status == "قيد الاختبار":
        notes = (
            "عبر معيار المحور الثلاثي فعلياً (consistent_sign=True، "
            "frac_significant>=0.34، n_ok=12) على 14 عملة/12 نافذة — لكن "
            "إعداد أضيق من H003 (50 عملة/30 نافذة)، واختبارات متعدّدة تراكمية "
            "عبر 3 دفعات (348 محاولة إجمالاً). يحتاج نفس ترقية H003 مع "
            "CVI_14/ENTROPY_14 معاً (لا بمعزل) قبل التصنيف النهائي."
        )
    elif name == "PVOL":
        notes = (
            "قيمته الأخيرة = close الحالي × volume، وclose مُرسى صفراً دائماً "
            "في هذا التمثيل، فالناتج صفر ثابت دائماً — لا نتيجة سلبية حقيقية، "
            "بل قيد تقني في هذا الإعداد تحديداً."
        )
    elif n_ok == 0:
        notes = (
            "غير قابل للاختبار بهذا الإعداد فعلياً — إمّا نافذة 32 شمعة أقصر "
            "من الحدّ الأدنى الداخلي للمؤشّر (فيُرجع NaN لكل عيّنة)، أو عمود "
            "مخرجات غير متوقَّع مع نافذة قصيرة (خطأ قبل أي حساب) — لا نتيجة "
            "سلبية حقيقية، بل قيد تقني في هذا الإعداد تحديداً."
        )
    else:
        notes = (
            "مُسجَّلة من تشغيل حقيقي فعلي (14 عملة متنوّعة، 31,604 عيّنة، 12 "
            "نافذة متحرّكة) — لا بيانات وهمية. consistent_sign=False يمنع "
            "القبول بصرف النظر عن frac_significant."
        )
    register_hypothesis(
        hyp_id=f"PANDAS_TA_BATCH3_{name}_{target}",
        hypothesis=(
            f"{name} ({params}) على هدف {target} — الدفعة الثالثة من مرشّحي "
            "pandas_ta (77 مرشّحاً، بعد تصحيح افتراض عدم قابلية المقارنة عبر "
            "الأصول)؛ راجع signal_discovery_lab.ipynb القسم ١٥ للمنهجية الكاملة."
        ),
        source="literature_mining",
        status=status,
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=notes,
    )

print(f'✅ سُجِّلت {len(PANDAS_TA_BATCH3_RESULTS)} توليفة (1 قيد الاختبار، البقية مرفوضة).')

## ١٦) ترقية الثلاثة "قيد الاختبار" إلى 50 أصلاً/30 نافذة — النتيجة النهائية

القسمان ١٤/١٥ سجّلا `CVI_14`/high، `ENTROPY_14`/low، `AOBV`/high "قيد الاختبار" مع تحفّظ صريح: الإعداد (14 عملة، 12 نافذة) أضيق من H003 (50 عملة، 30 نافذة)، واختبارات متعدّدة تراكمية (348 محاولة). هذا القسم يطبّق نفس ترقية H003 فعلياً — أُعيد بناء dataset حقيقي بـ50 أصلاً (نفس الأصول الـ14 الأصلية + 36 أصلاً إضافياً من نفس مصدر Drive، بـ`exclude_from_features=[]`) و30 نافذة متحرّكة، ثم أُعيد قياس الثلاثة بالضبط بنفس البارامترات.

### النتيجة: **الثلاثة فشلوا في اجتياز الترقية — لا نافذة اتساق واحدة نجت**

| المرشّح | الهدف | mean_ic (50/30) | frac_significant | consistent_sign |
|---|---|---|---|---|
| `CVI_14` | high | +0.1091 | 67% | ❌ False (4 نوافذ من 30 بإشارة معاكسة) |
| `ENTROPY_14` | low | -0.0905 | 60% | ❌ False (4 نوافذ من 30 بإشارة معاكسة) |
| `AOBV` | high | +0.0536 | 60% | ❌ False (4 نوافذ من 30 بإشارة معاكسة) |

بالتفصيل — نوافذ الإشارة المعاكسة:

- **`CVI_14`/high**: `mean_ic` لا يزال موجباً وقوياً نسبياً (+0.109، أعلى من نتيجته الأصلية +0.123 حتى)، وأغلب النوافذ (20/30) معنويّة — لكن نافذة واحدة على الأقل عكست الإشارة، فكسرت `consistent_sign` الذي كان `True` بالصدفة على عيّنة الـ12 نافذة الأصغر.
- **`ENTROPY_14`/low**: أوضح مثال على الفخّ — النوافذ 14 و15 و20 و27 انقلبت موجبة (+0.039 إلى +0.079) بعدما كانت **كل الـ12 نافذة الأصلية** سالبة بلا استثناء. عيّنة أكبر كشفت عدم استقرار حقيقي كان مخفيّاً وراء 12 نافذة فقط.
- **`AOBV`/high**: بالمثل، 4 نوافذ (16، 19، 25، 29) سالبة رغم أن الـ12 نافذة الأصلية كانت كلّها موجبة.

### الخلاصة — تأكيد تجريبي لفخّ العيّنة الصغيرة/الاختبارات المتعدّدة

**لا واحد من الثلاثة "قيد الاختبار" نجا من الترقية — الثلاثة يُرقَّون الآن إلى "مرفوضة" نهائياً.** هذه أهمّ نتيجة منهجية في هذا المسار بأكمله (الدفعات ١-٥، 143 مرشّحاً، 428+ محاولة): من بين كل ما اختُبِر على عيّنة 14 عملة/12 نافذة، **صفر** نجا من إعادة القياس على 50 عملة/30 نافذة — تماماً كما حذّرت خطة المشروع من "فخّ الاختبارات المتعدّدة"، وتماماً كما أثبتت تجربة H003 نفسها (لم يُعتمَد إلا بعد التوسّع من 5 إلى 50 أصلاً).

**الدرس العملي لأي عمل مستقبلي مشابه**: نتيجة `consistent_sign=True` على 12 نافذة فقط هي مرشِّح للفحص الإضافي، لا دليل كافٍ للقبول — الحكم النهائي يجب أن ينتظر دائماً نفس مقياس H003 (50 أصلاً على الأقلّ، 30 نافذة) قبل أي اعتماد عملي أو تسجيل "مقبولة". هذا يُغلق ملفّ مرشّحي `SURVEY_CANDIDATES_ROBUST` بالكامل: **H003 (`NATR_14`) يبقى الفرضية الوحيدة المقبولة من مصدر التقلّب/الزخم التقليدي في تاريخ هذا المشروع.**

### تسجيل الترقية النهائية في سجلّ التجارب (يستبدل حالة "قيد الاختبار")

In [ ]:
# @title
UPGRADE_50ASSET_RESULTS = [
    ("PANDAS_TA_BATCH2_CVI_14_high", "CVI_14", {'length': 14}, "high",
     0.10912707561890356, 0.10343947760741, 0.6666666666666666, False, 30),
    ("PANDAS_TA_BATCH2_ENTROPY_14_low", "ENTROPY_14", {'length': 14}, "low",
     -0.09051381781336083, 0.08626260814, 0.6, False, 30),
    ("PANDAS_TA_BATCH3_AOBV_high", "AOBV", {}, "high",
     0.05362558426601376, 0.07281374, 0.6, False, 30),
]

for hyp_id, name, params, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in UPGRADE_50ASSET_RESULTS:
    register_hypothesis(
        hyp_id=hyp_id,
        hypothesis=(
            f"{name} ({params}) على هدف {target} — مُرقّى إلى 50 أصلاً/30 نافذة "
            "(بنفس منهجية H003)؛ يستبدل التسجيل السابق 'قيد الاختبار' على 14 "
            "أصلاً/12 نافذة. راجع signal_discovery_lab.ipynb القسم ١٦."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "فشل اجتياز ترقية H003: consistent_sign=False على 50 أصلاً/30 نافذة "
            "حقيقية (4 نوافذ بإشارة معاكسة من أصل 30) رغم consistent_sign=True "
            "على العيّنة الأصغر (14 أصلاً/12 نافذة) — تأكيد تجريبي لفخّ "
            "الاختبارات المتعدّدة/العيّنة الصغيرة المذكور في خطة المشروع."
        ),
    )

print(f'✅ رُقّيت ونُقِلت {len(UPGRADE_50ASSET_RESULTS)} توليفة من قيد الاختبار إلى مرفوضة نهائياً.')

## ١٧) الدفعتان الرابعة والخامسة — نافذة أطول + إعادة تصنيف "مولّدات الإشارة"، لا نتائج جديدة

بعد إغلاق ملفّ الثلاثة "قيد الاختبار" في القسم ١٦، بقيت فجوتان صغيرتان في تغطية `SURVEY_CANDIDATES_ROBUST`:

**الدفعة الرابعة (11 مرشّحاً)**: مؤشّرات فشلت في الدفعة الثالثة لحاجتها نافذة تاريخ داخلية أطول من 32 شمعة (`AO`/`KVO`/`MMAR`/`T3`/`TEMA`/`VFI`/`AVOLUME`/`HT_PHASOR`/`HT_TRENDLINE`/`MAMA`/`ichimoku`). أُعيد بناء dataset بنافذة 64 شمعة (`window_sizes={'1D': 64}`) على نفس الـ14 عملة، مع تصغير بارامترات `kvo`/`t3` لتلائم الطول الجديد. هذا أنقذ 9 من 11 (`avolume` وحده احتاج أطول من 64 أيضاً).

**الدفعة الخامسة (17 مرشّحاً)**: مراجعة نقدية لقائمة "مولّدات إشارة منطقية/نظام معقّدة" المُستبعَدة سابقاً في الدفعات ١-٣ (`kdj`/`qqe`/`amat`/`cksp`/`decreasing`/`increasing`/`pmax`/`psar`/`sarext`/`smc_sweep`/`squeeze`/`squeeze_pro`/`td_seq`/`ttm_trend`/`hilo`/`supertrend`/`pvr`) — تبيّن أن أغلبها مذبذبات مستمرّة أو مستويات سعرية (بولية جزئياً كحدّ أقصى: 0/1/-1)، تُختبَر مباشرة بنفس الغلاف العام بلا حاجة لتشفير خاص، تماماً كمؤشّرات فئة `overlap` في الدفعة الثالثة. استُبعِد فعلياً فقط `long_run`/`short_run` (تحتاج مدخل سلسلتين صريح)، `tsignals`/`xsignals` (تغليف لإشارة موجودة مسبقاً)، `decay`/`edecay` (تحتاج مدخل سلسلة بولية جاهزة)، و`ma`/`vp` (غامض/ليست سلسلة زمنية).

### نتيجة الدفعة الرابعة (11 مرشّحاً × 3 أهداف، نافذة 64 شمعة)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign | n_ok |
|---|---|---|---|---|---|
| `AO_W64` | close | -0.0654 | 25% | ❌ False | 12 |
| `AO_W64` | high | +0.0114 | 8% | ❌ False | 12 |
| `AO_W64` | low | -0.1069 | 42% | ❌ False | 12 |
| `KVO_W64` | close | -0.0682 | 25% | ❌ False | 12 |
| `KVO_W64` | high | -0.0411 | 25% | ❌ False | 12 |
| `KVO_W64` | low | -0.0476 | 25% | ❌ False | 12 |
| `MMAR_W64` | close | +0.0973 | 50% | ❌ False | 12 |
| `MMAR_W64` | high | +0.0724 | 42% | ❌ False | 12 |
| `MMAR_W64` | low | +0.0381 | 42% | ❌ False | 12 |
| `T3_W64` | close | +0.0648 | 42% | ❌ False | 12 |
| `T3_W64` | high | +0.0627 | 42% | ❌ False | 12 |
| `T3_W64` | low | -0.0024 | 25% | ❌ False | 12 |
| `TEMA_14_W64` | close | +0.0355 | 42% | ❌ False | 12 |
| `TEMA_14_W64` | high | +0.0571 | 33% | ❌ False | 12 |
| `TEMA_14_W64` | low | -0.0346 | 25% | ❌ False | 12 |
| `VFI_W64` | close | -0.0443 | 17% | ❌ False | 12 |
| `VFI_W64` | high | -0.0183 | 25% | ❌ False | 12 |
| `VFI_W64` | low | -0.0547 | 25% | ❌ False | 12 |
| `AVOLUME_W64` | close | — | — | — | 0 |
| `AVOLUME_W64` | high | — | — | — | 0 |
| `AVOLUME_W64` | low | — | — | — | 0 |
| `HT_PHASOR_W64` | close | -0.0285 | 25% | ❌ False | 12 |
| `HT_PHASOR_W64` | high | +0.0137 | 17% | ❌ False | 12 |
| `HT_PHASOR_W64` | low | -0.0500 | 33% | ❌ False | 12 |
| `HT_TRENDLINE_W64` | close | +0.1023 | 58% | ❌ False | 12 |
| `HT_TRENDLINE_W64` | high | +0.0625 | 33% | ❌ False | 12 |
| `HT_TRENDLINE_W64` | low | +0.0707 | 33% | ❌ False | 12 |
| `MAMA_W64` | close | +0.0908 | 50% | ❌ False | 12 |
| `MAMA_W64` | high | +0.0667 | 33% | ❌ False | 12 |
| `MAMA_W64` | low | +0.0338 | 42% | ❌ False | 12 |
| `ICHIMOKU_W64` | close | +0.0795 | 58% | ❌ False | 12 |
| `ICHIMOKU_W64` | high | +0.0693 | 50% | ❌ False | 12 |
| `ICHIMOKU_W64` | low | +0.0259 | 42% | ❌ False | 12 |

**33 من 33 مرفوضة** — `AVOLUME_W64` غير قابل للاختبار حتى بنافذة 64 (NaN لكل عيّنة، يحتاج أطول).

### نتيجة الدفعة الخامسة (17 مرشّحاً × 3 أهداف، نافذة 32 الأصلية)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign | n_ok |
|---|---|---|---|---|---|
| `KDJ_14` | close | -0.0565 | 33% | ❌ False | 12 |
| `KDJ_14` | high | -0.0408 | 33% | ❌ False | 12 |
| `KDJ_14` | low | -0.0424 | 25% | ❌ False | 12 |
| `QQE` | close | — | — | — | 0 |
| `QQE` | high | — | — | — | 0 |
| `QQE` | low | — | — | — | 0 |
| `AMAT` | close | -0.0029 | 8% | ❌ False | 12 |
| `AMAT` | high | -0.0010 | 25% | ❌ False | 12 |
| `AMAT` | low | +0.0100 | 25% | ❌ False | 12 |
| `CKSP` | close | +0.0696 | 25% | ❌ False | 12 |
| `CKSP` | high | +0.0573 | 17% | ❌ False | 12 |
| `CKSP` | low | +0.0075 | 17% | ❌ False | 12 |
| `DECREASING_14` | close | +0.0164 | 17% | ❌ False | 12 |
| `DECREASING_14` | high | -0.0424 | 25% | ❌ False | 12 |
| `DECREASING_14` | low | +0.0325 | 25% | ❌ False | 12 |
| `INCREASING_14` | close | -0.0164 | 17% | ❌ False | 12 |
| `INCREASING_14` | high | +0.0424 | 25% | ❌ False | 12 |
| `INCREASING_14` | low | -0.0325 | 25% | ❌ False | 12 |
| `PMAX` | close | +0.0454 | 17% | ❌ False | 12 |
| `PMAX` | high | -0.0046 | 8% | ❌ False | 12 |
| `PMAX` | low | +0.0218 | 8% | ❌ False | 12 |
| `PSAR` | close | -0.0018 | 8% | ❌ False | 12 |
| `PSAR` | high | -0.0233 | 17% | ❌ False | 12 |
| `PSAR` | low | -0.0002 | 8% | ❌ False | 12 |
| `SAREXT` | close | -0.0148 | 8% | ❌ False | 12 |
| `SAREXT` | high | -0.0255 | 17% | ❌ False | 12 |
| `SAREXT` | low | +0.0267 | 8% | ❌ False | 12 |
| `SMC_SWEEP` | close | +0.0024 | 8% | ❌ False | 12 |
| `SMC_SWEEP` | high | -0.0011 | 0% | ❌ False | 12 |
| `SMC_SWEEP` | low | +0.0107 | 8% | ❌ False | 12 |
| `SQUEEZE` | close | -0.0524 | 42% | ❌ False | 12 |
| `SQUEEZE` | high | +0.0184 | 17% | ❌ False | 12 |
| `SQUEEZE` | low | -0.0907 | 50% | ❌ False | 12 |
| `SQUEEZE_PRO` | close | -0.0524 | 42% | ❌ False | 12 |
| `SQUEEZE_PRO` | high | +0.0184 | 17% | ❌ False | 12 |
| `SQUEEZE_PRO` | low | -0.0907 | 50% | ❌ False | 12 |
| `TD_SEQ` | close | -0.0439 | 25% | ❌ False | 12 |
| `TD_SEQ` | high | -0.0082 | 0% | ❌ False | 12 |
| `TD_SEQ` | low | -0.1011 | 33% | ❌ False | 12 |
| `TTM_TREND` | close | +0.0069 | 25% | ❌ False | 12 |
| `TTM_TREND` | high | -0.0166 | 17% | ❌ False | 12 |
| `TTM_TREND` | low | +0.0459 | 42% | ❌ False | 12 |
| `HILO` | close | +0.0580 | 33% | ❌ False | 12 |
| `HILO` | high | +0.0085 | 17% | ❌ False | 12 |
| `HILO` | low | +0.0494 | 33% | ❌ False | 12 |
| `SUPERTREND` | close | -0.0640 | 25% | ❌ False | 12 |
| `SUPERTREND` | high | -0.0019 | 8% | ❌ False | 12 |
| `SUPERTREND` | low | -0.0626 | 17% | ✅ True | 12 |
| `PVR` | close | +0.0359 | 17% | ❌ False | 12 |
| `PVR` | high | +0.0223 | 8% | ❌ False | 12 |
| `PVR` | low | +0.0161 | 33% | ❌ False | 12 |

**51 من 51 مرفوضة** (`SUPERTREND`/low وحيد بـ`consistent_sign=True` لكن `frac_significant=17%` دون حدّ 34% المطلوب). `QQE` غير قابل للاختبار (NaN لكل عيّنة على هذا التمثيل رغم نجاحه على بيانات تركيبية). ملاحظتان جانبيتان: `DECREASING_14`/`INCREASING_14` تعطيان قيماً متعاكسة تماماً (متوقَّع، أحدهما مكمّل الآخر رياضياً)، و`SQUEEZE`/`SQUEEZE_PRO` تعطيان نتائج مطابقة رقمياً بالكامل — على الأرجح تطابق داخلي في `pandas_ta_classic` لصيغتيهما (نفس ملاحظة `WCP`/`TYPPRICE` في الدفعة الثالثة).

### الخلاصة

84 محاولة إضافية (28 مرشّحاً)، **صفر نتائج جديدة تعبر معيار القبول**. بعد خمس دفعات (154 مرشّحاً فريداً، ~450 محاولة إجمالاً)، وبعد فشل الثلاثة الوحيدين الذين عبروا المعيار أصلاً في ترقية H003 (القسم ١٦)، **مسار `SURVEY_CANDIDATES_ROBUST` مُستنفَد بالكامل عملياً** — لا مبرّر لدفعات إضافية بنفس المنهجية. الخطوة التالية المنطقية الوحيدة المتبقّية هي تغيير المنهجية نفسها (ميزات هندسية جديدة من مصادر خارج pandas_ta، أو اختبار غير خطّي/شرطي بدل IC الخطّي المفرد)، لا توسيع نفس نوع الاختبار.

### تسجيل الدفعتين الرابعة والخامسة في سجلّ التجارب

In [ ]:
# @title
PANDAS_TA_BATCH45_RESULTS = [
    ("B4", "AO_W64", {}, "close", -0.06543125336214048, 0.10827592537421431, 0.25, False, 12),
    ("B4", "AO_W64", {}, "high", 0.01144387430067117, 0.10602941583088246, 0.08333333333333333, False, 12),
    ("B4", "AO_W64", {}, "low", -0.10691609260567636, 0.13475373734473653, 0.4166666666666667, False, 12),
    ("B4", "KVO_W64", {'fast': 10, 'slow': 20, 'signal': 5}, "close", -0.06822619410670613, 0.04621291075681919, 0.25, False, 12),
    ("B4", "KVO_W64", {'fast': 10, 'slow': 20, 'signal': 5}, "high", -0.04108093578661338, 0.0996679409333759, 0.25, False, 12),
    ("B4", "KVO_W64", {'fast': 10, 'slow': 20, 'signal': 5}, "low", -0.047614753616108, 0.08870713543721312, 0.25, False, 12),
    ("B4", "MMAR_W64", {}, "close", 0.09726957526482834, 0.1276187984887914, 0.5, False, 12),
    ("B4", "MMAR_W64", {}, "high", 0.07242375349683404, 0.12535104530710192, 0.4166666666666667, False, 12),
    ("B4", "MMAR_W64", {}, "low", 0.03806282716334721, 0.16355735547453132, 0.4166666666666667, False, 12),
    ("B4", "T3_W64", {'length': 5}, "close", 0.06475180386054226, 0.12577674488087842, 0.4166666666666667, False, 12),
    ("B4", "T3_W64", {'length': 5}, "high", 0.06270619651607463, 0.10297548074439274, 0.4166666666666667, False, 12),
    ("B4", "T3_W64", {'length': 5}, "low", -0.0023827429536868888, 0.1565586878806165, 0.25, False, 12),
    ("B4", "TEMA_14_W64", {'length': 14}, "close", 0.03554816945306035, 0.129910225112799, 0.4166666666666667, False, 12),
    ("B4", "TEMA_14_W64", {'length': 14}, "high", 0.05711624968133062, 0.09486738808816582, 0.3333333333333333, False, 12),
    ("B4", "TEMA_14_W64", {'length': 14}, "low", -0.03463880896737129, 0.1422369260475768, 0.25, False, 12),
    ("B4", "VFI_W64", {'length': 14}, "close", -0.04434869596573487, 0.07645203504188773, 0.16666666666666666, False, 12),
    ("B4", "VFI_W64", {'length': 14}, "high", -0.018301909248188416, 0.10325417599410477, 0.25, False, 12),
    ("B4", "VFI_W64", {'length': 14}, "low", -0.05467838845178604, 0.11139567189472543, 0.25, False, 12),
    ("B4", "AVOLUME_W64", {}, "close", None, None, None, None, 0),
    ("B4", "AVOLUME_W64", {}, "high", None, None, None, None, 0),
    ("B4", "AVOLUME_W64", {}, "low", None, None, None, None, 0),
    ("B4", "HT_PHASOR_W64", {}, "close", -0.02848123042667385, 0.11785179057145506, 0.25, False, 12),
    ("B4", "HT_PHASOR_W64", {}, "high", 0.013690731097592052, 0.10882943266602133, 0.16666666666666666, False, 12),
    ("B4", "HT_PHASOR_W64", {}, "low", -0.05004448013869509, 0.12286515012877783, 0.3333333333333333, False, 12),
    ("B4", "HT_TRENDLINE_W64", {}, "close", 0.10230701629724635, 0.11005713514307676, 0.5833333333333334, False, 12),
    ("B4", "HT_TRENDLINE_W64", {}, "high", 0.06250051876686954, 0.1332614821868969, 0.3333333333333333, False, 12),
    ("B4", "HT_TRENDLINE_W64", {}, "low", 0.070666861546486, 0.14451640680358682, 0.3333333333333333, False, 12),
    ("B4", "MAMA_W64", {}, "close", 0.09078026272024915, 0.13068789200530984, 0.5, False, 12),
    ("B4", "MAMA_W64", {}, "high", 0.06669678937715967, 0.11820043287266518, 0.3333333333333333, False, 12),
    ("B4", "MAMA_W64", {}, "low", 0.033810856323167396, 0.16624214474988055, 0.4166666666666667, False, 12),
    ("B4", "ICHIMOKU_W64", {}, "close", 0.07953559599592774, 0.11909685570979889, 0.5833333333333334, False, 12),
    ("B4", "ICHIMOKU_W64", {}, "high", 0.06925797685873082, 0.11437798068808602, 0.5, False, 12),
    ("B4", "ICHIMOKU_W64", {}, "low", 0.0258503528656521, 0.14783758798540494, 0.4166666666666667, False, 12),
    ("B5", "KDJ_14", {'length': 14}, "close", -0.05647165366821643, 0.09595377616543976, 0.3333333333333333, False, 12),
    ("B5", "KDJ_14", {'length': 14}, "high", -0.04082664446124756, 0.11554860402459699, 0.3333333333333333, False, 12),
    ("B5", "KDJ_14", {'length': 14}, "low", -0.04237100296180238, 0.0984376531849249, 0.25, False, 12),
    ("B5", "QQE", {}, "close", None, None, None, None, 0),
    ("B5", "QQE", {}, "high", None, None, None, None, 0),
    ("B5", "QQE", {}, "low", None, None, None, None, 0),
    ("B5", "AMAT", {}, "close", -0.0029439610952922118, 0.11515185014401867, 0.08333333333333333, False, 12),
    ("B5", "AMAT", {}, "high", -0.0010061018255466253, 0.09208004275529655, 0.25, False, 12),
    ("B5", "AMAT", {}, "low", 0.010035193378778812, 0.10259702177443837, 0.25, False, 12),
    ("B5", "CKSP", {}, "close", 0.06958247240007656, 0.11806548281581222, 0.25, False, 12),
    ("B5", "CKSP", {}, "high", 0.05727624434784109, 0.13613919565426813, 0.16666666666666666, False, 12),
    ("B5", "CKSP", {}, "low", 0.007538926714961188, 0.0926200498466253, 0.16666666666666666, False, 12),
    ("B5", "DECREASING_14", {'length': 14}, "close", 0.01637716243139264, 0.0859394636313059, 0.16666666666666666, False, 12),
    ("B5", "DECREASING_14", {'length': 14}, "high", -0.04236438966326569, 0.10295533285626596, 0.25, False, 12),
    ("B5", "DECREASING_14", {'length': 14}, "low", 0.03250009812910312, 0.08043385023422275, 0.25, False, 12),
    ("B5", "INCREASING_14", {'length': 14}, "close", -0.01637716243139264, 0.0859394636313059, 0.16666666666666666, False, 12),
    ("B5", "INCREASING_14", {'length': 14}, "high", 0.04236438966326569, 0.10295533285626596, 0.25, False, 12),
    ("B5", "INCREASING_14", {'length': 14}, "low", -0.03250009812910312, 0.08043385023422275, 0.25, False, 12),
    ("B5", "PMAX", {}, "close", 0.04540494938444917, 0.10331046656354942, 0.16666666666666666, False, 12),
    ("B5", "PMAX", {}, "high", -0.004550154064069268, 0.12328892147771066, 0.08333333333333333, False, 12),
    ("B5", "PMAX", {}, "low", 0.0217729914930745, 0.06488111094448223, 0.08333333333333333, False, 12),
    ("B5", "PSAR", {}, "close", -0.0018466082529266282, 0.09270873453793131, 0.08333333333333333, False, 12),
    ("B5", "PSAR", {}, "high", -0.02334880017799512, 0.11928593066898964, 0.16666666666666666, False, 12),
    ("B5", "PSAR", {}, "low", -0.00017682761644035225, 0.09759255143093876, 0.08333333333333333, False, 12),
    ("B5", "SAREXT", {}, "close", -0.014790033064512232, 0.05955593592624864, 0.08333333333333333, False, 12),
    ("B5", "SAREXT", {}, "high", -0.02552776427475537, 0.07774351624895635, 0.16666666666666666, False, 12),
    ("B5", "SAREXT", {}, "low", 0.026747066690300355, 0.06584598619378457, 0.08333333333333333, False, 12),
    ("B5", "SMC_SWEEP", {}, "close", 0.002443984531865756, 0.059597012263051814, 0.08333333333333333, False, 12),
    ("B5", "SMC_SWEEP", {}, "high", -0.0011267400079411273, 0.05675555328986917, 0.0, False, 12),
    ("B5", "SMC_SWEEP", {}, "low", 0.010745182301938785, 0.07129458118884976, 0.08333333333333333, False, 12),
    ("B5", "SQUEEZE", {}, "close", -0.052438897585137194, 0.10139233250229066, 0.4166666666666667, False, 12),
    ("B5", "SQUEEZE", {}, "high", 0.018435539840146953, 0.13125247727304198, 0.16666666666666666, False, 12),
    ("B5", "SQUEEZE", {}, "low", -0.09067914807526911, 0.06392021385942821, 0.5, False, 12),
    ("B5", "SQUEEZE_PRO", {}, "close", -0.052438897585137194, 0.10139233250229066, 0.4166666666666667, False, 12),
    ("B5", "SQUEEZE_PRO", {}, "high", 0.018435539840146953, 0.13125247727304198, 0.16666666666666666, False, 12),
    ("B5", "SQUEEZE_PRO", {}, "low", -0.09067914807526911, 0.06392021385942821, 0.5, False, 12),
    ("B5", "TD_SEQ", {}, "close", -0.04388043853779264, 0.13768090349079734, 0.25, False, 12),
    ("B5", "TD_SEQ", {}, "high", -0.00819984616046233, 0.11738732611704823, 0.0, False, 12),
    ("B5", "TD_SEQ", {}, "low", -0.10106446912734217, 0.07645442764063312, 0.3333333333333333, False, 12),
    ("B5", "TTM_TREND", {}, "close", 0.006932490016179731, 0.1304900154497789, 0.25, False, 12),
    ("B5", "TTM_TREND", {}, "high", -0.01663734725463061, 0.09983077926644299, 0.16666666666666666, False, 12),
    ("B5", "TTM_TREND", {}, "low", 0.04589918370129097, 0.1291521378772051, 0.4166666666666667, False, 12),
    ("B5", "HILO", {}, "close", 0.057972712702556974, 0.12361533602125249, 0.3333333333333333, False, 12),
    ("B5", "HILO", {}, "high", 0.008467552786580798, 0.13497619173811423, 0.16666666666666666, False, 12),
    ("B5", "HILO", {}, "low", 0.049445087116868534, 0.1060882166304692, 0.3333333333333333, False, 12),
    ("B5", "SUPERTREND", {}, "close", -0.06403654162117205, 0.06360440806906298, 0.25, False, 12),
    ("B5", "SUPERTREND", {}, "high", -0.0019083523289506361, 0.08332002185341533, 0.08333333333333333, False, 12),
    ("B5", "SUPERTREND", {}, "low", -0.0625861798306865, 0.04506751910735491, 0.16666666666666666, True, 12),
    ("B5", "PVR", {}, "close", 0.03587781277036474, 0.0685176922812411, 0.16666666666666666, False, 12),
    ("B5", "PVR", {}, "high", 0.022327996073856105, 0.06906466012712924, 0.08333333333333333, False, 12),
    ("B5", "PVR", {}, "low", 0.0160530455180445, 0.09953342379201127, 0.3333333333333333, False, 12)
]

CRASHED_45 = ['AVOLUME_W64', 'QQE']

for batch_label, name, params, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in PANDAS_TA_BATCH45_RESULTS:
    notes = (
        "غير قابل للاختبار بهذا الإعداد فعلياً — NaN لكل عيّنة حتى مع نافذة "
        "أطول (64 شمعة) أو على بيانات حقيقية (رغم نجاحه على بيانات تركيبية) "
        "— لا نتيجة سلبية حقيقية، بل قيد تقني."
    ) if name in CRASHED_45 else (
        "مُسجَّلة من تشغيل حقيقي فعلي — لا بيانات وهمية. consistent_sign=False "
        "(أو frac_significant<0.34) يمنع القبول."
    )
    register_hypothesis(
        hyp_id=f"PANDAS_TA_{batch_label}_{name}_{target}",
        hypothesis=(
            f"{name} ({params}) على هدف {target} — الدفعة {'الرابعة (نافذة 64)' if batch_label=='B4' else 'الخامسة (إعادة تصنيف مولّدات إشارة)'}؛ "
            "راجع signal_discovery_lab.ipynb القسم ١٧ للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=notes,
    )

print(f'✅ سُجِّلت {len(PANDAS_TA_BATCH45_RESULTS)} توليفة — كلّها مرفوضة/غير قابلة للاختبار.')

## ٢٠) أوّل بحث ميزات جديدة خارج pandas_ta — مقدّرات تقلّب/سيولة كلاسيكية، مرفوضة

بعد استنفاد pandas_ta بالكامل (الأقسام ١٣-١٨)، أوّل بحث فعلي عن ميزات جديدة كلياً من خارج المكتبة — استناداً إلى [مراجعة الأدبيات الخارجية](../docs/research/external_literature_review.md): مقدّرات تقلّب OHLC كلاسيكية (`Parkinson` 1980، `Garman-Klass` 1980، `Rogers-Satchell` 1991، `Yang-Zhang` 2000) ومقدّرا سيولة/انطباع سعري (`Amihud` 2002، `Roll` 1984) — **لا شيء منها في pandas_ta أصلاً** (تحقّق مباشر عبر `dir(df.ta)`). أُضيفت كميزات اختيارية جديدة في `crypto_data_pipeline_v6.ipynb` (`custom_settings["vol_estimator_windows"]`، نفس أسلوب `FRACTAL`/`RVI`/`FISHER`)، ثمّ اختُبِرت مباشرة على مقياس H003 الكامل (50 أصلاً، 30 نافذة) — بلا حاجة لعيّنة صغيرة أوّلاً بما أن الفحص الأوّلي الرخيص (IC مُجمَّع، لا محور كامل) أعطى مؤشّراً أوّلياً واضحاً على الاتجاه المتوقَّع.

### النتيجة الفعلية (6 مرشّحين × 3 أهداف = 18 محاولة، عبر المحور الكامل مباشرة)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `PARKINSON_14` | close | +0.030 | 33% | ❌ False |
| `PARKINSON_14` | high | +0.107 | 63% | ❌ False |
| `PARKINSON_14` | low | -0.072 | 53% | ❌ False |
| `GK_14` | close | +0.031 | 33% | ❌ False |
| `GK_14` | high | +0.104 | 60% | ❌ False |
| `GK_14` | low | -0.067 | 60% | ❌ False |
| `RS_14` | close | +0.033 | 33% | ❌ False |
| `RS_14` | high | +0.100 | 60% | ❌ False |
| `RS_14` | low | -0.062 | 57% | ❌ False |
| `YZ_14` | close | +0.033 | 37% | ❌ False |
| `YZ_14` | high | +0.101 | 57% | ❌ False |
| `YZ_14` | low | -0.065 | 53% | ❌ False |
| `AMIHUD_14` | close | +0.049 | 57% | ❌ False |
| `AMIHUD_14` | high | +0.009 | 37% | ❌ False |
| `AMIHUD_14` | low | +0.039 | 40% | ❌ False |
| `ROLL_SPREAD_14` | close | -0.018 | 30% | ❌ False |
| `ROLL_SPREAD_14` | high | +0.014 | 20% | ❌ False |
| `ROLL_SPREAD_14` | low | -0.037 | 40% | ❌ False |

**18 من 18 مرفوضة** — `consistent_sign=False` بلا استثناء واحد، حتى `PARKINSON_14`/high (أعلى `frac_significant`، 63%). الفحص الأوّلي الرخيص (IC مُجمَّع عبر كامل التاريخ دفعة واحدة، بلا فصل نوافذ زمنية) كان مضلِّلاً نسبياً — أظهر IC يبدو قوياً (حتى +0.27/-0.30) لكن بلا اتساق اتجاه حقيقي عبر نوافذ منفصلة. `Parkinson`/`GK`/`RS`/`YZ` قريبة جداً من `NATR_14` في كل شيء (بما فيها الفشل)، تدعم أنها تكرار للمعلومة نفسها لا مصدراً جديداً — و`NATR_14` (بصيغة `pandas_ta`/وايلدر) يبقى أفضل مقياس تقلّب متاح لهذا المشروع رغم أن هذه البدائل "الأكفأ إحصائياً" نظرياً في الأدبيات (`Yang-Zhang` خصوصاً).

### تسجيل مقدّرات التقلّب/السيولة الجديدة في سجلّ التجارب

In [ ]:
# @title
NEW_VOL_ESTIMATOR_RESULTS = [
    ("PARKINSON_14", "close", 0.02992833661939738, 0.1270676197841301, 0.3333333333333333, False, 30),
    ("PARKINSON_14", "high", 0.10680268631875736, 0.1013945967901031, 0.6333333333333333, False, 30),
    ("PARKINSON_14", "low", -0.07209898936478121, 0.15442638396396757, 0.5333333333333333, False, 30),
    ("GK_14", "close", 0.03129815932942805, 0.13273436522957593, 0.3333333333333333, False, 30),
    ("GK_14", "high", 0.10379669044738907, 0.10487517952758021, 0.6, False, 30),
    ("GK_14", "low", -0.06679503997070145, 0.15638088652064525, 0.6, False, 30),
    ("RS_14", "close", 0.03339002168434952, 0.13756864443905528, 0.3333333333333333, False, 30),
    ("RS_14", "high", 0.09979059130858414, 0.10643382968275636, 0.6, False, 30),
    ("RS_14", "low", -0.06174654671319894, 0.15897538176223353, 0.5666666666666667, False, 30),
    ("YZ_14", "close", 0.032957862319016235, 0.1345545630228724, 0.36666666666666664, False, 30),
    ("YZ_14", "high", 0.10145355386300943, 0.10560520888157253, 0.5666666666666667, False, 30),
    ("YZ_14", "low", -0.06479148625135485, 0.1573597275583901, 0.5333333333333333, False, 30),
    ("AMIHUD_14", "close", 0.04880871130039303, 0.09980902885443049, 0.5666666666666667, False, 30),
    ("AMIHUD_14", "high", 0.009130348373702255, 0.13448221819290695, 0.36666666666666664, False, 30),
    ("AMIHUD_14", "low", 0.03872758227292168, 0.09244475774103278, 0.4, False, 30),
    ("ROLL_SPREAD_14", "close", -0.01772591501143895, 0.08146744689245995, 0.3, False, 30),
    ("ROLL_SPREAD_14", "high", 0.014242862737265048, 0.08160421627407749, 0.2, False, 30),
    ("ROLL_SPREAD_14", "low", -0.036517937947357824, 0.0828824705278162, 0.4, False, 30)
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in NEW_VOL_ESTIMATOR_RESULTS:
    register_hypothesis(
        hyp_id=f"NEW_VOL_ESTIMATOR_{name}_{target}",
        hypothesis=(
            f"{name} على هدف {target} — أوّل بحث ميزات خارج pandas_ta (مقدّرات "
            "تقلّب/سيولة كلاسيكية: Parkinson/Garman-Klass/Rogers-Satchell/"
            "Yang-Zhang/Amihud/Roll). اختُبِر مباشرة على 50 أصلاً/30 نافذة "
            "(مقياس H003 الكامل). راجع signal_discovery_lab.ipynb القسم ٢٠ "
            "للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "مُسجَّلة من تشغيل حقيقي فعلي (50 أصلاً، 30 نافذة — مقياس H003 الكامل "
            "مباشرة، بلا ترقية عيّنة صغيرة). consistent_sign=False يمنع القبول. "
            "Parkinson/GK/RS/YZ قريبة جداً من NATR_14 (على الأرجح تكرار "
            "للمعلومة نفسها)؛ Amihud/Roll أضعف ومختلفان (سيولة لا تقلّب)."
        ),
    )

print(f'✅ سُجِّلت {len(NEW_VOL_ESTIMATOR_RESULTS)} توليفة — كلّها مرفوضة.')

## ٢٢) ثاني بحث ميزات جديدة خارج pandas_ta — أسّ هيرست وVariance Ratio، مرفوضة

بخلاف مقدّرات التقلّب في القسم السابق (تقيس حجم التقلّب فقط)، هذان المقدّران يختبران **مباشرة** فرضية الانعكاس مقابل الاستمرار: أسّ هيرست (`Hurst` 1951، عبر تدرّج التباين) وVariance Ratio (لو-ماكينلي `Lo-MacKinlay` 1988) — **لا شيء منهما في pandas_ta أصلاً** (تحقّق مباشر عبر `pandas_ta_classic.Category`، 193 دالة، لا `hurst` ولا `variance_ratio`). أُضيفا كميزتين اختياريتين جديدتين في `crypto_data_pipeline_v6.ipynb` (`custom_settings["mean_reversion_windows"]`)، مع اختبار ذاتي يتحقّق من مطابقتهما لتعريفهما الإحصائي على سلاسل AR(1) صناعية معروفة الخاصّية (انعكاس/عشوائي/اتجاه) — أقوى من فحص الحساسية النوعية المُستخدَم سابقاً. اختُبِرا مباشرة على مقياس H003 الكامل (50 أصلاً، 30 نافذة) بلا فحص أوّلي رخيص منفصل.

### النتيجة الفعلية (2 مرشّحين × 2 نافذة × 3 أهداف = 12 محاولة، عبر المحور الكامل مباشرة)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `VR_20` | close | +0.005 | 23% | ❌ False |
| `VR_20` | high | +0.008 | 37% | ❌ False |
| `VR_20` | low | -0.007 | 33% | ❌ False |
| `HURST_20` | close | +0.003 | 23% | ❌ False |
| `HURST_20` | high | +0.014 | 30% | ❌ False |
| `HURST_20` | low | -0.012 | 37% | ❌ False |
| `VR_40` | close | -0.005 | 27% | ❌ False |
| `VR_40` | high | +0.005 | 27% | ❌ False |
| `VR_40` | low | -0.007 | 20% | ❌ False |
| `HURST_40` | close | +0.001 | 30% | ❌ False |
| `HURST_40` | high | +0.009 | 27% | ❌ False |
| `HURST_40` | low | -0.008 | 33% | ❌ False |

**12 من 12 مرفوضة — أوضح بكثير من مقدّرات التقلّب.** `mean_ic` هنا شبه صفري (0.001-0.014 مطلقاً) مقابل 0.03-0.11 لمقدّرات التقلّب، و`frac_significant` منخفض (20-37%) بلا أي اقتراب من عتبة القبول. دليل إضافي (بعد `Fractal_reversal_w5` في H003) على أن أثر H003 خاصّ بالتقلّب المُطبَّع تحديداً لا انعكاساً هندسياً عاماً — راجع [التوثيق الكامل](../docs/research/hurst_variance_ratio_features.md) للمنهجية والتفسيرين المحتملين.

### تسجيل أسّ هيرست وVariance Ratio في سجلّ التجارب

In [ ]:
# @title
NEW_MEAN_REVERSION_RESULTS = [
    ("VR_20", "close", 0.005220627260358288, 0.09851931689845914, 0.23333333333333334, False, 30),
    ("VR_20", "high", 0.00819016294153498, 0.09981861514749675, 0.36666666666666664, False, 30),
    ("VR_20", "low", -0.007120549143527324, 0.09799961527288621, 0.3333333333333333, False, 30),
    ("HURST_20", "close", 0.0028897729997440155, 0.08882970488962057, 0.23333333333333334, False, 30),
    ("HURST_20", "high", 0.013580908916682056, 0.08476159675897824, 0.3, False, 30),
    ("HURST_20", "low", -0.011609850761688384, 0.09540086748541783, 0.36666666666666664, False, 30),
    ("VR_40", "close", -0.004968281265069995, 0.07970990705542111, 0.26666666666666666, False, 30),
    ("VR_40", "high", 0.004550460234386033, 0.09932256510382896, 0.26666666666666666, False, 30),
    ("VR_40", "low", -0.007413970505901109, 0.09743239815343047, 0.2, False, 30),
    ("HURST_40", "close", 0.000514687582644432, 0.0922219856676393, 0.3, False, 30),
    ("HURST_40", "high", 0.008901644426768444, 0.1006994768812442, 0.26666666666666666, False, 30),
    ("HURST_40", "low", -0.007770078557035034, 0.105346199404061, 0.3333333333333333, False, 30)
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in NEW_MEAN_REVERSION_RESULTS:
    register_hypothesis(
        hyp_id=f"NEW_MEAN_REVERSION_{name}_{target}",
        hypothesis=(
            f"{name} على هدف {target} — ثاني بحث ميزات خارج pandas_ta (أسّ "
            "هيرست وVariance Ratio، لو-ماكينلي 1988، يختبران الانعكاس/الاستمرار "
            "مباشرة بخلاف NATR_14 الذي يقيس حجم التقلّب فقط). اختُبِر مباشرة "
            "على 50 أصلاً/30 نافذة (مقياس H003 الكامل). راجع "
            "signal_discovery_lab.ipynb القسم ٢٢ للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "مُسجَّلة من تشغيل حقيقي فعلي (50 أصلاً، 30 نافذة — مقياس H003 الكامل "
            "مباشرة). consistent_sign=False وmean_ic شبه صفري (أوضح رفضاً من "
            "مقدّرات التقلّب السابقة). يدعم أن أثر H003 خاصّ بالتقلّب المُطبَّع "
            "لا انعكاساً هندسياً عاماً — نفس استنتاج Fractal_reversal_w5."
        ),
    )

print(f'✅ سُجِّلت {len(NEW_MEAN_REVERSION_RESULTS)} توليفة — كلّها مرفوضة.')

## ٢٤) ثالث بحث ميزات جديدة خارج pandas_ta — نسبة التقلّب القفزي (Bipower Variation)، مرفوضة

سؤال ثالث مختلف كلياً عن حجم التقلّب (`NATR`) واتجاهه (Hurst/VR): هل تحرّك السعر انتشار مستمرّ سلس أم قفزة/صدمة مفاجئة؟ `Bipower Variation` (`Barndorff-Nielsen & Shephard` 2004) يفصل التباين المُحقَّق إلى مكوّن مستمرّ (حاصل ضرب |عائد| متتاليين) ومكوّن قفزي (الباقي) — **ليست في pandas_ta أصلاً** (تحقّق مباشر عبر `pandas_ta_classic.Category`). أُضيفت كميزة اختيارية جديدة (`custom_settings["jump_windows"]`)، مع اختبار ذاتي يحقن قفزة صناعية ضخمة ويتحقّق من استجابة `JUMP_RATIO` مباشرةً حسب تعريفها الإحصائي. اختُبِرت مباشرة على مقياس H003 الكامل (50 أصلاً، 30 نافذة).

### النتيجة الفعلية (1 مرشّح × 2 نافذة × 3 أهداف = 6 محاولات، عبر المحور الكامل مباشرة)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `JUMP_RATIO_14` | close | +0.009 | 23% | ❌ False |
| `JUMP_RATIO_14` | high | +0.028 | 30% | ❌ False |
| `JUMP_RATIO_14` | low | -0.010 | 23% | ❌ False |
| `JUMP_RATIO_30` | close | -0.008 | 17% | ❌ False |
| `JUMP_RATIO_30` | high | +0.014 | 27% | ❌ False |
| `JUMP_RATIO_30` | low | -0.028 | 30% | ❌ False |

**6 من 6 مرفوضة** — مثل أسّ هيرست/Variance Ratio، رفض حاسم (`mean_ic` ≤0.028 مطلقاً، `frac_significant` ≤30%). **ثالث دليل مستقلّ** (بعد `Fractal_reversal_w5` وHurst/VR) على أن أثر H003 خاصّ بحجم التقلّب المُطبَّع تحديداً (صيغة `NATR_14` نفسها) لا ببنيته أو اتجاهه — راجع [التوثيق الكامل](../docs/research/jump_ratio_feature.md).

### تسجيل نسبة التقلّب القفزي في سجلّ التجارب

In [ ]:
# @title
NEW_JUMP_RATIO_RESULTS = [
    ("JUMP_RATIO_14", "close", 0.009390362845300577, 0.06923834720121436, 0.23333333333333334, False, 30),
    ("JUMP_RATIO_14", "high", 0.028278573486405834, 0.09083910055997604, 0.3, False, 30),
    ("JUMP_RATIO_14", "low", -0.010407493738068489, 0.05231025577908628, 0.23333333333333334, False, 30),
    ("JUMP_RATIO_30", "close", -0.007907590695624099, 0.06221791369358426, 0.16666666666666666, False, 30),
    ("JUMP_RATIO_30", "high", 0.013781079651990475, 0.08983024165405377, 0.26666666666666666, False, 30),
    ("JUMP_RATIO_30", "low", -0.027831819905740206, 0.07635892094155527, 0.3, False, 30)
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in NEW_JUMP_RATIO_RESULTS:
    register_hypothesis(
        hyp_id=f"NEW_JUMP_RATIO_{name}_{target}",
        hypothesis=(
            f"{name} على هدف {target} — ثالث بحث ميزات خارج pandas_ta (نسبة "
            "التقلّب القفزي، Bipower Variation، بارندورف-نيلسن وشيفارد 2004، "
            "يختبر بنية التقلّب (قفزي/مستمرّ) بدل حجمه أو اتجاهه). اختُبِر "
            "مباشرة على 50 أصلاً/30 نافذة (مقياس H003 الكامل). راجع "
            "signal_discovery_lab.ipynb القسم ٢٤ للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "مُسجَّلة من تشغيل حقيقي فعلي (50 أصلاً، 30 نافذة — مقياس H003 الكامل "
            "مباشرة). consistent_sign=False وmean_ic ضعيف جداً. ثالث دليل مستقلّ "
            "(بعد Fractal_reversal_w5 وHurst/VR) على أن أثر H003 خاصّ بصيغة "
            "NATR_14 تحديداً لا بمعناها الأعمّ (حجم/اتجاه/بنية التقلّب)."
        ),
    )

print(f'✅ سُجِّلت {len(NEW_JUMP_RATIO_RESULTS)} توليفة — كلّها مرفوضة.')

## ٢٦) الاستخدام "الصحيح" لمؤشّرات معروفة (RSI/MACD/ADX/Bollinger/SuperTrend) — مرفوضة، بملاحظات لافتة

منهجية مختلفة عن الأقسام ٢٠-٢٥: بدل ميزات **جديدة كلياً**، بحث في **كيف تُستخدَم مؤشّرات معروفة بصيغتها الصحيحة** حسب مصادرها الأصلية (كتاب وايلدر، موقع بولنجر، StockCharts، LuxAlgo) — لا قيمتها الخام فقط. RSI/MACD/ADX/DM±/BBANDS/Stochastic **موجودة أصلاً** في `feature_order` (يراها النموذج) لكن لم تُختبَر عبر IC صراحة من قبل (استُبعِدت من المسح كـ"مكرّرة")؛ SuperTrend غائب كلياً. لكل مؤشّر: بحث ويب عن استخدامه الصحيح، ثمّ اختبار **الصيغة الخام مقابل الصيغة الصحيحة** بصفّ منفصل، على مقياس H003 الكامل مباشرة. راجع [التوثيق الكامل بالمصادر](../docs/research/proper_indicator_signals.md).

### النتيجة الفعلية (15 مرشّحاً × 3 أهداف = 45 محاولة)

| المرشّح | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `RSI_14_raw` | close | -0.125 | 77% | ❌ False |
| `RSI_14_raw` | high | -0.024 | 40% | ❌ False |
| `RSI_14_raw` | low | -0.092 | 70% | ❌ False |
| `RSI_DIVERGENCE_14` | close | -0.075 | 53% | ❌ False |
| `RSI_DIVERGENCE_14` | high | -0.038 | 40% | ❌ False |
| `RSI_DIVERGENCE_14` | low | -0.075 | 57% | ❌ False |
| `RSI_DIVERGENCE_28` | close | -0.022 | 27% | ❌ False |
| `RSI_DIVERGENCE_28` | high | +0.001 | 23% | ❌ False |
| `RSI_DIVERGENCE_28` | low | -0.049 | 37% | ❌ False |
| `MACDh_raw` | close | -0.053 | 43% | ❌ False |
| `MACDh_raw` | high | +0.006 | 37% | ❌ False |
| `MACDh_raw` | low | -0.029 | 37% | ❌ False |
| `MACD_HIST_SIGN` | close | -0.037 | 37% | ❌ False |
| `MACD_HIST_SIGN` | high | +0.007 | 30% | ❌ False |
| `MACD_HIST_SIGN` | low | -0.013 | 37% | ❌ False |
| `ADX_14_raw` | close | +0.016 | 40% | ❌ False |
| `ADX_14_raw` | high | +0.038 | 40% | ❌ False |
| `ADX_14_raw` | low | -0.022 | 50% | ❌ False |
| `ADX_DI_SIGNAL` | close | -0.010 | 27% | ❌ False |
| `ADX_DI_SIGNAL` | high | -0.044 | 33% | ❌ False |
| `ADX_DI_SIGNAL` | low | -0.009 | 37% | ❌ False |
| `BB_PCTB_20` | close | -0.090 | 67% | ❌ False |
| `BB_PCTB_20` | high | -0.001 | 33% | ❌ False |
| `BB_PCTB_20` | low | -0.054 | 50% | ❌ False |
| `SUPERT_DIR_10` | close | -0.000 | 7% | ❌ False |
| `SUPERT_DIR_10` | high | +0.020 | 33% | ❌ False |
| `SUPERT_DIR_10` | low | -0.019 | 27% | ❌ False |
| `SUPERT_DIR_20` | close | +0.005 | 17% | ❌ False |
| `SUPERT_DIR_20` | high | +0.021 | 30% | ❌ False |
| `SUPERT_DIR_20` | low | -0.008 | 23% | ❌ False |
| `SUPERT_STRETCH_10` | close | -0.029 | 20% | ❌ False |
| `SUPERT_STRETCH_10` | high | +0.057 | 37% | ❌ False |
| `SUPERT_STRETCH_10` | low | -0.080 | 67% | ❌ False |
| `SUPERT_STRETCH_20` | close | -0.028 | 17% | ❌ False |
| `SUPERT_STRETCH_20` | high | +0.046 | 30% | ❌ False |
| `SUPERT_STRETCH_20` | low | -0.072 | 63% | ❌ False |
| `STOCHk_raw` | close | -0.057 | 40% | ❌ False |
| `STOCHk_raw` | high | -0.009 | 33% | ❌ False |
| `STOCHk_raw` | low | -0.020 | 43% | ❌ False |
| `STOCH_KD_SIGN` | close | +0.005 | 20% | ❌ False |
| `STOCH_KD_SIGN` | high | +0.022 | 30% | ❌ False |
| `STOCH_KD_SIGN` | low | +0.037 | 40% | ❌ False |
| `PSAR_DIR` | close | +0.001 | 13% | ❌ False |
| `PSAR_DIR` | high | +0.041 | 40% | ❌ False |
| `PSAR_DIR` | low | -0.021 | 33% | ❌ False |

**45 من 45 مرفوضة** (`consistent_sign=False` بلا استثناء) — لكن بملاحظتين لافتتين: (أ) `RSI_14` **الخام** أقوى بكثير من كل شيء آخر هنا (`frac_significant=77%` على close، ثاني أعلى رقم في هذا المشروع بعد `NATR_14`)، لكن الاتجاه يتقلّب بين النوافذ فيفشل `consistent_sign`؛ (ب) صيغة "التباعد الصحيحة" (`RSI_DIVERGENCE`) كانت **أضعف** من RSI الخام — عكس توصية وايلدر بأن التباعد أقوى ميزة، على الأرجح لأن التقريب المُستخدَم (فرق درجة معيارية) لا يلتقط جوهر التباعد الحرفي (مطابقة قمم/قيعان مؤكَّدة). التفاصيل والتفسير الكامل في [التوثيق](../docs/research/proper_indicator_signals.md#الخلاصة).

### تسجيل نتائج الاستخدام "الصحيح" للمؤشّرات في سجلّ التجارب

In [ ]:
# @title
PROPER_INDICATOR_RESULTS = [
    ("RSI_14_raw", "close", -0.1252702287228203, 0.10831721699326026, 0.7666666666666667, False, 30),
    ("RSI_14_raw", "high", -0.023506949933676063, 0.13771916637629564, 0.4, False, 30),
    ("RSI_14_raw", "low", -0.09198982861122183, 0.11424585042761445, 0.7, False, 30),
    ("RSI_DIVERGENCE_14", "close", -0.07466791596117357, 0.09227514327343445, 0.5333333333333333, False, 30),
    ("RSI_DIVERGENCE_14", "high", -0.03835156707900016, 0.109298407937157, 0.4, False, 30),
    ("RSI_DIVERGENCE_14", "low", -0.07548375690789458, 0.09007846708978093, 0.5666666666666667, False, 30),
    ("RSI_DIVERGENCE_28", "close", -0.022260353917127787, 0.08826942027247275, 0.26666666666666666, False, 30),
    ("RSI_DIVERGENCE_28", "high", 0.0010947645975180678, 0.11410873252166592, 0.23333333333333334, False, 30),
    ("RSI_DIVERGENCE_28", "low", -0.04915407537249937, 0.09689742819571028, 0.36666666666666664, False, 30),
    ("MACDh_raw", "close", -0.053191001926014184, 0.09639001160236932, 0.43333333333333335, False, 30),
    ("MACDh_raw", "high", 0.006256038569662831, 0.1543054522940101, 0.36666666666666664, False, 30),
    ("MACDh_raw", "low", -0.028840235872338117, 0.1226098318473646, 0.36666666666666664, False, 30),
    ("MACD_HIST_SIGN", "close", -0.03656303798545134, 0.0840685921393001, 0.36666666666666664, False, 30),
    ("MACD_HIST_SIGN", "high", 0.006677133575474372, 0.1343714525589514, 0.3, False, 30),
    ("MACD_HIST_SIGN", "low", -0.012927691334666409, 0.1170529116862466, 0.36666666666666664, False, 30),
    ("ADX_14_raw", "close", 0.01612472042279142, 0.09529267877189405, 0.4, False, 30),
    ("ADX_14_raw", "high", 0.03843521221257315, 0.07025149234649997, 0.4, False, 30),
    ("ADX_14_raw", "low", -0.021780630251513788, 0.1316925748546584, 0.5, False, 30),
    ("ADX_DI_SIGNAL", "close", -0.009985253921711209, 0.08945403420221124, 0.26666666666666666, False, 30),
    ("ADX_DI_SIGNAL", "high", -0.04350696337936255, 0.09122753527742891, 0.3333333333333333, False, 30),
    ("ADX_DI_SIGNAL", "low", -0.009481446233934331, 0.08978572743269066, 0.36666666666666664, False, 30),
    ("BB_PCTB_20", "close", -0.08963773699961726, 0.11038003233919175, 0.6666666666666666, False, 30),
    ("BB_PCTB_20", "high", -0.0007912246964129635, 0.12709150587782514, 0.3333333333333333, False, 30),
    ("BB_PCTB_20", "low", -0.05389897441514828, 0.13532002172992014, 0.5, False, 30),
    ("SUPERT_DIR_10", "close", -0.0003664596101689729, 0.05978507563373014, 0.06666666666666667, False, 30),
    ("SUPERT_DIR_10", "high", 0.019826634457939956, 0.1019495488571019, 0.3333333333333333, False, 30),
    ("SUPERT_DIR_10", "low", -0.01884108542433216, 0.06571152739677995, 0.26666666666666666, False, 30),
    ("SUPERT_DIR_20", "close", 0.005432620364509855, 0.051440509938379037, 0.16666666666666666, False, 30),
    ("SUPERT_DIR_20", "high", 0.020577865535552905, 0.09163040216518395, 0.3, False, 30),
    ("SUPERT_DIR_20", "low", -0.007502560308638853, 0.06899151561382134, 0.23333333333333334, False, 30),
    ("SUPERT_STRETCH_10", "close", -0.028593629588242037, 0.05885156822822878, 0.2, False, 30),
    ("SUPERT_STRETCH_10", "high", 0.056920553919204, 0.10459095022218079, 0.36666666666666664, False, 30),
    ("SUPERT_STRETCH_10", "low", -0.07993007730829836, 0.06520878805312552, 0.6666666666666666, False, 30),
    ("SUPERT_STRETCH_20", "close", -0.028187461321414312, 0.05837327135381944, 0.16666666666666666, False, 30),
    ("SUPERT_STRETCH_20", "high", 0.045601442767854765, 0.09252384051339205, 0.3, False, 30),
    ("SUPERT_STRETCH_20", "low", -0.07212535417778086, 0.07305689832746302, 0.6333333333333333, False, 30),
    ("STOCHk_raw", "close", -0.0570390675824473, 0.09860371961187063, 0.4, False, 30),
    ("STOCHk_raw", "high", -0.00876928680124369, 0.12772220245377397, 0.3333333333333333, False, 30),
    ("STOCHk_raw", "low", -0.020211011801141227, 0.11614588297423249, 0.43333333333333335, False, 30),
    ("STOCH_KD_SIGN", "close", 0.005089782226203886, 0.10330279473913323, 0.2, False, 30),
    ("STOCH_KD_SIGN", "high", 0.02172239604879418, 0.09757494433291396, 0.3, False, 30),
    ("STOCH_KD_SIGN", "low", 0.036727556721091424, 0.10178069616215157, 0.4, False, 30),
    ("PSAR_DIR", "close", 0.0008109602883167965, 0.06311121386312757, 0.13333333333333333, False, 30),
    ("PSAR_DIR", "high", 0.04145153252496155, 0.0667242541519961, 0.4, False, 30),
    ("PSAR_DIR", "low", -0.02107251252639962, 0.0886116959175765, 0.3333333333333333, False, 30)
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in PROPER_INDICATOR_RESULTS:
    register_hypothesis(
        hyp_id=f"PROPER_INDICATOR_{name}_{target}",
        hypothesis=(
            f"{name} على هدف {target} — اختبار الاستخدام الصحيح لمؤشّر معروف "
            "(RSI/MACD/ADX/Bollinger/SuperTrend) حسب مصادره الأصلية الموثَّقة "
            "(وايلدر/بولنجر/StockCharts/LuxAlgo)، لا قيمته الخام فقط. اختُبِر "
            "مباشرة على 50 أصلاً/30 نافذة (مقياس H003 الكامل). راجع "
            "signal_discovery_lab.ipynb القسم ٢٦ ووثيقة proper_indicator_signals.md "
            "للمنهجية والمصادر الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "مُسجَّلة من تشغيل حقيقي فعلي (50 أصلاً، 30 نافذة — مقياس H003 الكامل "
            "مباشرة). consistent_sign=False يمنع القبول رغم أن RSI_14 الخام "
            "وBB_PCTB_20 وSUPERT_STRETCH أظهرت frac_significant مرتفعاً نسبياً "
            "(63-77%) — إشارة قوية إحصائياً لكن بلا اتساق اتجاه عبر النوافذ."
        ),
    )

print(f'✅ سُجِّلت {len(PROPER_INDICATOR_RESULTS)} توليفة — كلّها مرفوضة.')

## ٢٨) متابعة السؤال المفتوح — هل تفاعل RSI×نظام الزخم يحلّ تناقض الاتجاه؟ نتيجة جزئية، تبقى مرفوضة

امتداد مباشر لتشخيص القسم ٢٦ (لماذا يتذبذب اتجاه `RSI_14` بين النوافذ؟). اختبار شرطي: تصنيف عيّنات كل نافذة إلى فئتين حسب `RET_24` النسبي *داخل النافذة نفسها* (أعلى 10% = نظام زخم قويّ)، ثمّ حساب IC(`RSI_14`, الهدف) منفصلاً لكل فئة (لا مرشّح واحد على كامل العيّنة). راجع [التوثيق الكامل](../docs/research/proper_indicator_signals.md#متابعة-السؤال-المفتوح-هل-تفاعل-rsiنظام-الزخم-يحلّ-التناقض-نتيجة-جزئية-تبقى-مرفوضة) للمنهجية والتفسير.

### النتيجة (3 أهداف × فئتان = 6 اختبارات IC مشروطة)

| الهدف | الفئة | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| close | زخم قويّ (p90+) | -0.053 | 17% | ❌ False |
| close | عادي | -0.117 | 77% | ❌ False |
| high | زخم قويّ (p90+) | +0.094 | 30% | ❌ False |
| high | عادي | -0.048 | 50% | ❌ False |
| low | زخم قويّ (p90+) | -0.109 | 37% | ❌ False |
| low | عادي | -0.064 | 50% | ❌ False |

**نتيجة مختلطة، لا حالة وسطى: تبقى مرفوضة** (لا `consistent_sign=True` في أي فئة). لكن على `high` الإشارة **تنعكس فعلاً** من سالبة (نظام عادي) إلى موجبة (نظام زخم قويّ) — الاتجاه المتوقَّع من الفرضية — لكن `frac_significant=30%` دون عتبة 34%. على `close`/`low` الإشارة تبقى سالبة لكن أضعف بوضوح في نظام الزخم القويّ.

### تسجيل اختبار تفاعل RSI×نظام الزخم في سجلّ التجارب

In [ ]:
# @title
RSI_REGIME_CONDITIONAL_RESULTS = [
    ("strong", "close", -0.053230674648334836, 0.15977295456961613, 0.16666666666666666, False, 30),
    ("normal", "close", -0.11677796090409172, 0.11414601264862582, 0.7666666666666667, False, 30),
    ("strong", "high", 0.0938205032541802, 0.1327008562127243, 0.3, False, 30),
    ("normal", "high", -0.04792503321802889, 0.13962130014641444, 0.5, False, 30),
    ("strong", "low", -0.10946548425205432, 0.19116378743511395, 0.36666666666666664, False, 30),
    ("normal", "low", -0.06367273184619927, 0.12542506744790352, 0.5, False, 30)
]

for regime, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in RSI_REGIME_CONDITIONAL_RESULTS:
    register_hypothesis(
        hyp_id=f"RSI_REGIME_CONDITIONAL_{regime}_{target}",
        hypothesis=(
            f"RSI_14 مشروطاً بنظام الزخم ({regime}) على هدف {target} — متابعة "
            "مباشرة لتشخيص لماذا يتذبذب اتجاه IC لـRSI_14 الخام بين النوافذ "
            "(القسم ٢٦). تصنيف نسبي (p90 لكل نافذة) حسب RET_24، لا مرشّح واحد "
            "على كامل العيّنة. راجع signal_discovery_lab.ipynb القسم ٢٨ "
            "ووثيقة proper_indicator_signals.md للمنهجية الكاملة."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=(
            "نتيجة مختلطة غير حاسمة: على high الإشارة تنعكس فعلاً بين النظامين "
            "(الاتجاه المتوقَّع) لكن frac_significant=30% دون العتبة 34%. على "
            "close/low الإشارة تضعف في نظام الزخم القويّ لكن لا تنعكس بالكامل. "
            "لا حالة وسطى: consistent_sign=False في كل الحالات يمنع القبول."
        ),
    )

print(f'✅ سُجِّلت {len(RSI_REGIME_CONDITIONAL_RESULTS)} توليفة مشروطة — كلّها مرفوضة (نتيجة جزئية).')

## ٢٩) اختبار ذاتي (بيانات تركيبية — بلا حاجة لـDrive)

يتحقّق من سلامة الوصلات (`clean_reg_target` يُزيل الأثر المصطنع فعلاً،
`scan_candidates` لا يتعطّل، الماسح يُرجع أعمدة اللوحة المتوقَّعة) على بيانات
عشوائية صغيرة — لا يثبت وجود إشارة حقيقية، فقط أن البنية تعمل.

In [ ]:
# @title
def run_discovery_lab_selftest():
    rng = np.random.default_rng(0)
    n_assets, n_per_asset, T, F = 3, 200, 8, len(FEATURE_ORDER) if 'FEATURE_ORDER' in globals() else 37
    feature_order = FEATURE_ORDER if 'FEATURE_ORDER' in globals() else [f"f{i}" for i in range(F)]
    n = n_assets * n_per_asset
    ts0 = pd.Timestamp("2022-01-01", tz="UTC")
    ts = pd.concat([pd.Series(pd.date_range(ts0, periods=n_per_asset, freq="1D"))
                    for _ in range(n_assets)], ignore_index=True)

    X = rng.normal(size=(n, T, F)).astype("float32")
    last_close = 100.0 * np.exp(rng.normal(scale=0.05, size=n).cumsum() / n_per_asset)
    body_idx = feature_order.index("BODY_ratio") if "BODY_ratio" in feature_order else 0
    # ✅ نزرع أثر مرجع "نفس النوع" عمداً: last_high يعتمد على BODY_ratio لا على
    #    حركة سعرية حقيقية — clean_reg_target يجب أن يُزيله، والهدف الخام (لو
    #    استُخدم بالخطأ) يجب أن يُظهره بوضوح.
    body_last = X[:, -1, body_idx]
    last_high = last_close * (1.0 + np.clip(-body_last, 0, None) * 0.05 + 1e-3)
    last_low = last_close * (1.0 - np.clip(body_last, 0, None) * 0.05 - 1e-3)
    future_high_max = last_close * (1.0 + rng.normal(scale=0.01, size=n))  # لا علاقة حقيقية بـbody_last
    future_low_min = last_close * (1.0 - np.abs(rng.normal(scale=0.01, size=n)))
    future_close = last_close * (1.0 + rng.normal(scale=0.01, size=n))

    y_high_reg_dirty = (future_high_max - last_high) / last_high  # مرجع "نفس النوع" (ملوَّث)
    y_low_reg_dirty = (future_low_min - last_low) / last_low
    y_close_reg = (future_close - last_close) / last_close

    last_candles = np.stack([last_high, last_low, last_close, ts.values.astype("int64"),
                             future_close, future_low_min, future_high_max], axis=1)
    flat = {"base_params": np.zeros((n, 2), "float32"), "last_candles": last_candles,
            "X_1D": X, "y": {"y_high_reg": y_high_reg_dirty, "y_low_reg": y_low_reg_dirty,
                             "y_close_reg": y_close_reg}}

    # ١) الحارس يُزيل الأثر المزروع فعلاً
    clean_high = clean_reg_target(flat, "high")
    from scipy.stats import spearmanr
    rho_dirty, _ = spearmanr(body_last, y_high_reg_dirty)
    rho_clean, _ = spearmanr(body_last, clean_high)
    assert abs(rho_dirty) > 0.3, f"الأثر المزروع ضعيف جداً للاختبار ({rho_dirty:.3f}) — أصلح البيانات التركيبية."
    assert abs(rho_clean) < abs(rho_dirty) / 3, (
        f"❌ clean_reg_target لم يُزل الأثر المزروع: dirty={rho_dirty:.3f} clean={rho_clean:.3f}")
    print(f"  ✅ clean_reg_target يُزيل أثر مرجع نفس النوع (dirty={rho_dirty:+.3f} → clean={rho_clean:+.3f})")

    # ٢) extract_feature_last_value / make_feature_predict_fn يعملان
    v = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    assert v.shape == (n,), "extract_feature_last_value: شكل خاطئ."
    predict_fn = make_feature_predict_fn(feature_order[0], transform=lambda x: -x,
                                         tf="1D", feature_order=feature_order)
    preds = predict_fn(flat, flat, flat)
    assert np.allclose(preds, -v), "make_feature_predict_fn: التحويل لم يُطبَّق بشكل صحيح."
    print("  ✅ extract_feature_last_value / make_feature_predict_fn تعملان بشكل صحيح")

    # ٢-ب) make_candidate_predict_fn: كل الأنواع الأربعة (feature/interaction/custom/trained)
    feat_b = feature_order[1] if len(feature_order) > 1 else feature_order[0]
    inter_cand = {"kind": "interaction", "feat_a": feature_order[0], "feat_b": feat_b, "op": "mul"}
    inter_fn = make_candidate_predict_fn(inter_cand, tf="1D", feature_order=feature_order)
    a = extract_feature_last_value(flat, feature_order[0], tf="1D", feature_order=feature_order)
    b = extract_feature_last_value(flat, feat_b, tf="1D", feature_order=feature_order)
    assert np.allclose(inter_fn(flat, flat, flat), a * b), "make_candidate_predict_fn: مسار التفاعل خاطئ."

    custom_cand = {"kind": "custom", "fn": lambda X_last, fo: X_last[:, 0] * 2.0}
    custom_fn = make_candidate_predict_fn(custom_cand, tf="1D", feature_order=feature_order)
    X_last_expected = extract_feature_matrix(flat, tf="1D", feature_order=feature_order)
    assert np.allclose(custom_fn(flat, flat, flat), X_last_expected[:, 0] * 2.0), (
        "make_candidate_predict_fn: مسار kind='custom' خاطئ.")

    trained_cand = {"kind": "trained", "builder": lambda feature_order: make_isolation_forest_predict_fn(
        [feature_order[0], feat_b], feature_order=feature_order, contamination=0.1)}
    trained_fn = make_candidate_predict_fn(trained_cand, feature_order=feature_order)
    trained_preds = trained_fn(flat, flat, flat)
    assert trained_preds.shape == (n,), "make_candidate_predict_fn: مسار kind='trained' أرجع شكلاً خاطئاً."

    series_cand = {"kind": "series", "feature": feature_order[0], "fn": lambda s: s[:, -1] * 3.0}
    series_fn = make_candidate_predict_fn(series_cand, tf="1D", feature_order=feature_order)
    series_full = extract_feature_series(flat, feature_order[0], tf="1D", feature_order=feature_order)
    assert series_full.shape == (n, T), "extract_feature_series: شكل خاطئ."
    assert np.allclose(series_fn(flat, flat, flat), series_full[:, -1] * 3.0), (
        "make_candidate_predict_fn: مسار kind='series' خاطئ.")
    print("  ✅ make_candidate_predict_fn يدعم الأنواع الخمسة (feature/interaction/custom/trained/series)")

    # ٣) scan_candidates يُرجع لوحة قيادة بالأعمدة المتوقَّعة، بلا انهيار
    windows_synth = [(flat, flat, flat)]
    tiny_candidates = [{"name": "f0", "track": "data_driven", "feature": feature_order[0], "transform": None}]
    board = scan_candidates(tiny_candidates, windows_synth, targets=("close", "high", "low"),
                            feature_order=feature_order, n_shuffles=20, min_samples=5)
    expected_cols = {"name", "track", "target", "status"}
    assert expected_cols.issubset(board.columns), f"أعمدة ناقصة في اللوحة: {board.columns.tolist()}"
    assert (board["status"] == "ok").all(), f"فشل تقييم بعض المرشّحين:\n{board}"
    print("  ✅ scan_candidates يُرجع لوحة قيادة سليمة بلا أخطاء")

    # ٤) classify_result: منطق حتمي بمعزل عن أي تدريب/عشوائية
    assert classify_result({"n_ok": 2, "consistent_sign": True, "frac_significant": 1.0}) == "قيد الاختبار", (
        "classify_result: n_ok قليل يجب أن يُرجع 'قيد الاختبار' بصرف النظر عن باقي الحقول")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.5}) == "مقبولة", (
        "classify_result: consistent_sign=True + frac_significant كافية يجب أن يُرجع 'مقبولة'")
    assert classify_result({"n_ok": 10, "consistent_sign": False, "frac_significant": 0.9}) == "مرفوضة", (
        "classify_result: consistent_sign=False يجب أن يُرجع 'مرفوضة' مهما كانت frac_significant")
    assert classify_result({"n_ok": 10, "consistent_sign": True, "frac_significant": 0.1}) == "مرفوضة", (
        "classify_result: frac_significant دون الحدّ الأدنى يجب أن يُرجع 'مرفوضة'")
    print("  ✅ classify_result يُطبِّق معيار القبول الموحّد بشكل صحيح (٤ حالات)")

    # ٥) run_batch_and_register: تسجيل فعلي إلى ملف مؤقّت (لا experiment_registry الحقيقي إطلاقاً)
    import tempfile, json as _json
    from pathlib import Path
    tmp_registry = Path(tempfile.mkdtemp()) / "registry_selftest.json"
    batch_board, registered_ids = run_batch_and_register(
        tiny_candidates, windows_synth, targets=("close", "high", "low"), feature_order=feature_order,
        id_prefix="SELFTEST", max_workers=2, registry_path=tmp_registry, n_shuffles=20, min_samples=5)
    assert len(registered_ids) == 3, f"يُتوقَّع تسجيل 3 (مرشّح واحد × 3 أهداف)، وُجد {len(registered_ids)}"
    assert tmp_registry.exists(), "run_batch_and_register: لم يُكتَب ملف السجلّ المؤقّت إطلاقاً"
    entries = _json.loads(tmp_registry.read_text(encoding="utf-8"))
    assert {e["id"] for e in entries} == set(registered_ids), "معرّفات السجلّ المكتوبة لا تطابق registered_ids"
    assert all(e["status"] in REGISTRY_STATUSES for e in entries), "حالة غير صالحة في سجلّ مكتوب فعلياً"
    assert all(e["id"].startswith("SELFTEST_") for e in entries), "id_prefix لم يُطبَّق على المعرّفات"
    print(f"  ✅ run_batch_and_register يقيّم بالتوازي (imap_ordered/default_workers) ويسجّل فعلياً "
          f"في ملف JSON مستقلّ ({len(registered_ids)} مدخلات، registry_path مُخصَّص لا الحقيقي)")

    print("✅ نجحت كل اختبارات مختبر بحث الإشارات الذاتية.")


run_discovery_lab_selftest()

## ٣٠) الجولة الأولى من "الفرضيات الأعمق" — 8 عائلات، 51 توليفة، صفر اكتشافات جديدة

بعد استنفاد pandas_ta والاستخدام "الصحيح" لمؤشّرات معروفة، طلب صاحب المشروع صراحةً فرضيات مبنية على معرفة اقتصادية-إحصائية حقيقية لا مسحاً آلياً. راجع [التوثيق الكامل](../docs/research/deeper_hypotheses_round1.md) للتعريفات الرياضية والمنهجية الكاملة لكل فرضية. الملخّص هنا للتسجيل في سجلّ التجارب فقط.

### النتيجة الفعلية (50 أصلاً، 30 نافذة، لكل التوليفات)

| العائلة | أفضل توليفة | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| اتفاق الزخم متعدّد الآفاق | ZSCORE/close | -0.103 | 67% | ❌ False |
| تفاعلات مُمركَزة | NATR×RSI/high,low | -0.196 / +0.162 | 93% / 97% | ✅ True (لكن إعادة اكتشاف H003) |
| ترتيب الزخم المقطعي | MOM_RANK_24/high | +0.035 | 33% | ❌ False |
| ارتباط BTC (مباشر) | MKT_CORR_20/low | +0.046 | 47% | ❌ False |
| ارتباط BTC (مشروط RSI) | ارتباط منخفض/close | -0.126 | 77% | ❌ False |
| عمر الاتجاه | BARS_SINCE_LOW_5/close | +0.043 | 50% | ❌ False |
| اتساق الاتجاه اليومي | DIR_CONSISTENCY_30/high | +0.040 | 37% | ❌ False |
| عدم تناظر التقلّب | VOL_ASYMMETRY_20/close | +0.045 | 37% | ❌ False |

**⚠️ ملاحظة مهمّة**: `NATR14_x_RSI14_CENTERED` يعبر معيار القبول رياضياً، لكن فحصاً فورياً ضد `h003_volatility_reversal.md` أظهر أنه إعادة صياغة رياضية شبه مطابقة لمكوّن `RSI_vol_adjusted_reversion` **المسجَّل أصلاً** ضمن H003 منذ بداية المشروع — لا يُحتسَب اكتشافاً جديداً.

### تسجيل الجولة الأولى من الفرضيات الأعمق في سجلّ التجارب

In [ ]:
# @title
DEEPER_HYP_ROUND1_RESULTS = [
    # (hyp_id, hypothesis_short, target, mean_ic, std_ic, frac_significant, consistent_sign, n_ok, status, notes)
    ("MOMENTUM_AGREEMENT_SIGN", "اتفاق إشارة الزخم عبر آفاق [3,6,12,24]", "close", -0.079134, 0.115153, 0.600000, False, 30,
     "مرفوضة", "أقوى نتيجة في العائلة، لكن consistent_sign=False."),
    ("MOMENTUM_AGREEMENT_SIGN", "اتفاق إشارة الزخم عبر آفاق [3,6,12,24]", "high", 0.007735, 0.117638, 0.233333, False, 30,
     "مرفوضة", None),
    ("MOMENTUM_AGREEMENT_SIGN", "اتفاق إشارة الزخم عبر آفاق [3,6,12,24]", "low", -0.047666, 0.118046, 0.466667, False, 30,
     "مرفوضة", None),
    ("MOMENTUM_AGREEMENT_ZSCORE", "اتفاق z-score الزخم عبر آفاق [3,6,12,24]", "close", -0.103407, 0.121838, 0.666667, False, 30,
     "مرفوضة", "أقوى توليفة في العائلة بأكملها (67%)، لكن consistent_sign=False."),
    ("MOMENTUM_AGREEMENT_ZSCORE", "اتفاق z-score الزخم عبر آفاق [3,6,12,24]", "high", -0.006923, 0.144342, 0.300000, False, 30,
     "مرفوضة", None),
    ("MOMENTUM_AGREEMENT_ZSCORE", "اتفاق z-score الزخم عبر آفاق [3,6,12,24]", "low", -0.057944, 0.133084, 0.533333, False, 30,
     "مرفوضة", None),
    ("NATR14_x_RSI14_CENTERED", "NATR_14 * (RSI_14-50) مُمركَز", "close", -0.041981, 0.081367, 0.500000, False, 30,
     "مرفوضة", None),
    ("NATR14_x_RSI14_CENTERED", "NATR_14 * (RSI_14-50) مُمركَز", "high", -0.195989, 0.097556, 0.933333, True, 30,
     "مرفوضة", "⚠️ يعبر المعيار رياضياً (consistent_sign=True) لكنه إعادة اكتشاف: شبه مطابق لـRSI_vol_adjusted_reversion المسجَّلة أصلاً ضمن H003 (-0.196 مقابل -0.198 موثَّقة) — لا يُحتسَب اكتشافاً جديداً، فيُصنَّف مرفوضاً هنا لعدم الجدّة."),
    ("NATR14_x_RSI14_CENTERED", "NATR_14 * (RSI_14-50) مُمركَز", "low", 0.162108, 0.068853, 0.966667, True, 30,
     "مرفوضة", "⚠️ نفس الملاحظة أعلاه (إعادة اكتشاف H003)، إشارة معاكسة على low كما هو متوقَّع من التماثل مع H003."),
    ("NATR14_x_BBPCTB20_CENTERED", "NATR_14 * (BBP_20_2.0-0.5) مُمركَز", "close", -0.099562, 0.110637, 0.733333, False, 30,
     "مرفوضة", None),
    ("NATR14_x_BBPCTB20_CENTERED", "NATR_14 * (BBP_20_2.0-0.5) مُمركَز", "high", -0.046374, 0.153643, 0.466667, False, 30,
     "مرفوضة", None),
    ("NATR14_x_BBPCTB20_CENTERED", "NATR_14 * (BBP_20_2.0-0.5) مُمركَز", "low", -0.026111, 0.141237, 0.433333, False, 30,
     "مرفوضة", "التفاعل الجديد الفعلي (غير مغطّى بـH003) — مرفوض بوضوح."),
    ("MOM_RANK_6", "الرتبة المئوية المقطعية للزخم (h=6)", "close", -0.006311, 0.026805, 0.066667, False, 30,
     "مرفوضة", "على الهدف المطلق الحالي. قد يستحق إعادة اختبار على هدف نسبي/سوقي-محايد مستقبلاً."),
    ("MOM_RANK_6", "الرتبة المئوية المقطعية للزخم (h=6)", "high", 0.033653, 0.054759, 0.333333, False, 30,
     "مرفوضة", None),
    ("MOM_RANK_6", "الرتبة المئوية المقطعية للزخم (h=6)", "low", -0.045158, 0.038399, 0.366667, False, 30,
     "مرفوضة", None),
    ("MOM_RANK_24", "الرتبة المئوية المقطعية للزخم (h=24)", "close", -0.007584, 0.025294, 0.033333, False, 30,
     "مرفوضة", None),
    ("MOM_RANK_24", "الرتبة المئوية المقطعية للزخم (h=24)", "high", 0.034768, 0.057424, 0.333333, False, 30,
     "مرفوضة", None),
    ("MOM_RANK_24", "الرتبة المئوية المقطعية للزخم (h=24)", "low", -0.048756, 0.038020, 0.333333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_10", "ارتباط بيرسون المتحرّك مع BTC (w=10)", "close", 0.015962, 0.097819, 0.366667, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_10", "ارتباط بيرسون المتحرّك مع BTC (w=10)", "high", -0.012696, 0.099568, 0.333333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_10", "ارتباط بيرسون المتحرّك مع BTC (w=10)", "low", 0.035018, 0.106730, 0.433333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_20", "ارتباط بيرسون المتحرّك مع BTC (w=20)", "close", 0.036327, 0.094447, 0.400000, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_20", "ارتباط بيرسون المتحرّك مع BTC (w=20)", "high", -0.004590, 0.110298, 0.266667, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_20", "ارتباط بيرسون المتحرّك مع BTC (w=20)", "low", 0.046331, 0.103120, 0.466667, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_30", "ارتباط بيرسون المتحرّك مع BTC (w=30)", "close", 0.023447, 0.080883, 0.300000, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_30", "ارتباط بيرسون المتحرّك مع BTC (w=30)", "high", -0.008715, 0.081016, 0.200000, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_30", "ارتباط بيرسون المتحرّك مع BTC (w=30)", "low", 0.029019, 0.108985, 0.333333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_REGIME_RSI_high", "RSI_14 مشروطاً بنظام ارتباط BTC عالٍ (MKT_CORR_20 p50+)", "close", -0.151050, 0.145949, 0.333333, False, 30,
     "مرفوضة", "لا انعكاس إشارة واضح بين نظامي الارتباط (بعكس تجربة RSI×نظام الزخم في القسم ٢٨)."),
    ("MKT_CORR_REGIME_RSI_low", "RSI_14 مشروطاً بنظام ارتباط BTC منخفض", "close", -0.126297, 0.111659, 0.766667, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_REGIME_RSI_high", "RSI_14 مشروطاً بنظام ارتباط BTC عالٍ", "high", -0.092668, 0.176327, 0.200000, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_REGIME_RSI_low", "RSI_14 مشروطاً بنظام ارتباط BTC منخفض", "high", -0.020270, 0.141402, 0.433333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_REGIME_RSI_high", "RSI_14 مشروطاً بنظام ارتباط BTC عالٍ", "low", -0.085875, 0.186867, 0.333333, False, 30,
     "مرفوضة", None),
    ("MKT_CORR_REGIME_RSI_low", "RSI_14 مشروطاً بنظام ارتباط BTC منخفض", "low", -0.097579, 0.111840, 0.633333, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_5", "عدد الشموع منذ آخر قمّة فركتالية (w=5)", "close", 0.005001, 0.072803, 0.300000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_5", "عدد الشموع منذ آخر قمّة فركتالية (w=5)", "high", 0.017250, 0.054763, 0.200000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_5", "عدد الشموع منذ آخر قمّة فركتالية (w=5)", "low", -0.015387, 0.081442, 0.333333, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_LOW_5", "عدد الشموع منذ آخر قاع فركتالي (w=5)", "close", 0.043099, 0.080488, 0.500000, False, 30,
     "مرفوضة", "أفضل توليفة في العائلة (50%)، لكن consistent_sign=False."),
    ("BARS_SINCE_LOW_5", "عدد الشموع منذ آخر قاع فركتالي (w=5)", "high", 0.053661, 0.088208, 0.466667, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_LOW_5", "عدد الشموع منذ آخر قاع فركتالي (w=5)", "low", -0.027616, 0.087602, 0.300000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_10", "عدد الشموع منذ آخر قمّة فركتالية (w=10)", "close", 0.017558, 0.097589, 0.233333, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_10", "عدد الشموع منذ آخر قمّة فركتالية (w=10)", "high", 0.028184, 0.083474, 0.300000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_HIGH_10", "عدد الشموع منذ آخر قمّة فركتالية (w=10)", "low", 0.012411, 0.083717, 0.300000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_LOW_10", "عدد الشموع منذ آخر قاع فركتالي (w=10)", "close", -0.000342, 0.087880, 0.200000, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_LOW_10", "عدد الشموع منذ آخر قاع فركتالي (w=10)", "high", 0.018450, 0.084195, 0.233333, False, 30,
     "مرفوضة", None),
    ("BARS_SINCE_LOW_10", "عدد الشموع منذ آخر قاع فركتالي (w=10)", "low", -0.021016, 0.100008, 0.300000, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_10", "اتساق اتجاه العائد اليومي (w=10)", "close", 0.012906, 0.089035, 0.233333, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_10", "اتساق اتجاه العائد اليومي (w=10)", "high", 0.026971, 0.080476, 0.333333, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_10", "اتساق اتجاه العائد اليومي (w=10)", "low", -0.019509, 0.094530, 0.266667, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_20", "اتساق اتجاه العائد اليومي (w=20)", "close", -0.003106, 0.079981, 0.200000, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_20", "اتساق اتجاه العائد اليومي (w=20)", "high", 0.025544, 0.077401, 0.266667, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_20", "اتساق اتجاه العائد اليومي (w=20)", "low", -0.026748, 0.100582, 0.366667, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_30", "اتساق اتجاه العائد اليومي (w=30)", "close", 0.016931, 0.066973, 0.266667, False, 30,
     "مرفوضة", None),
    ("DIR_CONSISTENCY_30", "اتساق اتجاه العائد اليومي (w=30)", "high", 0.040445, 0.096966, 0.366667, False, 30,
     "مرفوضة", "أفضل توليفة في العائلة (37%)، لا تزال دون العتبة."),
    ("DIR_CONSISTENCY_30", "اتساق اتجاه العائد اليومي (w=30)", "low", -0.011507, 0.062292, 0.233333, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_10", "عدم تناظر شبه-التباين هابط/صاعد (w=10)", "close", 0.022539, 0.100401, 0.300000, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_10", "عدم تناظر شبه-التباين هابط/صاعد (w=10)", "high", 0.003570, 0.121971, 0.300000, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_10", "عدم تناظر شبه-التباين هابط/صاعد (w=10)", "low", -0.013288, 0.122079, 0.333333, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_20", "عدم تناظر شبه-التباين هابط/صاعد (w=20)", "close", 0.045164, 0.097268, 0.366667, False, 30,
     "مرفوضة", "أفضل توليفة في العائلة (37%)."),
    ("VOL_ASYMMETRY_20", "عدم تناظر شبه-التباين هابط/صاعد (w=20)", "high", 0.018644, 0.138766, 0.333333, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_20", "عدم تناظر شبه-التباين هابط/صاعد (w=20)", "low", 0.025500, 0.100251, 0.333333, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_30", "عدم تناظر شبه-التباين هابط/صاعد (w=30)", "close", 0.025739, 0.108495, 0.400000, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_30", "عدم تناظر شبه-التباين هابط/صاعد (w=30)", "high", -0.027256, 0.122826, 0.366667, False, 30,
     "مرفوضة", None),
    ("VOL_ASYMMETRY_30", "عدم تناظر شبه-التباين هابط/صاعد (w=30)", "low", 0.033243, 0.128252, 0.333333, False, 30,
     "مرفوضة", None),
]

for hyp_id, hyp_short, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok, status, notes in DEEPER_HYP_ROUND1_RESULTS:
    register_hypothesis(
        hyp_id=f"{hyp_id}_{target}",
        hypothesis=(
            f"{hyp_short} — هدف {target}. جزء من الجولة الأولى لـ\"الفرضيات "
            "الأعمق\" (8 عائلات، بعد استنفاد pandas_ta والمؤشرات المعروفة). "
            "راجع docs/research/deeper_hypotheses_round1.md للمنهجية والتعريف "
            "الرياضي الكامل."
        ),
        source="literature_mining",
        status=status,
        report={
            "mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
            "consistent_sign": consistent_sign, "n_ok": n_ok,
        },
        notes=notes,
    )

n_rediscovery = sum(1 for r in DEEPER_HYP_ROUND1_RESULTS if r[9] and 'إعادة اكتشاف' in r[9])
print(f'✅ سُجِّلت {len(DEEPER_HYP_ROUND1_RESULTS)} توليفة من الجولة الأولى للفرضيات الأعمق '
      f'(كلّها مرفوضة رسمياً، منها {n_rediscovery} إعادة اكتشاف لـH003 مذكورة في الملاحظات).')


## ٣١) الفرضية التاسعة — الانتشار المتأخّر لعائد BTC (Lead-Lag Spillover)

امتداد لـ`market_context`: هل عائد BTC قبل يوم أو أكثر (لا اليوم نفسه) يحمل معلومة عن أصل آخر؟ راجع [التوثيق الكامل](../docs/research/deeper_hypotheses_round1.md#٨-الانتشار-المتأخّر-لعائد-btc-lead-lag-spillover).

### النتيجة الفعلية (50 أصلاً، 30 نافذة، `MKT_LAG_RET_{1,2,3,5}`)

أفضل توليفة `MKT_LAG_RET_3`/low: `frac_significant=50%` لكن `consistent_sign=False` — **مرفوضة بوضوح** في كل التوليفات الـ12.

### تسجيل الفرضية التاسعة في سجلّ التجارب

In [ ]:
# @title
MKT_LAG_RET_RESULTS = [
    ("MKT_LAG_RET_1", "close", -0.003008, 0.134962, 0.233333, False, 30),
    ("MKT_LAG_RET_1", "high", 0.013010, 0.128295, 0.466667, False, 30),
    ("MKT_LAG_RET_1", "low", 0.032764, 0.187374, 0.433333, False, 30),
    ("MKT_LAG_RET_2", "close", 0.027235, 0.140879, 0.466667, False, 30),
    ("MKT_LAG_RET_2", "high", 0.025265, 0.112910, 0.433333, False, 30),
    ("MKT_LAG_RET_2", "low", 0.053127, 0.141924, 0.466667, False, 30),
    ("MKT_LAG_RET_3", "close", -0.005353, 0.130141, 0.266667, False, 30),
    ("MKT_LAG_RET_3", "high", 0.009102, 0.102572, 0.433333, False, 30),
    ("MKT_LAG_RET_3", "low", 0.028811, 0.142379, 0.500000, False, 30),
    ("MKT_LAG_RET_5", "close", 0.003938, 0.166624, 0.366667, False, 30),
    ("MKT_LAG_RET_5", "high", 0.008980, 0.137150, 0.333333, False, 30),
    ("MKT_LAG_RET_5", "low", 0.027871, 0.162758, 0.466667, False, 30),
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in MKT_LAG_RET_RESULTS:
    register_hypothesis(
        hyp_id=f"{name}_{target}",
        hypothesis=(
            f"{name} (عائد BTC المتأخّر) على هدف {target} — الفرضية التاسعة "
            "من الجولة الأولى للفرضيات الأعمق: هل عائد BTC قبل k يوماً "
            "(لا المعاصر) يحمل معلومة عن أصل آخر؟ راجع "
            "docs/research/deeper_hypotheses_round1.md القسم ٨."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={"mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
                "consistent_sign": consistent_sign, "n_ok": n_ok},
        notes=None,
    )

print(f'✅ سُجِّلت {len(MKT_LAG_RET_RESULTS)} توليفة — الفرضية التاسعة (الانتشار المتأخّر) مرفوضة بالكامل.')


## ٣٢) نسبة كفاءة كوفمان الداخل-يومية (`EFF_RATIO_24H`) — أول ميزة من بيانات ساعية حقيقية

أول فرضية في المشروع تعتمد على مسار السعر **داخل** اليوم (بيانات ساعية حقيقية لأول مرّة، لا مستويات OHLC اليومية فقط). راجع [التوثيق الكامل](../docs/research/kaufman_efficiency_ratio.md) للمنهجية وطريقة الحساب.

### النتيجة الفعلية (50 أصلاً، 30 نافذة)

| الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|
| close | +0.012 | 23% | ❌ False |
| high | +0.039 | 37% | ❌ False |
| low | -0.034 | 40% | ❌ False |

**مرفوضة بوضوح** — نظافة الاتجاه الداخل-يومي في يوم واحد لا تتنبأ بحركة الأصل لاحقاً على هذا الأفق. أوّل نتيجة سلبية من فئة ميزات جديدة كلياً (بيانات ساعية)، تفتح الباب لاختبار ميزات داخل-يومية أخرى بنفس البنية.

### تسجيل نسبة كفاءة كوفمان في سجلّ التجارب

In [ ]:
# @title
EFF_RATIO_RESULTS = [
    ("close", 0.011961, 0.082404, 0.233333, False, 30),
    ("high", 0.038638, 0.069461, 0.366667, False, 30),
    ("low", -0.033550, 0.074402, 0.400000, False, 30),
]

for target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in EFF_RATIO_RESULTS:
    register_hypothesis(
        hyp_id=f"EFF_RATIO_24H_{target}",
        hypothesis=(
            f"نسبة كفاءة كوفمان (EFF_RATIO_24H، نافذة 24 ساعة متدحرجة في "
            f"مساحة اللوغاريتم، من بيانات ساعية حقيقية) على هدف {target} — "
            "أول ميزة في المشروع من مصدر بيانات داخل-يومي حقيقي. راجع "
            "docs/research/kaufman_efficiency_ratio.md."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={"mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
                "consistent_sign": consistent_sign, "n_ok": n_ok},
        notes="أوّل اختبار على بيانات ساعية حقيقية (50 أصلاً، أرشيف خارجي عبر intraday_efficiency) — مرفوضة، لكن تفتح بنية معمارية جاهزة لميزات داخل-يومية أخرى.",
    )

print(f'✅ سُجِّلت {len(EFF_RATIO_RESULTS)} توليفة — نسبة كفاءة كوفمان مرفوضة (أول ميزة من بيانات ساعية).')


## ٣٣) انحراف VWAP وتركّز الحجم الساعي — ميزتان إضافيتان من بيانات ساعية حقيقية

امتداد لـ[نسبة كفاءة كوفمان](../docs/research/kaufman_efficiency_ratio.md) عبر نفس بنية الأرشيف الخارجي. راجع [التوثيق الكامل](../docs/research/vwap_deviation_and_volume_concentration.md).

### النتيجة الفعلية (50 أصلاً، 30 نافذة)

| الميزة | الهدف | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| `VWAP_DEVIATION` | close | +0.032 | 30% | ❌ False |
| `VWAP_DEVIATION` | **high** | **+0.081** | **57%** | ❌ False |
| `VWAP_DEVIATION` | low | -0.036 | 47% | ❌ False |
| `VOL_CONC_HHI` | close | -0.005 | 30% | ❌ False |
| `VOL_CONC_HHI` | high | +0.025 | 33% | ❌ False |
| `VOL_CONC_HHI` | low | -0.021 | 30% | ❌ False |

**كلتاهما مرفوضة** — لكن `VWAP_DEVIATION`/high أقوى نتيجة بين كل الميزات الداخل-يومية حتى الآن (`frac_significant=57%`)، تستحقّ تشخيصاً إضافياً لاحقاً (لماذا `consistent_sign=False` رغم الاتساق الإحصائي العالي).

### تسجيل الميزتين في سجلّ التجارب

In [ ]:
# @title
VWAP_VOLCONC_RESULTS = [
    ("VWAP_DEVIATION", "close", 0.031728, 0.094377, 0.300000, False, 30,
     "أول اختبار على انحراف VWAP الساعي — مرفوضة."),
    ("VWAP_DEVIATION", "high", 0.081121, 0.085159, 0.566667, False, 30,
     "أقوى نتيجة بين كل الميزات الداخل-يومية حتى الآن (mean_ic=+0.081، frac_significant=57%) — لكن consistent_sign=False يمنع القبول. يستحقّ تشخيص نوافذ مخالِفة لاحقاً (لم يُنفَّذ بعد)."),
    ("VWAP_DEVIATION", "low", -0.035553, 0.086723, 0.466667, False, 30, None),
    ("VOL_CONC_HHI", "close", -0.005397, 0.101048, 0.300000, False, 30,
     "أول اختبار على تركّز الحجم الساعي (HHI) — مرفوضة بوضوح."),
    ("VOL_CONC_HHI", "high", 0.024973, 0.082328, 0.333333, False, 30, None),
    ("VOL_CONC_HHI", "low", -0.021210, 0.083475, 0.300000, False, 30, None),
]

for name, target, mean_ic, std_ic, frac_sig, consistent_sign, n_ok, notes in VWAP_VOLCONC_RESULTS:
    register_hypothesis(
        hyp_id=f"{name}_{target}",
        hypothesis=(
            f"{name} (ميزة من بيانات ساعية حقيقية، بنية أرشيف خارجي "
            "intraday_*) على هدف {target}. راجع "
            "docs/research/vwap_deviation_and_volume_concentration.md."
        ).format(target=target),
        source="literature_mining",
        status="مرفوضة",
        report={"mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
                "consistent_sign": consistent_sign, "n_ok": n_ok},
        notes=notes,
    )

print(f'✅ سُجِّلت {len(VWAP_VOLCONC_RESULTS)} توليفة — VWAP_DEVIATION وVOL_CONC_HHI مرفوضتان (ثاني وثالث ميزة من بيانات ساعية).')


## ٣٤) متابعة تشخيصية — نفس نافذة انهيار FTX تكسر إشارة RSI وVWAP_DEVIATION معاً

فحص كل نافذة من الثلاثين على حدة لـ`VWAP_DEVIATION`/high (بنفس أسلوب تشخيص RSI في القسم ٢٦) يكشف 5 نوافذ سالبة الاتجاه من أصل 30 — أوضحها يقع في فترات ارتداد حادّ بعد انهيار (Feb 2021 صعود متفجّر، أغسطس 2021 ما بعد انهيار مايو، يوليو 2022 ما بعد Terra/Celsius، ويناير 2023 ما بعد FTX). **النافذة الأخيرة (٢٠٢٢-١١-١٩ إلى ٢٠٢٣-٠١-٢١) هي نفسها بالضبط** التي كسرت اتّساق RSI_14 في القسم ٢٦ — تأكيد مستقلّ عبر ميزتين مختلفتي المصدر كلياً (RSI يومي، VWAP ساعي) أن هذه الفترة تُشكّل نظام سوق حقيقياً. راجع [التوثيق الكامل](../docs/research/vwap_deviation_and_volume_concentration.md#متابعة-تشخيصية-لماذا-ينعكس-اتجاه-vwap_deviationhigh-بين-النوافذ). لا يُغيّر قرار الرفض (لا حالة وسطى)، لكنه يرفع أولوية اختبار فرضية "نظام ارتداد ما بعد الانهيار" (تعريف مباشر: هبوط >30% خلال 30 يوماً ثمّ ارتداد) من متوسطة إلى أعلى.

## ٣٥) متابعة ثانية — نظام "ارتداد ما بعد الانهيار" بتعريف مباشر (على مستوى الأصل) — نتيجة مختلطة

اختبار مباشر لسؤال القسم ٣٤: `CRASH_REBOUND_REGIME` (هبوط ≥30% عن قمّة 60 يوماً خلال آخر 30 يوماً، لكلّ أصل على حدة) كتصفية لـ`RSI_14`. راجع [التوثيق الكامل](../docs/research/vwap_deviation_and_volume_concentration.md#متابعة-ثانية-هل-تعريف-مباشر-لنظام-ارتداد-ما-بعد-الانهيار-يحلّ-المشكلة-نتيجة-مختلطة-تبقى-مرفوضة) (يتضمّن أيضاً درساً منهجياً: الأقنعة الثنائية المبنية على ميزات رسمية تحتاج القيمة الخام لا `extract_feature_last_value` المُطبَّعة).

### النتيجة الفعلية (50 أصلاً، 30 نافذة، IC(RSI_14) مشروطاً)

| الهدف | الفئة | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|
| close | ارتداد ما بعد انهيار | -0.145 | 77% | ❌ False |
| close | عادي | -0.089 | 42% | ❌ False |
| high | ارتداد ما بعد انهيار | -0.037 | 37% | ❌ False |
| high | عادي | +0.075 | 23% | ❌ False |
| low | ارتداد ما بعد انهيار | -0.113 | 60% | ❌ False |
| low | عادي | -0.132 | 58% | ❌ False |

**نتيجة مختلطة، تبقى مرفوضة.** الانتشار الواسع للنظام على مستوى الأصل الفردي (73% من العيّنات) يجعله "الحالة الطبيعية" لأغلب العملات المتقلّبة لا نظاماً نادراً مميِّزاً — يدعم الانتقال لتعريف على مستوى السوق ككل بدل الأصل الفردي في أي محاولة مستقبلية.

### تسجيل نظام ارتداد ما بعد الانهيار في سجلّ التجارب

In [ ]:
# @title
CRASH_REBOUND_REGIME_RESULTS = [
    ("close", "crash_rebound", -0.144537, 0.120180, 0.766667, False, 30),
    ("close", "normal", -0.089055, 0.127301, 0.423077, False, 26),
    ("high", "crash_rebound", -0.037292, 0.142697, 0.366667, False, 30),
    ("high", "normal", 0.075228, 0.141015, 0.230769, False, 26),
    ("low", "crash_rebound", -0.113492, 0.124369, 0.600000, False, 30),
    ("low", "normal", -0.131796, 0.121341, 0.576923, False, 26),
]

for target, regime, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in CRASH_REBOUND_REGIME_RESULTS:
    register_hypothesis(
        hyp_id=f"CRASH_REBOUND_REGIME_RSI14_{regime}_{target}",
        hypothesis=(
            f"RSI_14 مشروطاً بنظام ارتداد ما بعد الانهيار ({regime}، تعريف "
            f"مباشر: هبوط ≥30% عن قمّة 60 يوماً خلال آخر 30 يوماً، على "
            f"مستوى كل أصل على حدة) على هدف {target} — متابعة لتأكيد مستقلّ "
            "(RSI وVWAP_DEVIATION ينكسران في نفس نافذة انهيار FTX). راجع "
            "docs/research/vwap_deviation_and_volume_concentration.md."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={"mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
                "consistent_sign": consistent_sign, "n_ok": n_ok},
        notes=(
            "نتيجة مختلطة: الإشارة على high تختلف بين النظامين لكن بعكس "
            "الاتجاه المتوقَّع وبضعف. النظام منتشر جداً (73% من العيّنات) "
            "على مستوى الأصل الفردي، ما يُضعف قدرته على عزل الظاهرة "
            "الملاحَظة على مستوى نافذة السوق ككل."
        ),
    )

print(f'✅ سُجِّلت {len(CRASH_REBOUND_REGIME_RESULTS)} توليفة — نظام ارتداد ما بعد الانهيار (تعريف مباشر) نتيجة مختلطة، مرفوضة.')


## ٣٦) متابعة ثالثة وأخيرة — نظام ارتداد ما بعد الانهيار على مستوى السوق ككل

ثالث وآخر محاولة لعزل ظاهرة انكسار RSI/VWAP في نوافذ محدَّدة: `MKT_CRASH_REBOUND_REGIME` (على مستوى العملة المرجعية BTC، مشترك بين كل الأصول). راجع [التوثيق الكامل](../docs/research/vwap_deviation_and_volume_concentration.md#متابعة-ثالثة-وأخيرة-نظام-على-مستوى-السوق-ككل-mkt_crash_rebound_regime-لا-يحسم-الأمر-أيضاً).

### النتيجة الفعلية

| الميزة | الهدف | الفئة | mean_ic | frac_significant | consistent_sign |
|---|---|---|---|---|---|
| `RSI_14` | high | نظام سوقي | -0.121 | 67% | ❌ False |
| `RSI_14` | high | عادي | +0.004 | 26% | ❌ False |
| `MKT_CRASH_REBOUND_REGIME` | high | نظام سوقي | +0.073 | 33% | ❌ False |

**لا يحسم الأمر.** الإشارة تختلف بين النظامين لكن بعكس اتجاه الفرضية الأصلية — النافذة الزمنية للتعريف (60/30 يوماً) أوسع من لحظة الارتداد الفعلية الضيّقة. ثلاث محاولات مستقلّة لعزل هذه الظاهرة (نسبي بالزخم، أصل فردي، سوق ككل) فشلت جميعها؛ الظاهرة نفسها حقيقية وموثَّقة (تأكيد مستقلّ عبر RSI وVWAP في نفس النافذة)، لكن تعريفها التشغيلي يحتاج دقّة زمنية أعلى بكثير — يُترَك مفتوحاً لمحاولة مستقبلية.

### تسجيل المتابعة الثالثة في سجلّ التجارب

In [ ]:
# @title
MKT_CRASH_REBOUND_RESULTS = [
    ("RSI_14", "close", "crash_rebound", -0.166708, 0.110876, 0.833333, False, 12),
    ("RSI_14", "close", "normal", -0.122084, 0.105891, 0.739130, False, 23),
    ("RSI_14", "high", "crash_rebound", -0.120832, 0.156070, 0.666667, False, 12),
    ("RSI_14", "high", "normal", 0.003510, 0.129103, 0.260870, False, 23),
    ("RSI_14", "low", "crash_rebound", -0.045284, 0.095922, 0.416667, False, 12),
    ("RSI_14", "low", "normal", -0.133692, 0.097583, 0.869565, False, 23),
    ("MKT_CRASH_REBOUND_REGIME", "close", "crash_rebound", 0.025849, 0.081103, 0.250000, False, 12),
    ("MKT_CRASH_REBOUND_REGIME", "close", "normal", 0.000877, 0.069861, 0.043478, False, 23),
    ("MKT_CRASH_REBOUND_REGIME", "high", "crash_rebound", 0.072936, 0.120370, 0.333333, False, 12),
    ("MKT_CRASH_REBOUND_REGIME", "high", "normal", -0.007999, 0.073114, 0.043478, False, 23),
    ("MKT_CRASH_REBOUND_REGIME", "low", "crash_rebound", -0.035868, 0.117716, 0.250000, False, 12),
    ("MKT_CRASH_REBOUND_REGIME", "low", "normal", 0.010110, 0.055014, 0.043478, False, 23),
]

for feature, target, regime, mean_ic, std_ic, frac_sig, consistent_sign, n_ok in MKT_CRASH_REBOUND_RESULTS:
    register_hypothesis(
        hyp_id=f"MKT_CRASH_REBOUND_{feature}_{regime}_{target}",
        hypothesis=(
            f"{feature} مشروطاً بنظام ارتداد ما بعد الانهيار على مستوى "
            f"السوق ككل ({regime}، عبر العملة المرجعية BTC، مشترك بين كل "
            f"الأصول) على هدف {target} — ثالث وآخر محاولة لعزل ظاهرة "
            "انكسار RSI/VWAP المشخَّصة في القسمين ٣٤-٣٥. راجع "
            "docs/research/vwap_deviation_and_volume_concentration.md."
        ),
        source="literature_mining",
        status="مرفوضة",
        report={"mean_ic": mean_ic, "std_ic": std_ic, "frac_significant": frac_sig,
                "consistent_sign": consistent_sign, "n_ok": n_ok},
        notes=(
            "ثالث محاولة مستقلّة لعزل نفس الظاهرة (بعد نسبي بالزخم في "
            "القسم ٢٨، وأصل فردي في القسم ٣٥) — لا تحسم الأمر أيضاً. "
            "الظاهرة الأصلية (تأكيد مستقلّ عبر RSI/VWAP في نفس نافذة "
            "انهيار FTX تحديداً) لا تزال حقيقية وموثَّقة، لكن نافذة "
            "التعريف الزمني هنا (60/30 يوماً) أوسع من لحظة الارتداد "
            "الفعلية — يُترَك مفتوحاً لتعريف أضيق مستقبلاً."
        ),
    )

print(f'✅ سُجِّلت {len(MKT_CRASH_REBOUND_RESULTS)} توليفة — المتابعة الثالثة والأخيرة لنظام ارتداد ما بعد الانهيار، لا تحسم الأمر.')
